# Clean Pipeline 1 — Data Preparation and OCT Self-Supervised Pretraining

This notebook contains only the reproducible data-management, protocol-freezing, and OCT self-supervised learning path required by the final glaucoma experiments.

**Intentionally removed:** GOALS segmentation, GAMMA, Ishikawa, unrelated baseline experiments, exploratory diagnostics, and obsolete training stages.

The notebook preserves the original project paths, frozen manifests, split logic, native-axis OCT handling, SSL cache, and DINO-style pretraining configuration.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/glaucoma_oct_research"

print("Exists:", os.path.exists(PROJECT_ROOT))

if os.path.exists(PROJECT_ROOT):
    print(os.listdir(PROJECT_ROOT))

In [ ]:
# ============================================================
# Stage 0: Colab Environment Setup
# ============================================================

import os
import random
from pathlib import Path

# Set deterministic CUDA behavior before importing PyTorch
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)

import numpy as np
np.random.seed(SEED)

# ------------------------------------------------------------
# Import PyTorch
# ------------------------------------------------------------

import torch

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/glaucoma_oct_research")

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

SPLIT_DIR = PROJECT_DIR / "splits"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
LOG_DIR = PROJECT_DIR / "logs"

for directory in [
    PROJECT_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    SPLIT_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Central configuration
# ------------------------------------------------------------

CONFIG = {
    "seed": SEED,
    "framework": "pytorch",
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Project paths
    "project_dir": str(PROJECT_DIR),
    "raw_data_dir": str(RAW_DATA_DIR),
    "processed_data_dir": str(PROCESSED_DATA_DIR),
    "split_dir": str(SPLIT_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "results_dir": str(RESULTS_DIR),
    "figures_dir": str(FIGURES_DIR),
    "log_dir": str(LOG_DIR),

    # Initial OCT settings
    "rnflt_size": (224, 224),
    "default_oct_volume_size": (200, 200, 200),

    # Training defaults
    "batch_size": 8,
    "num_workers": 2,
    "pin_memory": True,
    "mixed_precision": True,
}

# ------------------------------------------------------------
# Device information
# ------------------------------------------------------------

print("=" * 60)
print("Environment")
print("=" * 60)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory      : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("GPU             : CPU")

print()
print("Project directory:")
print(PROJECT_DIR)

print()
print("Created directories:")
for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    SPLIT_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOG_DIR,
]:
    print(f"  {directory}")

print()
print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

print()
print("=" * 60)
print("Stage 0 completed successfully.")
print("=" * 60)

In [ ]:
!pip install -q -U gdown==6.4.1

In [ ]:
# ==================== Stage 1 ====================

# ============================================================
# Stage 1: Dataset Source Verification and Acquisition
# ============================================================

import os
import sys
import json
import hashlib
import zipfile
import re
import subprocess
import importlib.metadata
import requests

from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download

assert "PROJECT_DIR" in globals(), "Run Stage 0 first."


# ============================================================
# Paths
# ============================================================

DATASET_DIR = RAW_DATA_DIR / "datasets"

DATASET_DIRS = {
    "harvard_gf": DATASET_DIR / "harvard_gf",
    "harvard_gdp": DATASET_DIR / "harvard_gdp",
    "goals": DATASET_DIR / "goals",
    "gamma": DATASET_DIR / "gamma",
    "retina_oct_glaucoma": DATASET_DIR / "retina_oct_glaucoma",
}

for path in DATASET_DIRS.values():
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# Dataset sources
# ============================================================

HARVARD_GF_REPO = "harvardairobotics/Harvard-GF"
HARVARD_GDP_REPO = "harvardairobotics/Harvard-GDP"

HARVARD_GDP_RNFLT_FOLDER_ID = (
    "1F9wM5W-00XMd2tqlt8b_pyi_yjkRwfpc"
)

HARVARD_GDP_RAW_OCT_FOLDER_ID = (
    "1e36KTZ_LdGzTkn6ajzv5JLxVxDH-NRqa"
)

HARVARD_GDP_RNFLT_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    + HARVARD_GDP_RNFLT_FOLDER_ID
)

HARVARD_GDP_RAW_OCT_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    + HARVARD_GDP_RAW_OCT_FOLDER_ID
)

GOALS_FILE_ID = (
    "1P1cLm9_Pwq4fum4lB1-LGUayO-NMvF5I"
)

GAMMA_FILE_ID = (
    "1thJDE1_TR-xa8f3H-0PwPpDxun7rV8Sw"
)

GOALS_SOURCE = (
    "https://ichallenges.grand-challenge.org/iChallenge-GON3/"
)

GAMMA_SOURCE = (
    "https://gamma.grand-challenge.org/"
)

ISHIKAWA_SOURCE = (
    "https://zenodo.org/records/7957454"
)

ISHIKAWA_RECORD_ID = "7957454"


# ============================================================
# Validation helpers
# ============================================================

def valid_zip(path):
    path = Path(path)

    return (
        path.exists()
        and path.stat().st_size > 1024 * 1024
        and zipfile.is_zipfile(path)
    )


def md5_file(path):
    digest = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def ensure_gdown():
    try:
        version = importlib.metadata.version(
            "gdown"
        )

        parts = [
            int(x)
            for x in re.findall(
                r"\d+",
                version,
            )[:3]
        ]

        while len(parts) < 3:
            parts.append(0)

        current_version = tuple(parts)

    except Exception:
        current_version = (0, 0, 0)

    if current_version < (6, 0, 0):

        print(
            "Installing/upgrading gdown>=6.0.0..."
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-U",
                "gdown>=6.0.0",
            ]
        )

    import gdown

    return gdown


def download_hf_archive(
    repo_id,
    archive_name,
    output_dir,
):
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = (
        output_dir
        / Path(archive_name).name
    )

    if valid_zip(output_path):

        print(
            f"Valid existing archive: "
            f"{output_path}"
        )

        return output_path

    if output_path.exists():
        output_path.unlink()

    print(
        f"Downloading "
        f"{repo_id}/{archive_name}"
    )

    downloaded = hf_hub_download(
        repo_id=repo_id,
        filename=archive_name,
        repo_type="dataset",
        local_dir=str(output_dir),
    )

    downloaded = Path(downloaded)

    if not valid_zip(downloaded):
        raise RuntimeError(
            f"Invalid Harvard archive: "
            f"{downloaded}"
        )

    return downloaded


def download_google_drive(
    file_id,
    output_path,
):
    output_path = Path(output_path)

    if valid_zip(output_path):

        print(
            f"Valid existing archive: "
            f"{output_path}"
        )

        return output_path

    if output_path.exists():
        output_path.unlink()

    gdown = ensure_gdown()

    print(
        f"Downloading Google Drive archive "
        f"to {output_path}"
    )

    result = gdown.download(
        id=file_id,
        output=str(output_path),
        quiet=False,
    )

    if (
        result is None
        or not valid_zip(output_path)
    ):
        raise RuntimeError(
            "Google Drive download failed "
            "or archive is invalid: "
            f"{output_path}"
        )

    return output_path


def download_google_drive_folder(
    folder_id,
    output_dir,
):
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    gdown = ensure_gdown()

    print(
        "Google Drive folder:"
    )

    print(
        f"https://drive.google.com/drive/folders/"
        f"{folder_id}"
    )

    print(
        f"Destination:\n{output_dir}"
    )

    print(
        "Downloading folder..."
    )

    gdown.download_folder(
        id=folder_id,
        output=str(output_dir),
        quiet=False,
    )

    return output_dir


def harvard_gdp_rnflt_files(
    root_dir,
):
    root_dir = Path(root_dir)

    files = []

    for path in root_dir.rglob(
        "data_*.npz"
    ):

        if re.fullmatch(
            r"data_\d{4}\.npz",
            path.name,
            flags=re.IGNORECASE,
        ):
            files.append(path)

    return sorted(
        files,
        key=lambda p: int(
            re.search(
                r"(\d{4})",
                p.stem,
            ).group(1)
        ),
    )


def validate_harvard_gdp_rnflt(
    root_dir,
):
    files = harvard_gdp_rnflt_files(
        root_dir
    )

    expected_ids = set(
        range(1, 1001)
    )

    found_ids = {
        int(
            re.search(
                r"(\d{4})",
                p.stem,
            ).group(1)
        )
        for p in files
    }

    complete = (
        len(files) == 1000
        and found_ids == expected_ids
    )

    return complete, files


def inspect_rnflt_schema(
    file_path,
):
    import numpy as np

    with np.load(
        file_path,
        allow_pickle=False,
    ) as data:

        fields = list(
            data.files
        )

        schema = {
            "fields": fields,
        }

        for key in [
            "rnflt",
            "glaucoma",
            "progression",
            "md",
            "tds",
            "age",
            "male",
            "race",
            "hispanic",
        ]:

            if key not in data.files:
                continue

            array = np.asarray(
                data[key]
            )

            schema[
                key
            ] = {
                "shape": list(
                    array.shape
                ),
                "dtype": str(
                    array.dtype
                ),
            }

        return schema


# ============================================================
# Zenodo helpers
# ============================================================

def get_zenodo_file(
    record_id,
    file_name,
):
    response = requests.get(
        f"https://zenodo.org/api/records/"
        f"{record_id}",
        timeout=60,
    )

    response.raise_for_status()

    record = response.json()

    for file_info in record.get(
        "files",
        [],
    ):

        if (
            file_info.get("key")
            == file_name
        ):
            return file_info

    raise RuntimeError(
        f"{file_name} not found in "
        f"Zenodo record {record_id}."
    )


def download_zenodo_resumable(
    url,
    output_path,
    expected_size=None,
    expected_md5=None,
):
    output_path = Path(
        output_path
    )

    part_path = Path(
        str(output_path)
        + ".part"
    )

    if valid_zip(output_path):

        size_ok = (
            expected_size is None
            or output_path.stat().st_size
            == expected_size
        )

        md5_ok = (
            expected_md5 is None
            or md5_file(output_path)
            == expected_md5
        )

        if size_ok and md5_ok:

            print(
                f"Valid existing archive: "
                f"{output_path}"
            )

            return output_path

        output_path.unlink()

    start = (
        part_path.stat().st_size
        if part_path.exists()
        else 0
    )

    headers = (
        {"Range": f"bytes={start}-"}
        if start
        else {}
    )

    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=120,
    )

    if (
        start
        and response.status_code == 200
    ):

        response.close()

        part_path.unlink(
            missing_ok=True
        )

        start = 0

        response = requests.get(
            url,
            stream=True,
            timeout=120,
        )

    response.raise_for_status()

    total = (
        expected_size
        or int(
            response.headers.get(
                "content-length",
                0,
            )
            or 0
        )
    )

    downloaded = start

    mode = (
        "ab"
        if start
        else "wb"
    )

    try:

        with open(
            part_path,
            mode,
        ) as f:

            for chunk in response.iter_content(
                chunk_size=8 * 1024 * 1024
            ):

                if not chunk:
                    continue

                f.write(chunk)

                downloaded += len(
                    chunk
                )

                if total:

                    print(
                        f"\rProgress: "
                        f"{downloaded / total * 100:6.2f}%",
                        end="",
                    )

        print()

    except KeyboardInterrupt:

        print(
            "\nDownload interrupted. "
            f"Partial file kept at: "
            f"{part_path}"
        )

        raise

    finally:
        response.close()

    if (
        expected_size is not None
        and part_path.stat().st_size
        != expected_size
    ):

        raise RuntimeError(
            "Zenodo archive size mismatch."
        )

    if not zipfile.is_zipfile(
        part_path
    ):

        raise RuntimeError(
            "Zenodo archive is not "
            "a valid ZIP file."
        )

    if (
        expected_md5 is not None
        and md5_file(part_path)
        != expected_md5
    ):

        raise RuntimeError(
            "Zenodo archive MD5 mismatch."
        )

    part_path.replace(
        output_path
    )

    return output_path


# ============================================================
# Main acquisition
# ============================================================

api = HfApi()

summary_paths = {}
archive_paths = {}

harvard_gdp_rnflt_dir = (
    DATASET_DIRS["harvard_gdp"]
    / "rnflt_maps"
)

harvard_gdp_rnflt_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 60)
print("Stage 1 - Dataset Acquisition")
print("=" * 60)


# ============================================================
# Harvard-GF
# ============================================================

print("\nHarvard-GF")

gf_files = api.list_repo_files(
    HARVARD_GF_REPO,
    repo_type="dataset",
)

gf_zip = next(
    (
        x
        for x in gf_files
        if x.lower().endswith(
            ".zip"
        )
    ),
    None,
)

gf_csv = next(
    (
        x
        for x in gf_files
        if "data_summary.csv"
        in x.lower()
    ),
    None,
)

if gf_zip is None:
    raise RuntimeError(
        "Harvard-GF archive not found."
    )

if gf_csv:

    summary_paths[
        "harvard_gf"
    ] = str(
        hf_hub_download(
            repo_id=HARVARD_GF_REPO,
            filename=gf_csv,
            repo_type="dataset",
            local_dir=str(
                DATASET_DIRS[
                    "harvard_gf"
                ]
            ),
        )
    )

archive_paths[
    "harvard_gf"
] = str(
    download_hf_archive(
        HARVARD_GF_REPO,
        gf_zip,
        DATASET_DIRS[
            "harvard_gf"
        ],
    )
)


# ============================================================
# Harvard-GDP raw OCT
# ============================================================

print("\nHarvard-GDP raw OCT")

gdp_files = api.list_repo_files(
    HARVARD_GDP_REPO,
    repo_type="dataset",
)

gdp_zip = next(
    (
        x
        for x in gdp_files
        if x.lower().endswith(
            ".zip"
        )
    ),
    None,
)

gdp_csv = next(
    (
        x
        for x in gdp_files
        if "data_summary.csv"
        in x.lower()
    ),
    None,
)

if gdp_zip is None:
    raise RuntimeError(
        "Harvard-GDP archive not found."
    )

if gdp_csv:

    summary_paths[
        "harvard_gdp"
    ] = str(
        hf_hub_download(
            repo_id=HARVARD_GDP_REPO,
            filename=gdp_csv,
            repo_type="dataset",
            local_dir=str(
                DATASET_DIRS[
                    "harvard_gdp"
                ]
            ),
        )
    )

archive_paths[
    "harvard_gdp"
] = str(
    download_hf_archive(
        HARVARD_GDP_REPO,
        gdp_zip,
        DATASET_DIRS[
            "harvard_gdp"
        ],
    )
)


# ============================================================
# Harvard-GDP RNFLT maps
# ============================================================

print("\nHarvard-GDP RNFLT maps")

rnflt_complete, rnflt_files = (
    validate_harvard_gdp_rnflt(
        harvard_gdp_rnflt_dir
    )
)

if rnflt_complete:

    print(
        "Valid existing Harvard-GDP "
        "RNFLT dataset found."
    )

    print(
        f"RNFLT files: "
        f"{len(rnflt_files)}"
    )

else:

    existing_count = len(
        rnflt_files
    )

    if existing_count:

        print(
            "Existing RNFLT directory is "
            f"incomplete: {existing_count} files."
        )

    download_google_drive_folder(
        HARVARD_GDP_RNFLT_FOLDER_ID,
        harvard_gdp_rnflt_dir,
    )

    rnflt_complete, rnflt_files = (
        validate_harvard_gdp_rnflt(
            harvard_gdp_rnflt_dir
        )
    )

    if not rnflt_complete:

        raise RuntimeError(
            "Harvard-GDP RNFLT folder download "
            "is incomplete. Expected exactly "
            "data_0001.npz ... data_1000.npz."
        )

print(
    f"Harvard-GDP RNFLT files: "
    f"{len(rnflt_files)}"
)

rnflt_schema = inspect_rnflt_schema(
    rnflt_files[0]
)

print(
    "RNFLT schema sample:"
)

print(
    f"  File: {rnflt_files[0]}"
)

print(
    f"  Fields: "
    f"{rnflt_schema['fields']}"
)

if "rnflt" in rnflt_schema:

    print(
        f"  rnflt: "
        f"{rnflt_schema['rnflt']['shape']} "
        f"{rnflt_schema['rnflt']['dtype']}"
    )

if "progression" in rnflt_schema:

    print(
        f"  progression: "
        f"{rnflt_schema['progression']['shape']} "
        f"{rnflt_schema['progression']['dtype']}"
    )


# ============================================================
# GOALS
# ============================================================

print("\nGOALS")

goals_path = (
    DATASET_DIRS["goals"]
    / "goals.zip"
)

archive_paths[
    "goals"
] = str(
    download_google_drive(
        GOALS_FILE_ID,
        goals_path,
    )
)


# ============================================================
# GAMMA
# ============================================================

print("\nGAMMA")

gamma_path = (
    DATASET_DIRS["gamma"]
    / "gamma.zip"
)

archive_paths[
    "gamma"
] = str(
    download_google_drive(
        GAMMA_FILE_ID,
        gamma_path,
    )
)


# ============================================================
# Ishikawa
# ============================================================

print(
    "\nRetina OCT Glaucoma / Ishikawa"
)

ishikawa_info = get_zenodo_file(
    ISHIKAWA_RECORD_ID,
    "retina-oct-glaucoma.zip",
)

ishikawa_path = (
    DATASET_DIRS[
        "retina_oct_glaucoma"
    ]
    / "retina-oct-glaucoma.zip"
)

checksum = ishikawa_info.get(
    "checksum"
)

expected_md5 = (
    checksum[4:]
    if (
        isinstance(
            checksum,
            str,
        )
        and checksum.startswith(
            "md5:"
        )
    )
    else None
)

archive_paths[
    "retina_oct_glaucoma"
] = str(
    download_zenodo_resumable(
        ishikawa_info[
            "links"
        ][
            "self"
        ],
        ishikawa_path,
        expected_size=(
            ishikawa_info.get(
                "size"
            )
        ),
        expected_md5=expected_md5,
    )
)


# ============================================================
# Source report
# ============================================================

sources_report = {

    "harvard_gf": {
        "source": (
            "https://huggingface.co/datasets/"
            + HARVARD_GF_REPO
        ),
        "archive": archive_paths[
            "harvard_gf"
        ],
        "metadata": summary_paths.get(
            "harvard_gf"
        ),
    },

    "harvard_gdp": {
        "source": (
            "https://huggingface.co/datasets/"
            + HARVARD_GDP_REPO
        ),
        "repository_documentation": (
            "https://github.com/"
            "Harvard-Ophthalmology-AI-Lab/"
            "Harvard-GDP"
        ),
        "rnflt_documented_source": (
            "https://github.com/"
            "luoyan407/"
            "Harvard-GDP"
        ),
        "archive": archive_paths[
            "harvard_gdp"
        ],
        "metadata": summary_paths.get(
            "harvard_gdp"
        ),
        "rnflt_dir": str(
            harvard_gdp_rnflt_dir
        ),
        "rnflt_source_url": (
            HARVARD_GDP_RNFLT_FOLDER_URL
        ),
        "raw_oct_source_url": (
            HARVARD_GDP_RAW_OCT_FOLDER_URL
        ),
        "rnflt_expected_count": 1000,
        "rnflt_expected_shape": [
            225,
            225,
        ],
        "progression_expected_count": 500,
        "rnflt_schema_sample": (
            rnflt_schema
        ),
    },

    "goals": {
        "source": GOALS_SOURCE,
        "archive": archive_paths[
            "goals"
        ],
    },

    "gamma": {
        "source": GAMMA_SOURCE,
        "archive": archive_paths[
            "gamma"
        ],
    },

    "retina_oct_glaucoma": {
        "source": ISHIKAWA_SOURCE,
        "archive": archive_paths[
            "retina_oct_glaucoma"
        ],
        "checksum": expected_md5,
    },
}


with open(
    DATASET_DIR
    / "dataset_sources.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        sources_report,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# Final validation
# ============================================================

missing = [
    name
    for name, path
    in archive_paths.items()
    if not Path(path).exists()
]

if missing:

    raise RuntimeError(
        f"Missing dataset archives: "
        f"{missing}"
    )


print("\n" + "=" * 60)
print(
    "Stage 1 completed successfully"
)
print("=" * 60)

for name, path in archive_paths.items():

    print(
        f"{name}: {path}"
    )

print(
    "harvard_gdp_rnflt: "
    f"{harvard_gdp_rnflt_dir}"
)

print("=" * 60)

In [ ]:
# ============================================================
# Stage 2: Final Dataset Audit and Unified Manifest
# ============================================================

import io
import os
import re
import sys
import json
import math
import shutil
import zipfile
import platform
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from PIL import Image


assert "PROJECT_DIR" in globals(), "Run Stage 0 first."


# ============================================================
# Paths
# ============================================================

DATASET_DIR = RAW_DATA_DIR / "datasets"

AUDIT_DIR = (
    RESULTS_DIR
    / "data_audit"
    / "stage2"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEMP_DIR = Path(
    "/content/stage2_gamma_temp"
)

if TEMP_DIR.exists():
    shutil.rmtree(
        TEMP_DIR,
        ignore_errors=True,
    )

TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SOURCE_REPORT_PATH = (
    DATASET_DIR
    / "dataset_sources.json"
)

if not SOURCE_REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Stage 1 source report not found:\n"
        f"{SOURCE_REPORT_PATH}"
    )

with open(
    SOURCE_REPORT_PATH,
    "r",
    encoding="utf-8",
) as f:
    SOURCE_REPORT = json.load(f)


# ============================================================
# Expected properties
# ============================================================

EXPECTED = {
    "harvard_gf": {
        "samples": 3300,
        "oct_shape": [200, 200, 200],
        "rnflt_shape": [200, 200],
    },

    "harvard_gdp": {
        "samples": 1000,
        "raw_oct_shape": [200, 200, 300],
        "rnflt_shape": [225, 225],
        "progression_labeled": 500,
        "progression_labeled_start": 1,
        "progression_labeled_end": 500,
        "progression_unlabeled_start": 501,
        "progression_unlabeled_end": 1000,
    },

    "goals": {
        "reference_samples": 300,
        "usable_samples": 299,
        "split_reference": {
            "Train": 100,
            "Validation": 100,
            "Test": 100,
        },
        "split_usable": {
            "Train": 100,
            "Validation": 100,
            "Test": 99,
        },
        "missing_reference_ids": {
            "Test": [300],
        },
    },

    "gamma": {
        "samples": 300,
        "split_samples": {
            "Train": 100,
            "Validation": 100,
            "Test": 100,
        },
        "oct_bscans": 256,
    },

    "retina_oct_glaucoma": {
        "samples": 884,
    },
}


PROGRESSION_NAMES = [
    "md",
    "vfi",
    "td_pointwise",
    "md_fast",
    "md_fast_no_p_cut",
    "td_pointwise_no_p_cut",
]

KNOWN_ISHIKAWA_PREFIXES = {
    "POAG_": 1,
    "NORMAL_": 0,
}

IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}


# ============================================================
# Containers
# ============================================================

checks = []
dataset_summaries = []
schemas = {}
discrepancies = []
manifest_objects = {}


# ============================================================
# General helpers
# ============================================================

def add_check(
    dataset,
    check,
    status,
    observed=None,
    expected=None,
    details=None,
):
    checks.append(
        {
            "dataset": dataset,
            "check": check,
            "status": status,
            "observed": observed,
            "expected": expected,
            "details": details,
        }
    )


def natural_key(value):
    value = str(value)

    return [
        int(x)
        if x.isdigit()
        else x.lower()
        for x in re.split(
            r"(\d+)",
            value,
        )
    ]


def is_real_zip_member(name):
    parts = PurePosixPath(
        name
    ).parts

    if any(
        part == "__MACOSX"
        for part in parts
    ):
        return False

    if any(
        part.startswith("._")
        for part in parts
    ):
        return False

    if PurePosixPath(
        name
    ).name in {
        ".DS_Store",
        "Thumbs.db",
    }:
        return False

    return True


def is_image_member(name):
    return (
        Path(name).suffix.lower()
        in IMAGE_EXTENSIONS
    )


def numeric_id_from_stem(stem):
    match = re.search(
        r"(\d+)",
        str(stem),
    )

    if match is None:
        return None

    return int(
        match.group(1)
    )


def numeric_id_from_member(name):
    return numeric_id_from_stem(
        Path(name).stem
    )


def scalar_value(value):
    array = np.asarray(value)

    if array.size == 0:
        return None

    if array.size == 1:
        item = array.reshape(-1)[0]

        if isinstance(
            item,
            np.generic,
        ):
            item = item.item()

        return item

    return array.tolist()


def normalize_binary(value):
    if value is None:
        return None

    text = str(value).strip().lower()

    if text in {
        "1",
        "1.0",
        "true",
        "yes",
        "y",
        "t",
    }:
        return 1

    if text in {
        "0",
        "0.0",
        "false",
        "no",
        "n",
        "f",
    }:
        return 0

    return None


def normalize_id(value):
    if value is None:
        return None

    if isinstance(
        value,
        (float, np.floating),
    ):
        if np.isnan(value):
            return None

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
    }:
        return None

    match = re.search(
        r"(\d+)",
        text,
    )

    if match is None:
        return None

    return int(
        match.group(1)
    )


def clean_label_value(value):
    if value is None:
        return None

    text = str(value).strip()

    if text.lower() in {
        "",
        "none",
        "nan",
        "na",
        "null",
    }:
        return None

    return text


def parse_progression_vector(
    value
):
    if value is None:
        return None, False

    array = np.asarray(
        value
    ).reshape(-1)

    values = []

    for item in array:

        cleaned = clean_label_value(
            item
        )

        if cleaned is None:
            return None, False

        normalized = normalize_binary(
            cleaned
        )

        if normalized is None:
            return None, False

        values.append(
            normalized
        )

    if len(values) != 6:
        return None, False

    return values, True


def safe_json(value):
    return json.dumps(
        value,
        ensure_ascii=False,
    )


def save_json(
    path,
    data,
):
    path = Path(path)

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )


def archive_locator(
    archive,
    member,
):
    return (
        f"zip://{Path(archive)}!{member}"
    )


def nested_archive_locator(
    outer_archive,
    nested_archive,
    member,
):
    return (
        f"nestedzip://"
        f"{Path(outer_archive)}!"
        f"{nested_archive}!"
        f"{member}"
    )


def open_image_from_zip(
    zf,
    member,
):
    raw = zf.read(
        member
    )

    image = Image.open(
        io.BytesIO(raw)
    )

    image.load()

    return image


def image_schema(
    image,
):
    array = np.asarray(
        image
    )

    result = {
        "mode": image.mode,
        "size": [
            int(image.width),
            int(image.height),
        ],
        "array_shape": list(
            array.shape
        ),
        "dtype": str(
            array.dtype
        ),
    }

    if array.size > 0:

        if array.ndim == 2:

            unique = np.unique(
                array
            )

            if len(unique) <= 50:

                result[
                    "unique_values"
                ] = [
                    (
                        x.item()
                        if isinstance(
                            x,
                            np.generic,
                        )
                        else x
                    )
                    for x in unique
                ]

    return result


def read_npz_schema_from_zip(
    zf,
    member,
):
    raw = zf.read(
        member
    )

    with np.load(
        io.BytesIO(raw),
        allow_pickle=False,
    ) as data:

        schema = {
            "fields": list(
                data.files
            )
        }

        for key in data.files:

            array = np.asarray(
                data[key]
            )

            entry = {
                "shape": list(
                    array.shape
                ),
                "dtype": str(
                    array.dtype
                ),
            }

            if key in {
                "glaucoma",
                "progression",
                "md",
                "vfi",
                "age",
                "male",
                "race",
                "hispanic",
                "language",
                "maritalstatus",
            }:

                entry[
                    "value_sample"
                ] = scalar_value(
                    array
                )

            schema[
                key
            ] = entry

        return schema


def read_mha_header(
    zf,
    member,
):
    with zf.open(
        member,
        "r",
    ) as f:

        raw = f.read(
            65536
        )

    text = raw.decode(
        "latin1",
        errors="ignore",
    )

    header = {}

    for line in text.splitlines():

        if "=" not in line:
            continue

        key, value = line.split(
            "=",
            1,
        )

        key = key.strip()
        value = value.strip()

        if key in {
            "ObjectType",
            "NDims",
            "DimSize",
            "ElementType",
            "ElementSpacing",
            "ElementDataFile",
        }:

            header[key] = value

    return header


def archive_infos(
    archive,
):
    with zipfile.ZipFile(
        archive,
        "r",
    ) as zf:

        return [
            info
            for info in zf.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]


def ensure_archive_exists(
    dataset,
):
    path = Path(
        SOURCE_REPORT[
            dataset
        ]["archive"]
    )

    if not path.exists():
        raise FileNotFoundError(
            f"{dataset} archive not found:\n"
            f"{path}"
        )

    if not zipfile.is_zipfile(
        path
    ):
        raise RuntimeError(
            f"{dataset} archive is not a valid ZIP:\n"
            f"{path}"
        )

    return path


def add_summary_row(
    dataset,
    row,
):
    dataset_summaries.append(
        {
            "dataset":
                dataset,
            **row,
        }
    )


# ============================================================
# Harvard-GF
# ============================================================

def load_summary_lookup(
    summary_path,
):
    if not summary_path:
        return {}, None, []

    path = Path(
        summary_path
    )

    if not path.exists():
        return {}, None, []

    df = pd.read_csv(
        path
    )

    if "filename" not in df.columns:
        return {}, df, []

    lookup = {}
    duplicates = []

    for _, record in df.iterrows():

        filename = str(
            record[
                "filename"
            ]
        ).strip()

        basename = Path(
            filename
        ).name

        if basename in lookup:
            duplicates.append(
                basename
            )

        lookup[
            basename
        ] = record.to_dict()

    return (
        lookup,
        df,
        duplicates,
    )


def audit_harvard_gf():

    dataset = "harvard_gf"

    archive = ensure_archive_exists(
        dataset
    )

    summary_path = (
        SOURCE_REPORT[
            dataset
        ].get("metadata")
    )

    (
        summary_lookup,
        summary_df,
        summary_duplicates,
    ) = load_summary_lookup(
        summary_path
    )

    if summary_duplicates:

        add_check(
            dataset,
            "summary_filename_duplicates",
            "CHECK",
            len(
                summary_duplicates
            ),
            0,
            summary_duplicates[:20],
        )

    manifest_rows = []

    glaucoma_counts = Counter()

    bad_oct_shapes = []
    bad_rnflt_shapes = []
    bad_glaucoma_labels = []

    schema_samples = []

    with zipfile.ZipFile(
        archive,
        "r",
    ) as zf:

        infos = [
            info
            for info
            in zf.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]

        npz_infos = [
            info
            for info
            in infos
            if re.fullmatch(
                r"data_\d{4}\.npz",
                Path(
                    info.filename
                ).name,
                flags=re.IGNORECASE,
            )
        ]

        npz_infos = sorted(
            npz_infos,
            key=lambda x:
                natural_key(
                    x.filename
                ),
        )

        ids = [
            numeric_id_from_stem(
                Path(
                    info.filename
                ).stem
            )
            for info in npz_infos
        ]

        expected_ids = set(
            range(
                1,
                EXPECTED[
                    dataset
                ]["samples"] + 1,
            )
        )

        found_ids = {
            x
            for x in ids
            if x is not None
        }

        duplicate_ids = [
            x
            for x, count
            in Counter(
                ids
            ).items()
            if count > 1
        ]

        missing_ids = sorted(
            expected_ids
            - found_ids
        )

        extra_ids = sorted(
            found_ids
            - expected_ids
        )

        add_check(
            dataset,
            "sample_count",
            (
                "PASS"
                if len(npz_infos)
                == EXPECTED[
                    dataset
                ]["samples"]
                else "FAIL"
            ),
            len(npz_infos),
            EXPECTED[
                dataset
            ]["samples"],
        )

        add_check(
            dataset,
            "sample_id_coverage",
            (
                "PASS"
                if (
                    not missing_ids
                    and not extra_ids
                )
                else "FAIL"
            ),
            {
                "missing":
                    missing_ids,
                "extra":
                    extra_ids,
            },
            "1..3300",
        )

        add_check(
            dataset,
            "duplicate_sample_ids",
            (
                "PASS"
                if not duplicate_ids
                else "FAIL"
            ),
            duplicate_ids,
            [],
        )

        sample_positions = sorted(
            set(
                [
                    0,
                    len(npz_infos) // 2,
                    len(npz_infos) - 1,
                ]
            )
        )

        for index in sample_positions:

            info = npz_infos[
                index
            ]

            schema = (
                read_npz_schema_from_zip(
                    zf,
                    info.filename,
                )
            )

            schema_samples.append(
                {
                    "member":
                        info.filename,
                    "schema":
                        schema,
                }
            )

        schemas[
            dataset
        ] = schema_samples

        required_fields = {
            "oct_bscans",
            "rnflt",
            "glaucoma",
        }

        schema_missing = sorted(
            required_fields
            - set(
                schema_samples[0][
                    "schema"
                ]["fields"]
            )
        )

        add_check(
            dataset,
            "required_npz_fields",
            (
                "PASS"
                if not schema_missing
                else "FAIL"
            ),
            schema_samples[0][
                "schema"
            ]["fields"],
            sorted(
                required_fields
            ),
            {
                "missing":
                    schema_missing
            },
        )

        for info in npz_infos:

            sid = numeric_id_from_stem(
                Path(
                    info.filename
                ).stem
            )

            row = {
                "dataset":
                    dataset,
                "split":
                    None,
                "sample_id":
                    sid,
                "patient_id":
                    None,
                "eye_id":
                    None,
                "patient_id_status":
                    "not_provided",
                "eye_id_status":
                    "not_provided",
                "sample_locator":
                    archive_locator(
                        archive,
                        info.filename,
                    ),
                "oct_locator":
                    archive_locator(
                        archive,
                        info.filename,
                    ),
                "rnflt_locator":
                    archive_locator(
                        archive,
                        info.filename,
                    ),
                "oct_format":
                    "npz_volume",
                "rnflt_format":
                    "npz_array",
                "oct_member":
                    info.filename,
                "rnflt_member":
                    info.filename,
                "oct_shape":
                    None,
                "rnflt_shape":
                    None,
                "glaucoma_label":
                    None,
                "md":
                    None,
                "age":
                    None,
                "male":
                    None,
                "race":
                    None,
                "hispanic":
                    None,
                "language":
                    None,
                "maritalstatus":
                    None,
                "tds_json":
                    None,
                "rnflt_min":
                    None,
                "rnflt_max":
                    None,
                "rnflt_mean":
                    None,
                "rnflt_finite_fraction":
                    None,
                "summary_gender":
                    None,
                "summary_ethnicity":
                    None,
                "summary_language":
                    None,
                "summary_maritalstatus":
                    None,
                "summary_use":
                    None,
            }

            try:

                raw = zf.read(
                    info.filename
                )

                with np.load(
                    io.BytesIO(raw),
                    allow_pickle=False,
                ) as data:

                    if "oct_bscans" in data.files:

                        oct_array = np.asarray(
                            data[
                                "oct_bscans"
                            ]
                        )

                        row[
                            "oct_shape"
                        ] = list(
                            oct_array.shape
                        )

                        if (
                            list(
                                oct_array.shape
                            )
                            != EXPECTED[
                                dataset
                            ]["oct_shape"]
                        ):

                            bad_oct_shapes.append(
                                {
                                    "sample_id":
                                        sid,
                                    "shape":
                                        list(
                                            oct_array.shape
                                        ),
                                }
                            )

                    if "rnflt" in data.files:

                        rnflt = np.asarray(
                            data[
                                "rnflt"
                            ]
                        )

                        row[
                            "rnflt_shape"
                        ] = list(
                            rnflt.shape
                        )

                        if (
                            list(
                                rnflt.shape
                            )
                            != EXPECTED[
                                dataset
                            ]["rnflt_shape"]
                        ):

                            bad_rnflt_shapes.append(
                                {
                                    "sample_id":
                                        sid,
                                    "shape":
                                        list(
                                            rnflt.shape
                                        ),
                                }
                            )

                        if rnflt.size > 0:

                            row[
                                "rnflt_min"
                            ] = float(
                                np.nanmin(
                                    rnflt
                                )
                            )

                            row[
                                "rnflt_max"
                            ] = float(
                                np.nanmax(
                                    rnflt
                                )
                            )

                            row[
                                "rnflt_mean"
                            ] = float(
                                np.nanmean(
                                    rnflt
                                )
                            )

                            row[
                                "rnflt_finite_fraction"
                            ] = float(
                                np.isfinite(
                                    rnflt
                                ).mean()
                            )

                    if "glaucoma" in data.files:

                        label = normalize_binary(
                            scalar_value(
                                data[
                                    "glaucoma"
                                ]
                            )
                        )

                        row[
                            "glaucoma_label"
                        ] = label

                        if label is None:

                            bad_glaucoma_labels.append(
                                {
                                    "sample_id":
                                        sid,
                                    "value":
                                        scalar_value(
                                            data[
                                                "glaucoma"
                                            ]
                                        ),
                                }
                            )

                        else:
                            glaucoma_counts[
                                label
                            ] += 1

                    for key in [
                        "md",
                        "age",
                        "male",
                        "race",
                        "hispanic",
                        "language",
                        "maritalstatus",
                    ]:

                        if key in data.files:

                            row[
                                key
                            ] = scalar_value(
                                data[key]
                            )

                    if "tds" in data.files:

                        row[
                            "tds_json"
                        ] = safe_json(
                            np.asarray(
                                data[
                                    "tds"
                                ]
                            ).tolist()
                        )

                summary_record = (
                    summary_lookup.get(
                        Path(
                            info.filename
                        ).name
                    )
                )

                if summary_record:

                    row[
                        "summary_gender"
                    ] = summary_record.get(
                        "gender"
                    )

                    row[
                        "summary_ethnicity"
                    ] = summary_record.get(
                        "ethnicity"
                    )

                    row[
                        "summary_language"
                    ] = summary_record.get(
                        "language"
                    )

                    row[
                        "summary_maritalstatus"
                    ] = summary_record.get(
                        "maritalstatus"
                    )

                    row[
                        "summary_use"
                    ] = summary_record.get(
                        "use"
                    )

            except Exception as exc:

                add_check(
                    dataset,
                    f"npz_read_{sid}",
                    "FAIL",
                    str(exc),
                    None,
                )

            manifest_rows.append(
                row
            )

    manifest = pd.DataFrame(
        manifest_rows
    )

    add_check(
        dataset,
        "oct_shapes_all_samples",
        (
            "PASS"
            if not bad_oct_shapes
            else "FAIL"
        ),
        (
            "all_expected"
            if not bad_oct_shapes
            else bad_oct_shapes[:20]
        ),
        EXPECTED[
            dataset
        ]["oct_shape"],
    )

    add_check(
        dataset,
        "rnflt_shapes_all_samples",
        (
            "PASS"
            if not bad_rnflt_shapes
            else "FAIL"
        ),
        (
            "all_expected"
            if not bad_rnflt_shapes
            else bad_rnflt_shapes[:20]
        ),
        EXPECTED[
            dataset
        ]["rnflt_shape"],
    )

    add_check(
        dataset,
        "glaucoma_labels",
        (
            "PASS"
            if not bad_glaucoma_labels
            else "FAIL"
        ),
        {
            str(k):
                int(v)
            for k, v
            in glaucoma_counts.items()
        },
        "binary 0/1",
        (
            None
            if not bad_glaucoma_labels
            else bad_glaucoma_labels[:20]
        ),
    )

    manifest_path = (
        AUDIT_DIR
        / "harvard_gf_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False,
    )

    if summary_df is not None:

        summary_df.to_csv(
            AUDIT_DIR
            / "harvard_gf_summary_copy.csv",
            index=False,
        )

    add_summary_row(
        dataset,
        {
            "samples_observed":
                len(manifest),
            "samples_expected":
                EXPECTED[
                    dataset
                ]["samples"],
            "identity_level":
                "sample_only",
            "patient_id":
                "not_provided",
            "eye_id":
                "not_provided",
            "primary_oct":
                "3D OCT volume",
            "primary_rnflt":
                "200x200 RNFLT map",
            "glaucoma_label_distribution":
                dict(
                    glaucoma_counts
                ),
            "manifest":
                str(
                    manifest_path
                ),
        },
    )

    return manifest


# ============================================================
# Harvard-GDP
# ============================================================

def audit_harvard_gdp():

    dataset = "harvard_gdp"

    archive = ensure_archive_exists(
        dataset
    )

    rnflt_dir = Path(
        SOURCE_REPORT[
            dataset
        ]["rnflt_dir"]
    )

    if not rnflt_dir.exists():
        raise FileNotFoundError(
            f"RNFLT directory not found:\n"
            f"{rnflt_dir}"
        )

    expected_ids = set(
        range(1, 1001)
    )

    expected_progression_ids = set(
        range(
            EXPECTED[dataset][
                "progression_labeled_start"
            ],
            EXPECTED[dataset][
                "progression_labeled_end"
            ] + 1,
        )
    )

    expected_unlabeled_ids = set(
        range(
            EXPECTED[dataset][
                "progression_unlabeled_start"
            ],
            EXPECTED[dataset][
                "progression_unlabeled_end"
            ] + 1,
        )
    )

    manifest_rows = []

    progression_counts = Counter()
    glaucoma_counts = Counter()

    bad_rnflt_shapes = []
    malformed_progression = []
    unlabeled_progression_ids = []

    with zipfile.ZipFile(
        archive,
        "r",
    ) as zf:

        infos = [
            info
            for info
            in zf.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]

        all_npz_infos = [
            info
            for info
            in infos
            if info.filename.lower().endswith(
                ".npz"
            )
        ]

        bscan_infos = [
            info
            for info
            in all_npz_infos
            if (
                "bscan"
                in [
                    x.lower()
                    for x
                    in PurePosixPath(
                        info.filename
                    ).parts
                ]
                and re.fullmatch(
                    r"data_\d{4}\.npz",
                    Path(
                        info.filename
                    ).name,
                    flags=re.IGNORECASE,
                )
            )
        ]

        bscan_infos = sorted(
            bscan_infos,
            key=lambda x:
                natural_key(
                    x.filename
                ),
        )

        bscan_ids = [
            numeric_id_from_stem(
                Path(
                    info.filename
                ).stem
            )
            for info in bscan_infos
        ]

        bscan_id_set = {
            x
            for x in bscan_ids
            if x is not None
        }

        duplicate_bscan_ids = [
            x
            for x, count
            in Counter(
                bscan_ids
            ).items()
            if count > 1
        ]

        missing_bscan_ids = sorted(
            expected_ids
            - bscan_id_set
        )

        extra_bscan_ids = sorted(
            bscan_id_set
            - expected_ids
        )

        add_check(
            dataset,
            "raw_oct_bscan_count",
            (
                "PASS"
                if len(bscan_infos)
                == 1000
                else "FAIL"
            ),
            len(bscan_infos),
            1000,
        )

        add_check(
            dataset,
            "raw_oct_bscan_id_coverage",
            (
                "PASS"
                if (
                    not missing_bscan_ids
                    and not extra_bscan_ids
                )
                else "FAIL"
            ),
            {
                "missing":
                    missing_bscan_ids,
                "extra":
                    extra_bscan_ids,
            },
            "1..1000",
        )

        add_check(
            dataset,
            "raw_oct_duplicate_ids",
            (
                "PASS"
                if not duplicate_bscan_ids
                else "FAIL"
            ),
            duplicate_bscan_ids,
            [],
        )

        other_npz_count = (
            len(all_npz_infos)
            - len(bscan_infos)
        )

        add_check(
            dataset,
            "other_npz_files_in_archive",
            "INFO",
            other_npz_count,
            "not used for raw OCT pairing",
            (
                "Only NPZ files inside "
                "Bscan/ are used for raw OCT pairing."
            ),
        )

        rnflt_files = sorted(
            [
                path
                for path
                in rnflt_dir.rglob(
                    "data_*.npz"
                )
                if re.fullmatch(
                    r"data_\d{4}\.npz",
                    path.name,
                    flags=re.IGNORECASE,
                )
            ],
            key=lambda p:
                natural_key(
                    p.name
                ),
        )

        rnflt_ids = [
            numeric_id_from_stem(
                path.stem
            )
            for path
            in rnflt_files
        ]

        rnflt_id_set = {
            x
            for x
            in rnflt_ids
            if x is not None
        }

        duplicate_rnflt_ids = [
            x
            for x, count
            in Counter(
                rnflt_ids
            ).items()
            if count > 1
        ]

        missing_rnflt_ids = sorted(
            expected_ids
            - rnflt_id_set
        )

        extra_rnflt_ids = sorted(
            rnflt_id_set
            - expected_ids
        )

        add_check(
            dataset,
            "rnflt_count",
            (
                "PASS"
                if len(rnflt_files)
                == 1000
                else "FAIL"
            ),
            len(rnflt_files),
            1000,
        )

        add_check(
            dataset,
            "rnflt_id_coverage",
            (
                "PASS"
                if (
                    not missing_rnflt_ids
                    and not extra_rnflt_ids
                )
                else "FAIL"
            ),
            {
                "missing":
                    missing_rnflt_ids,
                "extra":
                    extra_rnflt_ids,
            },
            "1..1000",
        )

        add_check(
            dataset,
            "rnflt_duplicate_ids",
            (
                "PASS"
                if not duplicate_rnflt_ids
                else "FAIL"
            ),
            duplicate_rnflt_ids,
            [],
        )

        raw_by_id = {
            numeric_id_from_stem(
                Path(
                    info.filename
                ).stem
            ):
                info
            for info
            in bscan_infos
        }

        rnflt_by_id = {
            numeric_id_from_stem(
                path.stem
            ):
                path
            for path
            in rnflt_files
        }

        sample_indices = sorted(
            set(
                [
                    0,
                    len(bscan_infos) // 2,
                    len(bscan_infos) - 1,
                ]
            )
        )

        raw_schema_samples = []

        for index in sample_indices:

            info = bscan_infos[
                index
            ]

            schema = (
                read_npz_schema_from_zip(
                    zf,
                    info.filename,
                )
            )

            raw_schema_samples.append(
                {
                    "member":
                        info.filename,
                    "schema":
                        schema,
                }
            )

        schemas[
            "harvard_gdp_raw_oct"
        ] = raw_schema_samples

        if raw_schema_samples:

            required_fields = {
                "bscans"
            }

            missing_fields = (
                required_fields
                - set(
                    raw_schema_samples[0][
                        "schema"
                    ]["fields"]
                )
            )

            add_check(
                dataset,
                "raw_oct_required_fields",
                (
                    "PASS"
                    if not missing_fields
                    else "FAIL"
                ),
                raw_schema_samples[0][
                    "schema"
                ]["fields"],
                sorted(
                    required_fields
                ),
                {
                    "missing":
                        sorted(
                            missing_fields
                        )
                },
            )

            raw_shape_ok = all(
                sample[
                    "schema"
                ].get(
                    "bscans",
                    {}
                ).get(
                    "shape"
                )
                == EXPECTED[
                    dataset
                ]["raw_oct_shape"]
                for sample
                in raw_schema_samples
            )

            add_check(
                dataset,
                "raw_oct_shape_samples",
                (
                    "PASS"
                    if raw_shape_ok
                    else "FAIL"
                ),
                [
                    sample[
                        "schema"
                    ].get(
                        "bscans",
                        {}
                    ).get(
                        "shape"
                    )
                    for sample
                    in raw_schema_samples
                ],
                EXPECTED[
                    dataset
                ]["raw_oct_shape"],
            )

    first_rnflt_schema = None

    if rnflt_files:

        first_rnflt = rnflt_files[0]

        with np.load(
            first_rnflt,
            allow_pickle=False,
        ) as data:

            first_rnflt_schema = {
                "fields":
                    list(
                        data.files
                    )
            }

            for key in data.files:

                array = np.asarray(
                    data[key]
                )

                first_rnflt_schema[
                    key
                ] = {
                    "shape":
                        list(
                            array.shape
                        ),
                    "dtype":
                        str(
                            array.dtype
                        ),
                }

        schemas[
            "harvard_gdp_rnflt"
        ] = {
            "file":
                str(
                    first_rnflt
                ),
            "schema":
                first_rnflt_schema,
        }

    for sid in sorted(
        expected_ids
    ):

        raw_info = raw_by_id.get(
            sid
        )

        rnflt_path = rnflt_by_id.get(
            sid
        )

        row = {
            "dataset":
                dataset,
            "split":
                None,
            "sample_id":
                sid,
            "patient_id":
                None,
            "eye_id":
                None,
            "patient_id_status":
                "not_provided",
            "eye_id_status":
                "not_provided",
            "sample_locator":
                None,
            "oct_locator":
                None,
            "rnflt_locator":
                None,
            "oct_format":
                (
                    "npz_bscan_volume"
                    if raw_info
                    else None
                ),
            "rnflt_format":
                (
                    "npz_array"
                    if rnflt_path
                    else None
                ),
            "oct_member":
                (
                    raw_info.filename
                    if raw_info
                    else None
                ),
            "rnflt_member":
                (
                    str(
                        rnflt_path
                    )
                    if rnflt_path
                    else None
                ),
            "oct_shape":
                None,
            "rnflt_shape":
                None,
            "glaucoma_label":
                None,
            "progression_status":
                "not_labeled",
            "progression_label":
                None,
            "progression_md":
                None,
            "progression_vfi":
                None,
            "progression_td_pointwise":
                None,
            "progression_md_fast":
                None,
            "progression_md_fast_no_p_cut":
                None,
            "progression_td_pointwise_no_p_cut":
                None,
            "md":
                None,
            "age":
                None,
            "male":
                None,
            "race":
                None,
            "tds_json":
                None,
            "rnflt_min":
                None,
            "rnflt_max":
                None,
            "rnflt_mean":
                None,
            "rnflt_finite_fraction":
                None,
            "pair_status":
                "missing_pair",
        }

        if raw_info is not None:

            row[
                "sample_locator"
            ] = archive_locator(
                archive,
                raw_info.filename,
            )

            row[
                "oct_locator"
            ] = archive_locator(
                archive,
                raw_info.filename,
            )

            row[
                "oct_shape"
            ] = EXPECTED[
                dataset
            ]["raw_oct_shape"]

        if rnflt_path is not None:

            row[
                "rnflt_locator"
            ] = str(
                rnflt_path
            )

            try:

                with np.load(
                    rnflt_path,
                    allow_pickle=False,
                ) as data:

                    if "rnflt" in data.files:

                        rnflt = np.asarray(
                            data[
                                "rnflt"
                            ]
                        )

                        row[
                            "rnflt_shape"
                        ] = list(
                            rnflt.shape
                        )

                        if (
                            list(
                                rnflt.shape
                            )
                            != EXPECTED[
                                dataset
                            ]["rnflt_shape"]
                        ):

                            bad_rnflt_shapes.append(
                                {
                                    "sample_id":
                                        sid,
                                    "shape":
                                        list(
                                            rnflt.shape
                                        ),
                                }
                            )

                        row[
                            "rnflt_min"
                        ] = float(
                            np.nanmin(
                                rnflt
                            )
                        )

                        row[
                            "rnflt_max"
                        ] = float(
                            np.nanmax(
                                rnflt
                            )
                        )

                        row[
                            "rnflt_mean"
                        ] = float(
                            np.nanmean(
                                rnflt
                            )
                        )

                        row[
                            "rnflt_finite_fraction"
                        ] = float(
                            np.isfinite(
                                rnflt
                            ).mean()
                        )

                    if "glaucoma" in data.files:

                        label = normalize_binary(
                            scalar_value(
                                data[
                                    "glaucoma"
                                ]
                            )
                        )

                        row[
                            "glaucoma_label"
                        ] = label

                        if label is not None:
                            glaucoma_counts[
                                label
                            ] += 1

                    if "md" in data.files:

                        row[
                            "md"
                        ] = scalar_value(
                            data["md"]
                        )

                    if "age" in data.files:

                        row[
                            "age"
                        ] = scalar_value(
                            data["age"]
                        )

                    if "male" in data.files:

                        row[
                            "male"
                        ] = scalar_value(
                            data["male"]
                        )

                    if "race" in data.files:

                        row[
                            "race"
                        ] = scalar_value(
                            data["race"]
                        )

                    if "tds" in data.files:

                        row[
                            "tds_json"
                        ] = safe_json(
                            np.asarray(
                                data[
                                    "tds"
                                ]
                            ).tolist()
                        )

                    if "progression" in data.files:

                        progression_values, valid = (
                            parse_progression_vector(
                                data[
                                    "progression"
                                ]
                            )
                        )

                        if valid:

                            row[
                                "progression_status"
                            ] = "labeled"

                            row[
                                "progression_label"
                            ] = safe_json(
                                progression_values
                            )

                            for index, name in enumerate(
                                PROGRESSION_NAMES
                            ):

                                row[
                                    f"progression_{name}"
                                ] = (
                                    progression_values[
                                        index
                                    ]
                                )

                            for index, name in enumerate(
                                PROGRESSION_NAMES
                            ):

                                progression_counts[
                                    name,
                                    progression_values[
                                        index
                                    ]
                                ] += 1

                        else:

                            row[
                                "progression_status"
                            ] = "not_labeled"

                            if sid in expected_progression_ids:

                                malformed_progression.append(
                                    {
                                        "sample_id":
                                            sid,
                                        "raw_value":
                                            scalar_value(
                                                data[
                                                    "progression"
                                                ]
                                            ),
                                    }
                                )

                            elif sid in expected_unlabeled_ids:

                                unlabeled_progression_ids.append(
                                    sid
                                )

                            else:

                                malformed_progression.append(
                                    {
                                        "sample_id":
                                            sid,
                                        "raw_value":
                                            scalar_value(
                                                data[
                                                    "progression"
                                                ]
                                            ),
                                    }
                                )

            except Exception as exc:

                add_check(
                    dataset,
                    f"rnflt_read_{sid}",
                    "FAIL",
                    str(exc),
                    None,
                )

        if (
            raw_info is not None
            and rnflt_path is not None
        ):

            row[
                "pair_status"
            ] = "paired"

        manifest_rows.append(
            row
        )

    manifest = pd.DataFrame(
        manifest_rows
    )

    paired_count = int(
        (
            manifest[
                "pair_status"
            ]
            == "paired"
        ).sum()
    )

    add_check(
        dataset,
        "raw_oct_rnflt_pairing",
        (
            "PASS"
            if paired_count
            == 1000
            else "FAIL"
        ),
        paired_count,
        1000,
    )

    add_check(
        dataset,
        "rnflt_shape_all_samples",
        (
            "PASS"
            if not bad_rnflt_shapes
            else "FAIL"
        ),
        (
            "all_expected"
            if not bad_rnflt_shapes
            else bad_rnflt_shapes[:20]
        ),
        EXPECTED[
            dataset
        ]["rnflt_shape"],
    )

    progression_labeled_ids = (
        manifest.loc[
            manifest[
                "progression_status"
            ]
            == "labeled",
            "sample_id",
        ]
        .astype(int)
        .tolist()
    )

    observed_progression_labeled_ids = set(
        progression_labeled_ids
    )

    add_check(
        dataset,
        "progression_labeled_samples",
        (
            "PASS"
            if len(
                progression_labeled_ids
            )
            == EXPECTED[
                dataset
            ]["progression_labeled"]
            else "FAIL"
        ),
        len(
            progression_labeled_ids
        ),
        EXPECTED[
            dataset
        ]["progression_labeled"],
    )

    add_check(
        dataset,
        "progression_label_id_coverage",
        (
            "PASS"
            if (
                observed_progression_labeled_ids
                == expected_progression_ids
            )
            else "FAIL"
        ),
        {
            "min":
                min(
                    progression_labeled_ids
                )
                if progression_labeled_ids
                else None,
            "max":
                max(
                    progression_labeled_ids
                )
                if progression_labeled_ids
                else None,
            "missing":
                sorted(
                    expected_progression_ids
                    - observed_progression_labeled_ids
                ),
            "extra":
                sorted(
                    observed_progression_labeled_ids
                    - expected_progression_ids
                ),
        },
        (
            f"{EXPECTED[dataset]['progression_labeled_start']}"
            f".."
            f"{EXPECTED[dataset]['progression_labeled_end']}"
        ),
    )

    observed_unlabeled_ids = set(
        unlabeled_progression_ids
    )

    add_check(
        dataset,
        "progression_unlabeled_scope",
        (
            "PASS"
            if (
                observed_unlabeled_ids
                == expected_unlabeled_ids
            )
            else "FAIL"
        ),
        {
            "count":
                len(
                    observed_unlabeled_ids
                ),
            "min":
                min(
                    observed_unlabeled_ids
                )
                if observed_unlabeled_ids
                else None,
            "max":
                max(
                    observed_unlabeled_ids
                )
                if observed_unlabeled_ids
                else None,
            "missing":
                sorted(
                    expected_unlabeled_ids
                    - observed_unlabeled_ids
                ),
            "extra":
                sorted(
                    observed_unlabeled_ids
                    - expected_unlabeled_ids
                ),
        },
        (
            f"{EXPECTED[dataset]['progression_unlabeled_start']}"
            f".."
            f"{EXPECTED[dataset]['progression_unlabeled_end']}"
        ),
    )

    add_check(
        dataset,
        "progression_malformed_values",
        (
            "PASS"
            if not malformed_progression
            else "FAIL"
        ),
        len(
            malformed_progression
        ),
        0,
        (
            None
            if not malformed_progression
            else malformed_progression[:20]
        ),
    )

    manifest_path = (
        AUDIT_DIR
        / "harvard_gdp_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False,
    )

    progression_distribution = []

    for name in PROGRESSION_NAMES:

        count_0 = progression_counts[
            name,
            0
        ]

        count_1 = progression_counts[
            name,
            1
        ]

        progression_distribution.append(
            {
                "progression_label":
                    name,
                "negative":
                    count_0,
                "positive":
                    count_1,
                "total":
                    count_0 + count_1,
            }
        )

    pd.DataFrame(
        progression_distribution
    ).to_csv(
        AUDIT_DIR
        / "harvard_gdp_progression_distribution.csv",
        index=False,
    )

    pd.DataFrame(
        [
            {
                "label":
                    k,
                "count":
                    v,
            }
            for k, v
            in glaucoma_counts.items()
        ]
    ).to_csv(
        AUDIT_DIR
        / "harvard_gdp_glaucoma_distribution.csv",
        index=False,
    )

    add_summary_row(
        dataset,
        {
            "samples_observed":
                len(manifest),
            "samples_expected":
                1000,
            "paired_samples":
                paired_count,
            "progression_labeled":
                len(
                    progression_labeled_ids
                ),
            "progression_unlabeled":
                len(
                    observed_unlabeled_ids
                ),
            "identity_level":
                "sample_only",
            "patient_id":
                "not_provided",
            "eye_id":
                "not_provided",
            "primary_oct":
                "3D B-scan volume",
            "primary_rnflt":
                "225x225 RNFLT map",
            "glaucoma_label_distribution":
                dict(
                    glaucoma_counts
                ),
            "manifest":
                str(
                    manifest_path
                ),
        },
    )

    return manifest


# ============================================================
# GOALS
# ============================================================

def read_excel_from_zip(
    zf,
    member,
):
    raw = zf.read(
        member
    )

    return pd.read_excel(
        io.BytesIO(raw),
        engine="openpyxl",
    )


def audit_goals():

    dataset = "goals"

    archive = ensure_archive_exists(
        dataset
    )

    manifest_rows = []
    label_distribution = []
    structure_report = {}
    discrepancy_report = []

    split_cfg = {
        "Train": {
            "reference_start": 1,
            "reference_end": 100,
            "usable_expected": 100,
        },
        "Validation": {
            "reference_start": 101,
            "reference_end": 200,
            "usable_expected": 100,
        },
        "Test": {
            "reference_start": 201,
            "reference_end": 300,
            "usable_expected": 99,
        },
    }

    with zipfile.ZipFile(
        archive,
        "r",
    ) as zf:

        infos = [
            info
            for info
            in zf.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]

        for split, cfg in (
            split_cfg.items()
        ):

            image_infos = [
                info
                for info
                in infos
                if (
                    f"GOALS/{split}/Image/"
                    in info.filename
                    and info.filename.lower()
                    .endswith(".png")
                )
            ]

            mask_infos = [
                info
                for info
                in infos
                if (
                    f"GOALS/{split}/Layer_Masks/"
                    in info.filename
                    and info.filename.lower()
                    .endswith(".png")
                )
            ]

            show_infos = [
                info
                for info
                in infos
                if (
                    f"GOALS/{split}/Layer_show/"
                    in info.filename
                    and info.filename.lower()
                    .endswith(".png")
                )
            ]

            xlsx_members = [
                info.filename
                for info
                in infos
                if (
                    f"GOALS/{split}/"
                    in info.filename
                    and info.filename.lower()
                    .endswith(".xlsx")
                )
            ]

            image_map = {
                numeric_id_from_stem(
                    Path(
                        info.filename
                    ).stem
                ):
                    info.filename
                for info
                in image_infos
            }

            mask_map = {
                numeric_id_from_stem(
                    Path(
                        info.filename
                    ).stem
                ):
                    info.filename
                for info
                in mask_infos
            }

            show_map = {
                numeric_id_from_stem(
                    Path(
                        info.filename
                    ).stem
                ):
                    info.filename
                for info
                in show_infos
            }

            reference_ids = set(
                range(
                    cfg[
                        "reference_start"
                    ],
                    cfg[
                        "reference_end"
                    ] + 1,
                )
            )

            image_ids = set(
                image_map
            )

            mask_ids = set(
                mask_map
            )

            show_ids = set(
                show_map
            )

            missing_reference = sorted(
                reference_ids
                - image_ids
            )

            extra_images = sorted(
                image_ids
                - reference_ids
            )

            missing_masks = sorted(
                image_ids
                - mask_ids
            )

            missing_show = sorted(
                image_ids
                - show_ids
            )

            add_check(
                dataset,
                f"{split}_usable_sample_count",
                (
                    "PASS"
                    if len(image_ids)
                    == cfg[
                        "usable_expected"
                    ]
                    else "FAIL"
                ),
                len(image_ids),
                cfg[
                    "usable_expected"
                ],
            )

            add_check(
                dataset,
                f"{split}_image_mask_pairing",
                (
                    "PASS"
                    if image_ids
                    == mask_ids
                    else "FAIL"
                ),
                {
                    "images":
                        len(image_ids),
                    "masks":
                        len(mask_ids),
                    "missing_masks":
                        missing_masks,
                },
                "same IDs",
            )

            add_check(
                dataset,
                f"{split}_layer_show_pairing",
                (
                    "PASS"
                    if image_ids
                    == show_ids
                    else "FAIL"
                ),
                {
                    "images":
                        len(image_ids),
                    "layer_show":
                        len(show_ids),
                    "missing_layer_show":
                        missing_show,
                },
                "same IDs",
            )

            if (
                missing_reference
                or extra_images
            ):

                discrepancy = {
                    "split":
                        split,
                    "reference_expected_count":
                        len(
                            reference_ids
                        ),
                    "usable_observed_count":
                        len(
                            image_ids
                        ),
                    "missing_reference_ids":
                        missing_reference,
                    "extra_image_ids":
                        extra_images,
                    "severity":
                        "documented_data_discrepancy",
                }

                discrepancy_report.append(
                    discrepancy
                )

                add_check(
                    dataset,
                    f"{split}_reference_coverage",
                    "INFO",
                    {
                        "missing":
                            missing_reference,
                        "extra":
                            extra_images,
                    },
                    f"{cfg['reference_start']}..{cfg['reference_end']}",
                    "Reference set contains an ID not present in the supplied archive.",
                )

            label_map = {}

            if not xlsx_members:

                add_check(
                    dataset,
                    f"{split}_label_file",
                    "FAIL",
                    None,
                    "GC label Excel file",
                )

            else:

                label_member = sorted(
                    xlsx_members,
                    key=natural_key,
                )[0]

                label_df = (
                    read_excel_from_zip(
                        zf,
                        label_member,
                    )
                )

                required_columns = {
                    "ImgName",
                    "GC_Label",
                }

                missing_columns = sorted(
                    required_columns
                    - set(
                        label_df.columns
                    )
                )

                add_check(
                    dataset,
                    f"{split}_label_schema",
                    (
                        "PASS"
                        if not missing_columns
                        else "FAIL"
                    ),
                    label_df.columns.tolist(),
                    [
                        "ImgName",
                        "GC_Label",
                    ],
                    {
                        "missing":
                            missing_columns
                    },
                )

                if not missing_columns:

                    for _, record in (
                        label_df.iterrows()
                    ):

                        sid = normalize_id(
                            record[
                                "ImgName"
                            ]
                        )

                        if sid is None:
                            continue

                        label_value = record[
                            "GC_Label"
                        ]

                        if pd.isna(
                            label_value
                        ):
                            label_map[
                                sid
                            ] = None
                        else:
                            label_map[
                                sid
                            ] = int(
                                label_value
                            )

                    invalid_labels = [
                        sid
                        for sid, value
                        in label_map.items()
                        if value not in {
                            0,
                            1,
                        }
                    ]

                    add_check(
                        dataset,
                        f"{split}_label_count",
                        (
                            "PASS"
                            if len(
                                label_map
                            )
                            == len(
                                image_ids
                            )
                            else "FAIL"
                        ),
                        len(
                            label_map
                        ),
                        len(
                            image_ids
                        ),
                    )

                    add_check(
                        dataset,
                        f"{split}_label_id_pairing",
                        (
                            "PASS"
                            if set(
                                label_map
                            )
                            == image_ids
                            else "FAIL"
                        ),
                        {
                            "labels":
                                len(
                                    label_map
                                ),
                            "images":
                                len(
                                    image_ids
                                ),
                        },
                        "same IDs",
                    )

                    add_check(
                        dataset,
                        f"{split}_label_values",
                        (
                            "PASS"
                            if not invalid_labels
                            else "FAIL"
                        ),
                        sorted(
                            set(
                                label_map.values()
                            )
                        ),
                        [
                            0,
                            1,
                        ],
                        {
                            "invalid_ids":
                                invalid_labels
                        },
                    )

                    counter = Counter(
                        value
                        for value
                        in label_map.values()
                        if value is not None
                    )

                    for label, count in (
                        counter.items()
                    ):
                        label_distribution.append(
                            {
                                "split":
                                    split,
                                "gc_label":
                                    label,
                                "count":
                                    count,
                            }
                        )

            structure_report[
                split
            ] = {
                "reference_count":
                    len(
                        reference_ids
                    ),
                "usable_image_count":
                    len(
                        image_ids
                    ),
                "layer_mask_count":
                    len(
                        mask_ids
                    ),
                "layer_show_count":
                    len(
                        show_ids
                    ),
                "xlsx_files":
                    xlsx_members,
                "missing_reference_ids":
                    missing_reference,
                "extra_image_ids":
                    extra_images,
                "missing_layer_mask_ids":
                    missing_masks,
                "missing_layer_show_ids":
                    missing_show,
            }

            sample_ids_for_schema = (
                sorted(
                    image_ids,
                )
            )

            if sample_ids_for_schema:

                positions = sorted(
                    set(
                        [
                            0,
                            len(
                                sample_ids_for_schema
                            ) // 2,
                            len(
                                sample_ids_for_schema
                            ) - 1,
                        ]
                    )
                )

                split_schema = []

                for position in positions:

                    sid = (
                        sample_ids_for_schema[
                            position
                        ]
                    )

                    image_member = (
                        image_map[
                            sid
                        ]
                    )

                    image = (
                        open_image_from_zip(
                            zf,
                            image_member,
                        )
                    )

                    entry = {
                        "sample_id":
                            sid,
                        "image_member":
                            image_member,
                        "image_schema":
                            image_schema(
                                image
                            ),
                    }

                    if sid in mask_map:

                        mask = (
                            open_image_from_zip(
                                zf,
                                mask_map[sid],
                            )
                        )

                        entry[
                            "layer_mask_member"
                        ] = mask_map[sid]

                        entry[
                            "layer_mask_schema"
                        ] = image_schema(
                            mask
                        )

                        entry[
                            "image_mask_shape_match"
                        ] = (
                            image.size
                            == mask.size
                        )

                    split_schema.append(
                        entry
                    )

                schemas[
                    f"goals_{split.lower()}"
                ] = split_schema

                shape_mismatches = [
                    x
                    for x
                    in split_schema
                    if (
                        x.get(
                            "image_mask_shape_match"
                        )
                        is False
                    )
                ]

                add_check(
                    dataset,
                    f"{split}_image_mask_shape",
                    (
                        "PASS"
                        if not shape_mismatches
                        else "FAIL"
                    ),
                    (
                        "matched"
                        if not shape_mismatches
                        else shape_mismatches
                    ),
                    "same image/mask dimensions",
                )

            for sid in sorted(
                image_ids
            ):

                image_member = (
                    image_map[
                        sid
                    ]
                )

                mask_member = (
                    mask_map.get(
                        sid
                    )
                )

                show_member = (
                    show_map.get(
                        sid
                    )
                )

                manifest_rows.append(
                    {
                        "dataset":
                            dataset,
                        "split":
                            split,
                        "sample_id":
                            sid,
                        "patient_id":
                            None,
                        "eye_id":
                            None,
                        "patient_id_status":
                            "not_provided",
                        "eye_id_status":
                            "not_provided",
                        "sample_locator":
                            archive_locator(
                                archive,
                                image_member,
                            ),
                        "oct_locator":
                            archive_locator(
                                archive,
                                image_member,
                            ),
                        "rnflt_locator":
                            None,
                        "oct_format":
                            "2D_oct_image",
                        "oct_member":
                            image_member,
                        "layer_mask_locator":
                            (
                                archive_locator(
                                    archive,
                                    mask_member,
                                )
                                if mask_member
                                else None
                            ),
                        "layer_mask_member":
                            mask_member,
                        "layer_show_locator":
                            (
                                archive_locator(
                                    archive,
                                    show_member,
                                )
                                if show_member
                                else None
                            ),
                        "layer_show_member":
                            show_member,
                        "layer_annotation_status":
                            (
                                "paired"
                                if mask_member
                                else "missing"
                            ),
                        "gc_label":
                            label_map.get(
                                sid
                            ),
                    }
                )

    manifest = pd.DataFrame(
        manifest_rows
    )

    manifest_path = (
        AUDIT_DIR
        / "goals_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False,
    )

    save_json(
        AUDIT_DIR
        / "goals_structure.json",
        structure_report,
    )

    save_json(
        AUDIT_DIR
        / "goals_discrepancies.json",
        {
            "reference_total":
                300,
            "usable_total":
                299,
            "discrepancies":
                discrepancy_report,
            "explicit_missing_test_id":
                300,
        },
    )

    pd.DataFrame(
        label_distribution
    ).to_csv(
        AUDIT_DIR
        / "goals_gc_label_distribution.csv",
        index=False,
    )

    add_summary_row(
        dataset,
        {
            "samples_observed":
                len(manifest),
            "samples_expected":
                299,
            "reference_nominal_samples":
                300,
            "identity_level":
                "sample_only",
            "patient_id":
                "not_provided",
            "eye_id":
                "not_provided",
            "primary_oct":
                "2D OCT image",
            "layer_annotation":
                "Layer_Masks",
            "layer_show":
                "auxiliary/visualization file",
            "missing_reference_samples":
                "Test/300",
            "manifest":
                str(
                    manifest_path
                ),
        },
    )

    return manifest


# ============================================================
# GAMMA
# ============================================================

def extract_nested_zip(
    outer_archive,
    nested_member,
    output_path,
):
    output_path = Path(
        output_path
    )

    if output_path.exists():
        output_path.unlink()

    with zipfile.ZipFile(
        outer_archive,
        "r",
    ) as outer_zip:

        info = outer_zip.getinfo(
            nested_member
        )

        with outer_zip.open(
            info,
            "r",
        ) as source:

            with open(
                output_path,
                "wb",
            ) as target:

                shutil.copyfileobj(
                    source,
                    target,
                    length=64 * 1024 * 1024,
                )

    if not zipfile.is_zipfile(
        output_path
    ):
        raise RuntimeError(
            f"Invalid nested ZIP:\n"
            f"{output_path}"
        )

    return output_path


def find_multimodality_root(
    members,
):
    candidates = []

    for member in members:

        parts = (
            PurePosixPath(
                member
            ).parts
        )

        for i, part in enumerate(
            parts
        ):

            if (
                part.lower()
                == "multi-modality_images"
            ):

                candidates.append(
                    parts[
                        : i + 1
                    ]
                )

                break

    if not candidates:
        return None

    return list(
        max(
            Counter(
                tuple(x)
                for x
                in candidates
            ).items(),
            key=lambda x:
                x[1],
        )[0]
    )


def parse_gamma_label_file(
    nested_zip,
    xlsx_members,
):
    candidates = [
        member
        for member
        in xlsx_members
        if (
            "glaucoma_grading"
            in member.lower()
        )
    ]

    if not candidates:
        return (
            {},
            None,
            None,
            None,
        )

    label_member = sorted(
        candidates,
        key=natural_key,
    )[0]

    with zipfile.ZipFile(
        nested_zip,
        "r",
    ) as zf:

        df = pd.read_excel(
            io.BytesIO(
                zf.read(
                    label_member
                )
            ),
            engine="openpyxl",
        )

    required = {
        "data",
        "non",
        "early",
        "mid_advanced",
    }

    missing = sorted(
        required
        - set(
            df.columns
        )
    )

    if missing:
        return (
            {},
            label_member,
            df.columns.tolist(),
            {
                "missing_columns":
                    missing
            },
        )

    label_map = {}
    invalid_rows = []

    for _, record in (
        df.iterrows()
    ):

        sid = normalize_id(
            record[
                "data"
            ]
        )

        if sid is None:
            continue

        values = []

        for column in [
            "non",
            "early",
            "mid_advanced",
        ]:

            normalized = normalize_binary(
                record[
                    column
                ]
            )

            values.append(
                normalized
            )

        if any(
            value is None
            for value in values
        ):

            invalid_rows.append(
                sid
            )
            continue

        if sum(values) != 1:

            invalid_rows.append(
                sid
            )
            continue

        class_index = int(
            np.argmax(
                values
            )
        )

        grade_names = [
            "non",
            "early",
            "mid_advanced",
        ]

        grade = (
            grade_names[
                class_index
            ]
        )

        label_map[
            sid
        ] = {
            "grade":
                grade,
            "grade_index":
                class_index,
            "glaucoma_binary":
                0
                if grade == "non"
                else 1,
            "one_hot": values,
        }

    label_info = {
        "rows":
            len(df),
        "valid_rows":
            len(label_map),
        "invalid_rows":
            invalid_rows,
        "columns":
            df.columns.tolist(),
    }

    return (
        label_map,
        label_member,
        df.columns.tolist(),
        label_info,
    )


def parse_gamma_samples(
    nested_zip,
    outer_archive,
    nested_member,
    split,
):

    with zipfile.ZipFile(
        nested_zip,
        "r",
    ) as zf:

        infos = [
            info
            for info
            in zf.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]

        members = [
            info.filename
            for info
            in infos
        ]

        root = find_multimodality_root(
            members
        )

        if root is None:
            raise RuntimeError(
                f"GAMMA {split}: "
                "multi-modality_images not found."
            )

        root_len = len(
            root
        )

        sample_files = defaultdict(
            list
        )

        for info in infos:

            parts = PurePosixPath(
                info.filename
            ).parts

            if len(parts) <= root_len:
                continue

            if tuple(
                parts[:root_len]
            ) != tuple(root):
                continue

            sample_component = parts[
                root_len
            ]

            if not re.fullmatch(
                r"\d{4}",
                sample_component,
            ):
                continue

            sid = int(
                sample_component
            )

            sample_files[
                sid
            ].append(
                info
            )

        xlsx_members = sorted(
            [
                info.filename
                for info
                in infos
                if info.filename.lower()
                .endswith(".xlsx")
            ],
            key=natural_key,
        )

        (
            label_map,
            label_member,
            label_columns,
            label_info,
        ) = parse_gamma_label_file(
            nested_zip,
            xlsx_members,
        )

        rows = []

        ambiguous_oct_samples = []

        for sid in sorted(
            sample_files
        ):

            infos_for_sample = (
                sample_files[
                    sid
                ]
            )

            image_infos = [
                info
                for info
                in infos_for_sample
                if is_image_member(
                    info.filename
                )
            ]

            sample_dir_name = f"{sid:04d}"

            direct_fundus = [
                info
                for info
                in image_infos
                if (
                    len(
                        PurePosixPath(
                            info.filename
                        ).parts
                    )
                    == root_len + 2
                )
            ]

            direct_fundus_exact = [
                info
                for info
                in direct_fundus
                if (
                    Path(
                        info.filename
                    ).stem
                    == sample_dir_name
                )
            ]

            if direct_fundus_exact:

                fundus_info = sorted(
                    direct_fundus_exact,
                    key=lambda x:
                        natural_key(
                            x.filename
                        ),
                )[0]

            elif direct_fundus:

                fundus_info = sorted(
                    direct_fundus,
                    key=lambda x:
                        natural_key(
                            x.filename
                        ),
                )[0]

            else:

                fundus_info = None

            oct_groups = defaultdict(
                list
            )

            for info in image_infos:

                parts = PurePosixPath(
                    info.filename
                ).parts

                relative = parts[
                    root_len + 1:
                ]

                if len(relative) <= 1:
                    continue

                child = relative[0]

                if len(relative) >= 2:

                    oct_groups[
                        child
                    ].append(
                        info
                    )

            exact_oct_group = (
                oct_groups.get(
                    sample_dir_name
                )
            )

            if exact_oct_group:

                oct_infos = exact_oct_group

            elif len(
                oct_groups
            ) == 1:

                oct_infos = next(
                    iter(
                        oct_groups.values()
                    )
                )

            else:

                sorted_groups = sorted(
                    oct_groups.values(),
                    key=len,
                    reverse=True,
                )

                if (
                    sorted_groups
                    and
                    len(
                        sorted_groups[0]
                    ) >= 200
                ):

                    oct_infos = (
                        sorted_groups[0]
                    )

                    ambiguous_oct_samples.append(
                        sid
                    )

                else:

                    oct_infos = []

            oct_infos = sorted(
                oct_infos,
                key=lambda x:
                    natural_key(
                        x.filename
                    ),
            )

            fundus_member = (
                fundus_info.filename
                if fundus_info
                else None
            )

            oct_first = (
                oct_infos[0].filename
                if oct_infos
                else None
            )

            oct_last = (
                oct_infos[-1].filename
                if oct_infos
                else None
            )

            label = label_map.get(
                sid
            )

            row = {
                "dataset":
                    "gamma",
                "split":
                    split,
                "sample_id":
                    sid,
                "patient_id":
                    None,
                "eye_id":
                    None,
                "patient_id_status":
                    "not_provided",
                "eye_id_status":
                    "not_provided",
                "sample_locator":
                    (
                        nested_archive_locator(
                            outer_archive,
                            nested_member,
                            fundus_member
                            or (
                                oct_first
                                or ""
                            ),
                        )
                        if (
                            fundus_member
                            or oct_first
                        )
                        else None
                    ),
                "fundus_locator":
                    (
                        nested_archive_locator(
                            outer_archive,
                            nested_member,
                            fundus_member,
                        )
                        if fundus_member
                        else None
                    ),
                "fundus_member":
                    fundus_member,
                "oct_locator":
                    (
                        nested_archive_locator(
                            outer_archive,
                            nested_member,
                            oct_first,
                        )
                        if oct_first
                        else None
                    ),
                "oct_first_member":
                    oct_first,
                "oct_last_member":
                    oct_last,
                "oct_member_count":
                    len(
                        oct_infos
                    ),
                "oct_format":
                    (
                        "jpeg_sequence"
                        if oct_first
                        else None
                    ),
                "fundus_format":
                    (
                        Path(
                            fundus_member
                        ).suffix.lower()
                        if fundus_member
                        else None
                    ),
                "disc_cup_mask_member":
                    None,
                "glaucoma_grade":
                    (
                        label[
                            "grade"
                        ]
                        if label
                        else None
                    ),
                "glaucoma_grade_index":
                    (
                        label[
                            "grade_index"
                        ]
                        if label
                        else None
                    ),
                "glaucoma_binary_label":
                    (
                        label[
                            "glaucoma_binary"
                        ]
                        if label
                        else None
                    ),
                "glaucoma_one_hot":
                    (
                        safe_json(
                            label[
                                "one_hot"
                            ]
                        )
                        if label
                        else None
                    ),
                "label_status":
                    (
                        "labeled"
                        if label
                        else "missing"
                    ),
                "sample_status":
                    (
                        "PASS"
                        if (
                            fundus_member
                            and
                            len(
                                oct_infos
                            )
                            == EXPECTED[
                                "gamma"
                            ]["oct_bscans"]
                        )
                        else "CHECK"
                    ),
            }

            disc_masks = [
                info
                for info
                in infos
                if (
                    "Disc_Cup_Mask"
                    in PurePosixPath(
                        info.filename
                    ).parts
                    and
                    numeric_id_from_stem(
                        Path(
                            info.filename
                        ).stem
                    )
                    == sid
                    and is_image_member(
                        info.filename
                    )
                )
            ]

            if disc_masks:

                row[
                    "disc_cup_mask_member"
                ] = sorted(
                    [
                        info.filename
                        for info
                        in disc_masks
                    ],
                    key=natural_key,
                )[0]

            if fundus_member:

                fundus = (
                    open_image_from_zip(
                        zf,
                        fundus_member,
                    )
                )

                row[
                    "fundus_schema"
                ] = safe_json(
                    image_schema(
                        fundus
                    )
                )

            if oct_first:

                oct_slice = (
                    open_image_from_zip(
                        zf,
                        oct_first,
                    )
                )

                row[
                    "oct_sample_schema"
                ] = safe_json(
                    image_schema(
                        oct_slice
                    )
                )

            rows.append(
                row
            )

        return {
            "rows":
                rows,
            "total_files":
                len(
                    members
                ),
            "multimodality_root":
                "/".join(
                    root
                ),
            "xlsx_members":
                xlsx_members,
            "label_member":
                label_member,
            "label_columns":
                label_columns,
            "label_info":
                label_info,
            "ambiguous_oct_samples":
                ambiguous_oct_samples,
        }


def audit_gamma():

    dataset = "gamma"

    archive = ensure_archive_exists(
        dataset
    )

    manifest_rows = []
    structure_report = {
        "nested_archives": []
    }

    outer_zip = zipfile.ZipFile(
        archive,
        "r",
    )

    try:

        outer_infos = [
            info
            for info
            in outer_zip.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]

        nested_infos = [
            info
            for info
            in outer_infos
            if (
                Path(
                    info.filename
                ).name.lower()
                in {
                    "train.zip",
                    "validation.zip",
                    "test.zip",
                }
            )
        ]

        nested_infos = sorted(
            nested_infos,
            key=lambda x:
                natural_key(
                    x.filename
                ),
        )

        add_check(
            dataset,
            "nested_archive_count",
            (
                "PASS"
                if len(
                    nested_infos
                )
                == 3
                else "FAIL"
            ),
            len(
                nested_infos
            ),
            3,
        )

        for nested_info in (
            nested_infos
        ):

            split = (
                Path(
                    nested_info.filename
                ).stem.capitalize()
            )

            temp_nested = (
                TEMP_DIR
                / f"{split}.zip"
            )

            print(
                f"\nGAMMA {split}"
            )

            print(
                "Preparing temporary nested ZIP..."
            )

            extract_nested_zip(
                archive,
                nested_info.filename,
                temp_nested,
            )

            try:

                result = parse_gamma_samples(
                    temp_nested,
                    archive,
                    nested_info.filename,
                    split,
                )

                rows = result[
                    "rows"
                ]

                manifest_rows.extend(
                    rows
                )

                observed = len(
                    rows
                )

                expected = EXPECTED[
                    dataset
                ][
                    "split_samples"
                ][
                    split
                ]

                add_check(
                    dataset,
                    f"{split}_sample_count",
                    (
                        "PASS"
                        if observed
                        == expected
                        else "FAIL"
                    ),
                    observed,
                    expected,
                )

                valid_oct = sum(
                    1
                    for row
                    in rows
                    if row[
                        "oct_member_count"
                    ]
                    == 256
                )

                fundus_count = sum(
                    1
                    for row
                    in rows
                    if row[
                        "fundus_member"
                    ]
                    is not None
                )

                labeled_count = sum(
                    1
                    for row
                    in rows
                    if row[
                        "label_status"
                    ]
                    == "labeled"
                )

                add_check(
                    dataset,
                    f"{split}_oct_256_count",
                    (
                        "PASS"
                        if valid_oct
                        == observed
                        else "FAIL"
                    ),
                    valid_oct,
                    observed,
                )

                add_check(
                    dataset,
                    f"{split}_fundus_count",
                    (
                        "PASS"
                        if fundus_count
                        == observed
                        else "FAIL"
                    ),
                    fundus_count,
                    observed,
                )

                add_check(
                    dataset,
                    f"{split}_label_count",
                    (
                        "PASS"
                        if labeled_count
                        == observed
                        else "FAIL"
                    ),
                    labeled_count,
                    observed,
                    result[
                        "label_info"
                    ],
                )

                if result[
                    "ambiguous_oct_samples"
                ]:

                    add_check(
                        dataset,
                        f"{split}_oct_structure_ambiguity",
                        "CHECK",
                        result[
                            "ambiguous_oct_samples"
                        ],
                        [],
                    )

                class_counter = Counter(
                    row[
                        "glaucoma_grade"
                    ]
                    for row
                    in rows
                    if row[
                        "glaucoma_grade"
                    ]
                    is not None
                )

                structure_report[
                    "nested_archives"
                ].append(
                    {
                        "split":
                            split,
                        "member":
                            nested_info.filename,
                        "compressed_size_bytes":
                            nested_info.compress_size,
                        "uncompressed_size_bytes":
                            nested_info.file_size,
                        "total_files":
                            result[
                                "total_files"
                            ],
                        "multimodality_root":
                            result[
                                "multimodality_root"
                            ],
                        "xlsx_members":
                            result[
                                "xlsx_members"
                            ],
                        "label_member":
                            result[
                                "label_member"
                            ],
                        "label_columns":
                            result[
                                "label_columns"
                            ],
                        "label_info":
                            result[
                                "label_info"
                            ],
                        "class_distribution":
                            dict(
                                class_counter
                            ),
                        "ambiguous_oct_samples":
                            result[
                                "ambiguous_oct_samples"
                            ],
                    }
                )

            finally:

                if temp_nested.exists():
                    temp_nested.unlink()

    finally:

        outer_zip.close()

    manifest = pd.DataFrame(
        manifest_rows
    )

    add_check(
        dataset,
        "total_sample_count",
        (
            "PASS"
            if len(
                manifest
            )
            == EXPECTED[
                dataset
            ]["samples"]
            else "FAIL"
        ),
        len(
            manifest
        ),
        EXPECTED[
            dataset
        ]["samples"],
    )

    manifest_path = (
        AUDIT_DIR
        / "gamma_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False,
    )

    save_json(
        AUDIT_DIR
        / "gamma_structure.json",
        structure_report,
    )

    label_distribution_rows = []

    for nested_info in (
        structure_report[
            "nested_archives"
        ]
    ):

        split = nested_info[
            "split"
        ]

        for label, count in (
            nested_info[
                "class_distribution"
            ].items()
        ):

            label_distribution_rows.append(
                {
                    "split":
                        split,
                    "glaucoma_grade":
                        label,
                    "count":
                        count,
                }
            )

    pd.DataFrame(
        label_distribution_rows
    ).to_csv(
        AUDIT_DIR
        / "gamma_label_distribution.csv",
        index=False,
    )

    add_summary_row(
        dataset,
        {
            "samples_observed":
                len(manifest),
            "samples_expected":
                300,
            "identity_level":
                "sample_only",
            "patient_id":
                "not_provided_in_manifest",
            "eye_id":
                "not_provided_in_manifest",
            "primary_oct":
                "3D OCT image sequence",
            "oct_bscans_expected":
                256,
            "secondary_modality":
                "fundus",
            "primary_task":
                "three-class glaucoma grading",
            "manifest":
                str(
                    manifest_path
                ),
        },
    )

    return manifest


# ============================================================
# Retina OCT Glaucoma / Ishikawa
# ============================================================

def ishikawa_filename_label(
    stem,
):
    text = str(
        stem
    )

    for prefix, label in (
        KNOWN_ISHIKAWA_PREFIXES.items()
    ):

        if text.startswith(
            prefix
        ):
            return (
                label,
                prefix,
            )

    return (
        None,
        None,
    )


def audit_ishikawa():

    dataset = "retina_oct_glaucoma"

    archive = ensure_archive_exists(
        dataset
    )

    manifest_rows = []
    label_counter = Counter()
    unknown_prefixes = Counter()
    schema_samples = []

    with zipfile.ZipFile(
        archive,
        "r",
    ) as zf:

        infos = [
            info
            for info
            in zf.infolist()
            if (
                not info.is_dir()
                and is_real_zip_member(
                    info.filename
                )
            )
        ]

        image_infos = [
            info
            for info
            in infos
            if (
                "imagesTr"
                in PurePosixPath(
                    info.filename
                ).parts
                and info.filename.lower()
                .endswith(".mha")
            )
        ]

        image_infos = sorted(
            image_infos,
            key=lambda x:
                natural_key(
                    x.filename
                ),
        )

        add_check(
            dataset,
            "oct_sample_count",
            (
                "PASS"
                if len(
                    image_infos
                )
                == EXPECTED[
                    dataset
                ]["samples"]
                else "FAIL"
            ),
            len(
                image_infos
            ),
            EXPECTED[
                dataset
            ]["samples"],
        )

        sample_indices = sorted(
            set(
                [
                    0,
                    len(image_infos) // 2,
                    len(image_infos) - 1,
                ]
            )
        )

        for index in sample_indices:

            info = image_infos[
                index
            ]

            schema_samples.append(
                {
                    "member":
                        info.filename,
                    "mha_header":
                        read_mha_header(
                            zf,
                            info.filename,
                        ),
                }
            )

        schemas[
            dataset
        ] = schema_samples

        header_reference = None
        header_mismatches = []

        if schema_samples:

            header_reference = schema_samples[0][
                "mha_header"
            ]

            for sample in schema_samples[1:]:

                current = sample[
                    "mha_header"
                ]

                for key in [
                    "NDims",
                    "DimSize",
                    "ElementType",
                    "ElementSpacing",
                ]:

                    if (
                        current.get(key)
                        != header_reference.get(key)
                    ):

                        header_mismatches.append(
                            {
                                "member":
                                    sample[
                                        "member"
                                    ],
                                "field":
                                    key,
                                "reference":
                                    header_reference.get(
                                        key
                                    ),
                                "observed":
                                    current.get(
                                        key
                                    ),
                            }
                        )

        add_check(
            dataset,
            "mha_header_consistency_samples",
            (
                "PASS"
                if not header_mismatches
                else "CHECK"
            ),
            (
                "matched"
                if not header_mismatches
                else header_mismatches
            ),
            "same sampled metadata",
        )

        for info in image_infos:

            stem = Path(
                info.filename
            ).stem

            label, prefix = (
                ishikawa_filename_label(
                    stem
                )
            )

            if label is not None:

                label_counter[
                    label
                ] += 1

            else:

                unknown_prefix = (
                    stem.split(
                        "_",
                        1,
                    )[0]
                    + "_"
                )

                unknown_prefixes[
                    unknown_prefix
                ] += 1

            manifest_rows.append(
                {
                    "dataset":
                        dataset,
                    "split":
                        None,
                    "sample_id":
                        stem,
                    "patient_id":
                        None,
                    "eye_id":
                        None,
                    "patient_id_status":
                        "not_provided",
                    "eye_id_status":
                        "not_provided",
                    "sample_locator":
                        archive_locator(
                            archive,
                            info.filename,
                        ),
                    "oct_locator":
                        archive_locator(
                            archive,
                            info.filename,
                        ),
                    "oct_format":
                        "mha_3d_volume",
                    "oct_member":
                        info.filename,
                    "classification_label":
                        label,
                    "classification_label_source":
                        (
                            "official_filename_prefix"
                            if label
                            is not None
                            else "unknown"
                        ),
                    "filename_class_prefix":
                        prefix,
                }
            )

    add_check(
        dataset,
        "source_encoded_filename_labels",
        (
            "PASS"
            if not unknown_prefixes
            else "FAIL"
        ),
        {
            str(k):
                int(v)
            for k, v
            in label_counter.items()
        },
        "recognized POAG_/NORMAL_ prefixes",
        (
            None
            if not unknown_prefixes
            else dict(
                unknown_prefixes
            )
        ),
    )

    if (
        len(
            label_counter
        ) == 1
    ):

        only_label = next(
            iter(
                label_counter
            )
        )

        discrepancies.append(
            {
                "dataset":
                    dataset,
                "type":
                    "observed_class_distribution",
                "details":
                    (
                        "All scans in the supplied "
                        "archive use the same explicit "
                        "class prefix."
                    ),
                "distribution":
                    {
                        str(k):
                            int(v)
                        for k, v
                        in label_counter.items()
                    },
            }
        )

    manifest = pd.DataFrame(
        manifest_rows
    )

    manifest_path = (
        AUDIT_DIR
        / "retina_oct_glaucoma_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False,
    )

    pd.DataFrame(
        [
            {
                "classification_label":
                    label,
                "count":
                    count,
            }
            for label, count
            in sorted(
                label_counter.items()
            )
        ]
    ).to_csv(
        AUDIT_DIR
        / "retina_oct_glaucoma_label_distribution.csv",
        index=False,
    )

    add_summary_row(
        dataset,
        {
            "samples_observed":
                len(manifest),
            "samples_expected":
                884,
            "identity_level":
                "study/sample filename",
            "patient_id":
                "not_provided",
            "eye_id":
                "not_provided",
            "primary_oct":
                "3D MHA volume",
            "classification_definition":
                "0=Normal, 1=POAG",
            "observed_filename_label_distribution":
                dict(
                    label_counter
                ),
            "manifest":
                str(
                    manifest_path
                ),
        },
    )

    return manifest


# ============================================================
# Run audits
# ============================================================

print(
    "=" * 70
)

print(
    "Stage 2 - Final Dataset Audit"
)

print(
    "=" * 70
)

harvard_gf_manifest = audit_harvard_gf()

harvard_gdp_manifest = audit_harvard_gdp()

goals_manifest = audit_goals()

gamma_manifest = audit_gamma()

ishikawa_manifest = audit_ishikawa()

manifest_objects = {
    "harvard_gf":
        harvard_gf_manifest,
    "harvard_gdp":
        harvard_gdp_manifest,
    "goals":
        goals_manifest,
    "gamma":
        gamma_manifest,
    "retina_oct_glaucoma":
        ishikawa_manifest,
}


# ============================================================
# Unified manifest
# ============================================================

unified_manifest = pd.concat(
    list(
        manifest_objects.values()
    ),
    ignore_index=True,
    sort=False,
)

canonical_columns = [
    "dataset",
    "split",
    "sample_id",
    "patient_id",
    "eye_id",
    "patient_id_status",
    "eye_id_status",
    "sample_locator",
    "oct_locator",
    "oct_format",
    "oct_member",
    "oct_shape",
    "rnflt_locator",
    "rnflt_format",
    "rnflt_member",
    "rnflt_shape",
    "glaucoma_label",
    "glaucoma_grade",
    "glaucoma_grade_index",
    "glaucoma_binary_label",
    "gc_label",
    "classification_label",
    "classification_label_source",
    "filename_class_prefix",
    "progression_status",
    "progression_label",
    "progression_md",
    "progression_vfi",
    "progression_td_pointwise",
    "progression_md_fast",
    "progression_md_fast_no_p_cut",
    "progression_td_pointwise_no_p_cut",
    "md",
    "tds_json",
    "age",
    "male",
    "race",
    "hispanic",
    "language",
    "maritalstatus",
    "summary_gender",
    "summary_ethnicity",
    "summary_language",
    "summary_maritalstatus",
    "summary_use",
    "rnflt_min",
    "rnflt_max",
    "rnflt_mean",
    "rnflt_finite_fraction",
    "fundus_locator",
    "fundus_member",
    "fundus_format",
    "oct_first_member",
    "oct_last_member",
    "oct_member_count",
    "oct_dir",
    "disc_cup_mask_member",
    "layer_mask_locator",
    "layer_mask_member",
    "layer_show_locator",
    "layer_show_member",
    "layer_annotation_status",
    "image_schema",
    "layer_mask_schema",
    "fundus_schema",
    "oct_sample_schema",
    "sample_status",
    "label_status",
    "pair_status",
]

existing_canonical_columns = [
    column
    for column
    in canonical_columns
    if column
    in unified_manifest.columns
]

remaining_columns = [
    column
    for column
    in unified_manifest.columns
    if column
    not in existing_canonical_columns
]

unified_manifest = unified_manifest[
    existing_canonical_columns
    + remaining_columns
]

if "sample_id" in unified_manifest.columns:

    unified_manifest[
        "sample_id"
    ] = (
        unified_manifest[
            "sample_id"
        ]
        .astype("string")
    )

if "dataset" in unified_manifest.columns:

    unified_manifest[
        "dataset"
    ] = (
        unified_manifest[
            "dataset"
        ]
        .astype("string")
    )

if "split" in unified_manifest.columns:

    unified_manifest[
        "split"
    ] = (
        unified_manifest[
            "split"
        ]
        .astype("string")
    )

unified_manifest_path = (
    AUDIT_DIR
    / "unified_manifest.csv"
)

unified_manifest.to_csv(
    unified_manifest_path,
    index=False,
)


# ============================================================
# Unified manifest integrity
# ============================================================

expected_manifest_counts = {
    "harvard_gf":
        3300,
    "harvard_gdp":
        1000,
    "goals":
        299,
    "gamma":
        300,
    "retina_oct_glaucoma":
        884,
}

unified_counts = (
    unified_manifest[
        "dataset"
    ]
    .value_counts()
    .to_dict()
)

for dataset, expected_count in (
    expected_manifest_counts.items()
):

    observed = int(
        unified_counts.get(
            dataset,
            0,
        )
    )

    add_check(
        "unified_manifest",
        f"{dataset}_row_count",
        (
            "PASS"
            if observed
            == expected_count
            else "FAIL"
        ),
        observed,
        expected_count,
    )


add_check(
    "unified_manifest",
    "total_row_count",
    (
        "PASS"
        if len(
            unified_manifest
        )
        == sum(
            expected_manifest_counts.values()
        )
        else "FAIL"
    ),
    len(
        unified_manifest
    ),
    sum(
        expected_manifest_counts.values()
    ),
)


# ============================================================
# Dataset overview
# ============================================================

overview_df = pd.DataFrame(
    dataset_summaries
)

overview_path = (
    AUDIT_DIR
    / "dataset_overview.csv"
)

overview_df.to_csv(
    overview_path,
    index=False,
)


# ============================================================
# Validation checks
# ============================================================

checks_df = pd.DataFrame(
    checks
)

checks_path = (
    AUDIT_DIR
    / "validation_checks.csv"
)

checks_df.to_csv(
    checks_path,
    index=False,
)

status_counts = (
    checks_df[
        "status"
    ]
    .value_counts()
    .to_dict()
    if not checks_df.empty
    else {}
)


# ============================================================
# Discrepancies
# ============================================================

all_discrepancies = list(
    discrepancies
)

for item in [
    {
        "dataset":
            "goals",
        "type":
            "missing_reference_sample",
        "details":
            "Reference ID 300 is absent from the supplied Test archive.",
        "missing_id":
            300,
    }
]:
    all_discrepancies.append(
        item
    )

discrepancies_path = (
    AUDIT_DIR
    / "dataset_discrepancies.json"
)

save_json(
    discrepancies_path,
    {
        "count":
            len(
                all_discrepancies
            ),
        "items":
            all_discrepancies,
    },
)


# ============================================================
# Critical failures and checks
# ============================================================

critical_failures = (
    checks_df[
        checks_df[
            "status"
        ]
        == "FAIL"
    ]
    .to_dict(
        orient="records"
    )
)

noncritical_checks = (
    checks_df[
        checks_df[
            "status"
        ]
        == "CHECK"
    ]
    .to_dict(
        orient="records"
    )
)


# ============================================================
# Column schema
# ============================================================

def hashable_for_unique(value):

    if value is None:
        return None

    if isinstance(
        value,
        (list, tuple, dict, set),
    ):
        return json.dumps(
            value,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

    if isinstance(
        value,
        np.ndarray,
    ):
        return json.dumps(
            value.tolist(),
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    return value


column_schema = []

for column in unified_manifest.columns:

    series = unified_manifest[
        column
    ]

    non_null_count = int(
        series.notna().sum()
    )

    null_count = int(
        series.isna().sum()
    )

    non_null_values = series[
        series.notna()
    ]

    normalized_values = (
        non_null_values.map(
            hashable_for_unique
        )
        if not non_null_values.empty
        else pd.Series(
            dtype="object"
        )
    )

    unique_count = int(
        normalized_values.nunique(
            dropna=True
        )
    )

    column_schema.append(
        {
            "column":
                column,

            "dtype":
                str(
                    series.dtype
                ),

            "non_null":
                non_null_count,

            "null":
                null_count,

            "unique":
                unique_count,
        }
    )

column_schema_path = (
    AUDIT_DIR
    / "unified_manifest_schema.json"
)

save_json(
    column_schema_path,
    column_schema,
)


# ============================================================
# Environment report
# ============================================================

environment_report = {
    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "python":
        sys.version,
    "platform":
        platform.platform(),
    "numpy":
        np.__version__,
    "pandas":
        pd.__version__,
    "project_dir":
        str(
            PROJECT_DIR
        ),
    "audit_dir":
        str(
            AUDIT_DIR
        ),
    "seed":
        CONFIG["seed"],
}

try:

    import torch

    environment_report[
        "pytorch"
    ] = torch.__version__

    environment_report[
        "cuda_available"
    ] = bool(
        torch.cuda.is_available()
    )

except Exception:

    environment_report[
        "pytorch"
    ] = None

    environment_report[
        "cuda_available"
    ] = False


environment_path = (
    AUDIT_DIR
    / "environment_report.json"
)

save_json(
    environment_path,
    environment_report,
)


# ============================================================
# Final report
# ============================================================

final_report = {
    "stage":
        "2_final",

    "purpose": (
        "Final dataset audit, raw OCT "
        "structure validation, label "
        "validation, cross-file pairing, "
        "discrepancy tracking, and unified "
        "manifest construction."
    ),

    "project":
        "glaucoma_oct_research",

    "seed":
        CONFIG["seed"],

    "source_report":
        str(
            SOURCE_REPORT_PATH
        ),

    "audit_directory":
        str(
            AUDIT_DIR
        ),

    "datasets":
        dataset_summaries,

    "validation":
        {
            "total_checks":
                int(
                    len(
                        checks_df
                    )
                ),
            "status_counts":
                status_counts,
            "critical_failures":
                critical_failures,
            "noncritical_checks":
                noncritical_checks,
        },

    "documented_discrepancies":
        all_discrepancies,

    "manifests":
        {
            "harvard_gf":
                str(
                    AUDIT_DIR
                    / "harvard_gf_manifest.csv"
                ),
            "harvard_gdp":
                str(
                    AUDIT_DIR
                    / "harvard_gdp_manifest.csv"
                ),
            "goals":
                str(
                    AUDIT_DIR
                    / "goals_manifest.csv"
                ),
            "gamma":
                str(
                    AUDIT_DIR
                    / "gamma_manifest.csv"
                ),
            "retina_oct_glaucoma":
                str(
                    AUDIT_DIR
                    / "retina_oct_glaucoma_manifest.csv"
                ),
            "unified":
                str(
                    unified_manifest_path
                ),
        },

    "reports":
        {
            "dataset_overview":
                str(
                    overview_path
                ),
            "validation_checks":
                str(
                    checks_path
                ),
            "discrepancies":
                str(
                    discrepancies_path
                ),
            "unified_manifest_schema":
                str(
                    column_schema_path
                ),
            "environment":
                str(
                    environment_path
                ),
        },

    "schemas":
        schemas,
}


report_path = (
    AUDIT_DIR
    / "stage2_final_report.json"
)

save_json(
    report_path,
    final_report,
)


# ============================================================
# Cleanup
# ============================================================

if TEMP_DIR.exists():

    shutil.rmtree(
        TEMP_DIR,
        ignore_errors=True,
    )


# ============================================================
# Final console summary
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "Stage 2 completed"
)

print(
    "=" * 70
)

print(
    "\nDataset summary:"
)

for _, row in (
    overview_df.iterrows()
):

    print(
        f"  {row['dataset']}: "
        f"{row['samples_observed']} samples"
    )

print(
    "\nValidation status:"
)

for status in [
    "PASS",
    "INFO",
    "CHECK",
    "FAIL",
]:

    count = int(
        status_counts.get(
            status,
            0,
        )
    )

    if count:
        print(
            f"  {status}: {count}"
        )

print(
    "\nCritical failures:"
)

if critical_failures:

    for failure in (
        critical_failures
    ):

        print(
            f"  {failure['dataset']} | "
            f"{failure['check']} | "
            f"{failure['details']}"
        )

else:

    print(
        "  None"
    )

print(
    "\nDocumented discrepancies:"
)

if all_discrepancies:

    for item in (
        all_discrepancies
    ):

        print(
            f"  {item}"
        )

else:

    print(
        "  None"
    )

print(
    "\nOutput files:"
)

print(
    f"  Dataset overview:\n"
    f"    {overview_path}"
)

print(
    f"  Validation checks:\n"
    f"    {checks_path}"
)

print(
    f"  Unified manifest:\n"
    f"    {unified_manifest_path}"
)

print(
    f"  Discrepancies:\n"
    f"    {discrepancies_path}"
)

print(
    f"  Manifest schema:\n"
    f"    {column_schema_path}"
)

print(
    f"  Final report:\n"
    f"    {report_path}"
)

print(
    "=" * 70
)

## Harvard-only operational data loaders

Only the two Harvard datasets used by the final experiments are retained.

In [ ]:
# ============================================================
# STAGE 4 — OPERATIONAL DATASET LOADERS
# ============================================================

import io
import gc
import re
import json
import hashlib
import shutil
import zipfile

from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from PIL import Image


# ============================================================
# 1. Paths
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/glaucoma_oct_research"
)

MANIFEST_PATH = (
    PROJECT_DIR
    / "results"
    / "data_audit"
    / "stage2"
    / "unified_manifest.csv"
)

GAMMA_CACHE_DIR = Path(
    "/content/stage4_gamma_cache"
)

GAMMA_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Unified manifest not found:\n"
        f"{MANIFEST_PATH}"
    )

manifest = pd.read_csv(
    MANIFEST_PATH
)


# ============================================================
# 2. Constants
# ============================================================

PROGRESSION_NAMES = [
    "md",
    "vfi",
    "td_pointwise",
    "md_fast",
    "md_fast_no_p_cut",
    "td_pointwise_no_p_cut",
]

IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}

MHA_TYPES = {
    "MET_CHAR": np.int8,
    "MET_UCHAR": np.uint8,
    "MET_SHORT": np.int16,
    "MET_USHORT": np.uint16,
    "MET_INT": np.int32,
    "MET_UINT": np.uint32,
    "MET_LONG": np.int64,
    "MET_ULONG": np.uint64,
    "MET_LONG_LONG": np.int64,
    "MET_ULONG_LONG": np.uint64,
    "MET_FLOAT": np.float32,
    "MET_DOUBLE": np.float64,
}


# ============================================================
# 3. General helpers
# ============================================================

def get_dataset_rows(
    dataset_name,
    split=None,
):

    rows = manifest[
        manifest["dataset"]
        .astype(str)
        .str.strip()
        .str.lower()
        == dataset_name.strip().lower()
    ].copy()

    if split is not None:

        rows = rows[
            rows["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            == str(split).strip().lower()
        ].copy()

    return rows.reset_index(
        drop=True
    )


def scalar_or_none(value):

    if value is None:
        return None

    try:

        if pd.isna(value):
            return None

    except Exception:

        pass

    return value


def clean_string(value):

    value = scalar_or_none(
        value
    )

    if value is None:
        return None

    return str(value).strip()


def int_or_none(value):

    value = scalar_or_none(
        value
    )

    if value is None:
        return None

    try:

        return int(
            float(value)
        )

    except Exception:

        return None


def float_or_none(value):

    value = scalar_or_none(
        value
    )

    if value is None:
        return None

    try:

        return float(value)

    except Exception:

        return None


def to_tensor_copy(
    array,
    dtype=None,
):

    array = np.asarray(
        array
    )

    if dtype is not None:

        array = array.astype(
            dtype,
            copy=True
        )

    else:

        array = np.array(
            array,
            copy=True
        )

    return torch.from_numpy(
        array
    )


def make_metadata(
    row
):

    fields = [
        "dataset",
        "split",
        "sample_id",
        "patient_id",
        "eye_id",
        "patient_id_status",
        "eye_id_status",
        "oct_format",
        "oct_member",
        "oct_shape",
        "rnflt_format",
        "rnflt_member",
        "rnflt_shape",
        "glaucoma_label",
        "glaucoma_grade",
        "glaucoma_grade_index",
        "glaucoma_binary_label",
        "gc_label",
        "classification_label",
        "classification_label_source",
        "filename_class_prefix",
        "progression_status",
        "progression_label",
        "age",
        "male",
        "race",
        "hispanic",
        "oct_member_count",
        "layer_annotation_status",
        "sample_status",
        "label_status",
        "pair_status",
    ]

    metadata = {}

    for field in fields:

        if field not in row.index:
            continue

        metadata[field] = (
            scalar_or_none(
                row[field]
            )
        )

    return metadata


# ============================================================
# 4. Progression parsing
# ============================================================

def parse_progression_row(
    row
):

    if (
        "progression_status"
        not in row.index
    ):

        return None

    status = clean_string(
        row["progression_status"]
    )

    if status is None:
        return None

    # Only progression-labeled samples receive a target.
    if status.lower() != "labeled":
        return None

    values = []

    for name in PROGRESSION_NAMES:

        column = (
            f"progression_{name}"
        )

        if column not in row.index:
            return None

        value = int_or_none(
            row[column]
        )

        if value is None:
            return None

        values.append(
            value
        )

    return torch.tensor(
        values,
        dtype=torch.int64
    )


# ============================================================
# 5. Locator handling
# ============================================================

def parse_locator(
    locator
):

    locator = scalar_or_none(
        locator
    )

    if locator is None:

        raise ValueError(
            "Locator is empty."
        )

    locator = str(
        locator
    ).strip()

    if locator.startswith(
        "zip://"
    ):

        payload = locator[
            len("zip://"):
        ]

        archive, member = (
            payload.split(
                "!",
                1
            )
        )

        return {
            "type": "zip",
            "archive": archive,
            "member": member,
        }

    if locator.startswith(
        "nestedzip://"
    ):

        payload = locator[
            len("nestedzip://"):
        ]

        parts = payload.split(
            "!",
            2
        )

        if len(parts) != 3:

            raise ValueError(
                f"Invalid nested ZIP locator:\n"
                f"{locator}"
            )

        return {
            "type": "nestedzip",
            "outer_archive": parts[0],
            "nested_archive": parts[1],
            "member": parts[2],
        }

    return {
        "type": "file",
        "path": locator,
    }


def read_file_bytes(
    locator
):

    info = parse_locator(
        locator
    )

    if info["type"] == "file":

        with open(
            info["path"],
            "rb"
        ) as f:

            return f.read()

    if info["type"] == "zip":

        with zipfile.ZipFile(
            info["archive"],
            "r"
        ) as zf:

            return zf.read(
                info["member"]
            )

    raise RuntimeError(
        "Nested ZIP requires "
        "read_nested_member()."
    )


def ensure_nested_zip_cached(
    outer_archive,
    nested_archive,
):

    cache_key = hashlib.sha1(
        (
            str(outer_archive)
            + "!"
            + str(nested_archive)
        ).encode(
            "utf-8"
        )
    ).hexdigest()[:16]

    cache_path = (
        GAMMA_CACHE_DIR
        / (
            f"{cache_key}_"
            f"{Path(nested_archive).name}"
        )
    )

    if (
        cache_path.exists()
        and cache_path.stat().st_size > 0
    ):

        return cache_path

    partial_path = (
        cache_path.parent
        / (
            cache_path.name
            + ".partial"
        )
    )

    if partial_path.exists():

        partial_path.unlink()

    with zipfile.ZipFile(
        outer_archive,
        "r"
    ) as outer_zip:

        with outer_zip.open(
            nested_archive,
            "r"
        ) as source:

            with open(
                partial_path,
                "wb"
            ) as target:

                shutil.copyfileobj(
                    source,
                    target,
                    length=16 * 1024 * 1024
                )

    partial_path.replace(
        cache_path
    )

    return cache_path


def read_nested_member(
    locator
):

    info = parse_locator(
        locator
    )

    nested_path = (
        ensure_nested_zip_cached(
            info["outer_archive"],
            info["nested_archive"]
        )
    )

    with zipfile.ZipFile(
        nested_path,
        "r"
    ) as zf:

        return zf.read(
            info["member"]
        )


def read_any_bytes(
    locator
):

    info = parse_locator(
        locator
    )

    if info["type"] == "nestedzip":

        return read_nested_member(
            locator
        )

    return read_file_bytes(
        locator
    )


# ============================================================
# 6. NPZ / image helpers
# ============================================================

def read_npz(
    locator
):

    raw = read_any_bytes(
        locator
    )

    try:

        with np.load(
            io.BytesIO(raw),
            allow_pickle=False
        ) as data:

            result = {
                key: np.asarray(
                    data[key]
                )
                for key in data.files
            }

    finally:

        del raw
        gc.collect()

    return result


def read_image(
    locator,
    mode="RGB",
):

    raw = read_any_bytes(
        locator
    )

    try:

        image = Image.open(
            io.BytesIO(raw)
        ).convert(
            mode
        )

        array = np.asarray(
            image
        )

        array = np.array(
            array,
            copy=True
        )

    finally:

        del raw

    tensor = torch.from_numpy(
        array
    )

    if tensor.ndim == 2:

        return tensor.unsqueeze(
            0
        )

    return tensor.permute(
        2,
        0,
        1
    )


def get_image_members(
    zf,
    directory
):

    directory = str(
        PurePosixPath(
            directory
        )
    )

    members = []

    for name in zf.namelist():

        path = PurePosixPath(
            name
        )

        if str(path.parent) != directory:
            continue

        if (
            path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        members.append(
            name
        )

    def natural_key(
        value
    ):

        return [
            int(part)
            if part.isdigit()
            else part.lower()
            for part in re.split(
                r"(\d+)",
                str(value)
            )
        ]

    return sorted(
        members,
        key=natural_key
    )


# ============================================================
# 7. Base Dataset
# ============================================================

class BaseOCTDataset(
    Dataset
):

    dataset_name = None
    role = "main"

    def __init__(
        self,
        split=None
    ):

        self.split = split

        self.manifest = (
            get_dataset_rows(
                self.dataset_name,
                split=split
            )
        )

        if len(self.manifest) == 0:

            raise ValueError(
                f"No samples found for "
                f"{self.dataset_name}, "
                f"split={split}"
            )

    def __len__(
        self
    ):

        return len(
            self.manifest
        )

    def get_row(
        self,
        index
    ):

        return self.manifest.iloc[
            index
        ]

    def build_sample(
        self,
        row,
        oct_data=None,
        rnflt=None,
        glaucoma=None,
        progression=None,
        fundus=None,
        layer_mask=None,
        layer_show=None,
        glaucoma_grade=None,
        metadata_extra=None,
    ):

        metadata = make_metadata(
            row
        )

        if metadata_extra:

            metadata.update(
                metadata_extra
            )

        return {
            "oct": oct_data,
            "rnflt": rnflt,
            "glaucoma": glaucoma,
            "progression": progression,
            "fundus": fundus,
            "layer_mask": layer_mask,
            "layer_show": layer_show,
            "glaucoma_grade": glaucoma_grade,
            "metadata": metadata,
        }


# ============================================================
# 8. Harvard datasets
# ============================================================

class HarvardNPZDataset(
    BaseOCTDataset
):

    oct_key = None

    def __getitem__(
        self,
        index
    ):

        row = self.get_row(
            index
        )

        fields = read_npz(
            row["oct_locator"]
        )

        if self.oct_key not in fields:

            raise KeyError(
                f"{self.dataset_name}: "
                f"expected NPZ field "
                f"'{self.oct_key}', "
                f"available fields: "
                f"{list(fields.keys())}"
            )

        oct_data = to_tensor_copy(
            fields[
                self.oct_key
            ]
        )

        rnflt = None

        # Some Harvard samples have RNFLT
        # inside the same NPZ.
        if "rnflt" in fields:

            rnflt = to_tensor_copy(
                fields["rnflt"],
                dtype=np.float32
            )

        # Otherwise load RNFLT from its
        # separate manifest locator.
        rnflt_locator = (
            scalar_or_none(
                row.get(
                    "rnflt_locator"
                )
            )
        )

        oct_locator = (
            scalar_or_none(
                row.get(
                    "oct_locator"
                )
            )
        )

        if (
            rnflt is None
            and rnflt_locator is not None
            and rnflt_locator != oct_locator
        ):

            rnflt_fields = read_npz(
                rnflt_locator
            )

            if "rnflt" in rnflt_fields:

                rnflt = to_tensor_copy(
                    rnflt_fields[
                        "rnflt"
                    ],
                    dtype=np.float32
                )

            del rnflt_fields

        glaucoma = int_or_none(
            row.get(
                "glaucoma_label"
            )
        )

        progression = (
            parse_progression_row(
                row
            )
        )

        metadata_extra = {
            "oct_key":
                self.oct_key,
            "oct_tensor_shape":
                tuple(
                    oct_data.shape
                ),
            "oct_tensor_dtype":
                str(
                    oct_data.dtype
                ),
            "rnflt_tensor_shape":
                (
                    tuple(
                        rnflt.shape
                    )
                    if rnflt is not None
                    else None
                ),
        }

        return self.build_sample(
            row,
            oct_data=oct_data,
            rnflt=rnflt,
            glaucoma=glaucoma,
            progression=progression,
            metadata_extra=metadata_extra,
        )


class HarvardGFDataset(
    HarvardNPZDataset
):

    dataset_name = (
        "harvard_gf"
    )

    oct_key = "oct_bscans"

    role = "main"


class HarvardGDPDataset(
    HarvardNPZDataset
):

    dataset_name = (
        "harvard_gdp"
    )

    # Harvard-GDP raw OCT uses "bscans".
    oct_key = "bscans"

    role = "main"

# ============================================================
# Minimal final loader interface
# ============================================================
# Only Harvard-GF and Harvard-GDP are used by the final pipeline.
print("Harvard-GF and Harvard-GDP loaders are ready.")


## Frozen split generation

Create the deterministic GDP glaucoma split used by the protocol. No unrelated endpoint or segmentation split is generated.

In [ ]:
# ============================================================
# Stage 5 Paths
# ============================================================

from pathlib import Path

assert "PROJECT_DIR" in globals(), (
    "Run Stage 0 first."
)

STAGE5_DIR = (
    PROJECT_DIR
    / "results"
    / "stage5"
)

STAGE5_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("Stage 5 paths initialized")
print("=" * 70)

print(
    f"STAGE5_DIR: {STAGE5_DIR}"
)

In [ ]:
# ============================================================
# Stage 5.4: Final Split Generation
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

assert "manifest" in globals(), (
    "Run Stage 5.3A first."
)

print("=" * 70)
print("Stage 5.4 - Final Split Generation")
print("=" * 70)

# ------------------------------------------------------------
# 1. Select Harvard-GDP samples
# ------------------------------------------------------------

gdp = manifest[
    manifest["dataset"].eq("harvard_gdp")
].copy()

status = (
    gdp["progression_status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

labeled_mask = status.eq("labeled")
unlabeled_mask = ~labeled_mask

labeled = gdp[
    labeled_mask
].copy()

unlabeled = gdp[
    unlabeled_mask
].copy()

if len(gdp) != 1000:
    raise RuntimeError(
        f"Expected 1000 GDP samples, found {len(gdp)}."
    )

if len(labeled) != 500:
    raise RuntimeError(
        f"Expected 500 labeled GDP samples, found {len(labeled)}."
    )

if len(unlabeled) != 500:
    raise RuntimeError(
        f"Expected 500 unlabeled GDP samples, found {len(unlabeled)}."
    )

print()
print("GDP progression scope:")
print(f"Total samples      : {len(gdp)}")
print(f"Labeled samples    : {len(labeled)}")
print(f"Unlabeled samples  : {len(unlabeled)}")

# ------------------------------------------------------------
# 2. Check subject identity using sample_id
# ------------------------------------------------------------

sample_id = (
    labeled["sample_id"]
    .astype("string")
    .str.strip()
)

missing_sample = (
    sample_id.isna()
    | sample_id.eq("")
    | sample_id.str.lower().isin(
        ["nan", "none", "null"]
    )
)

duplicate_sample = sample_id[
    sample_id.notna()
].duplicated(
    keep=False
)

print()
print("Subject identity audit:")
print(
    f"Missing sample_id   : {missing_sample.sum()}"
)
print(
    f"Duplicated sample_id: {duplicate_sample.sum()}"
)

if missing_sample.any():
    raise RuntimeError(
        "Some labeled GDP samples have missing sample_id."
    )

if duplicate_sample.any():
    repeated = (
        sample_id[duplicate_sample]
        .value_counts()
        .head(10)
    )

    print()
    print("Repeated sample IDs:")
    display(repeated)

    raise RuntimeError(
        "Repeated sample_id detected. "
        "The GDP split cannot safely proceed."
    )

labeled["split_group_id"] = sample_id

print()
print(
    "GDP identity note: "
    "sample_id is used as the split group identifier "
    "because the dataset provides one subject-level sample "
    "per patient."
)

# ------------------------------------------------------------
# 3. Build the stratification label
# ------------------------------------------------------------

labeled["glaucoma_binary"] = pd.to_numeric(
    labeled["glaucoma_label"],
    errors="coerce"
)

labeled["td_pointwise_binary"] = pd.to_numeric(
    labeled["progression_td_pointwise"],
    errors="coerce"
)

if labeled[
    ["glaucoma_binary", "td_pointwise_binary"]
].isna().any().any():
    raise RuntimeError(
        "Missing values found in stratification variables."
    )

labeled["combined_stratum"] = (
    labeled["glaucoma_binary"]
    .astype(int)
    .astype(str)
    + "_"
    + labeled["td_pointwise_binary"]
    .astype(int)
    .astype(str)
)

print()
print("Stratification groups:")
display(
    labeled["combined_stratum"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 4. Train / temporary split
# ------------------------------------------------------------

indices = np.arange(len(labeled))

splitter_1 = StratifiedShuffleSplit(
    n_splits=1,
    train_size=350,
    test_size=150,
    random_state=42
)

train_idx, temp_idx = next(
    splitter_1.split(
        indices,
        labeled["combined_stratum"]
    )
)

# ------------------------------------------------------------
# 5. Validation / test split
# ------------------------------------------------------------

temp_strata = labeled.iloc[
    temp_idx
]["combined_stratum"]

temp_indices = np.arange(len(temp_idx))

splitter_2 = StratifiedShuffleSplit(
    n_splits=1,
    train_size=75,
    test_size=75,
    random_state=42
)

val_rel_idx, test_rel_idx = next(
    splitter_2.split(
        temp_indices,
        temp_strata
    )
)

val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

# ------------------------------------------------------------
# 6. Create split labels
# ------------------------------------------------------------

labeled["final_split"] = "unused"

labeled.loc[
    labeled.index[train_idx],
    "final_split"
] = "train"

labeled.loc[
    labeled.index[val_idx],
    "final_split"
] = "validation"

labeled.loc[
    labeled.index[test_idx],
    "final_split"
] = "test"

# ------------------------------------------------------------
# 7. Put unlabeled GDP samples into SSL pool
# ------------------------------------------------------------

unlabeled["final_split"] = "ssl_pool"

# ------------------------------------------------------------
# 8. Final GDP split manifest
# ------------------------------------------------------------

gdp_split = pd.concat(
    [
        labeled,
        unlabeled
    ],
    axis=0
).sort_index()

# ------------------------------------------------------------
# 9. Split summary
# ------------------------------------------------------------

summary = (
    gdp_split
    .groupby(
        ["final_split"],
        dropna=False
    )
    .size()
    .rename("count")
    .reset_index()
)

print()
print("=" * 70)
print("Final split counts")
print("=" * 70)

display(summary)

# ------------------------------------------------------------
# 10. Stratification verification
# ------------------------------------------------------------

verification = (
    labeled
    .groupby(
        ["final_split", "combined_stratum"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("=" * 70)
print("Stratification verification")
print("=" * 70)

display(verification)

# ------------------------------------------------------------
# 11. Progression and glaucoma counts
# ------------------------------------------------------------

target_summary = (
    labeled
    .groupby("final_split")
    .agg(
        total=("sample_id", "size"),
        glaucoma_positive=(
            "glaucoma_binary",
            "sum"
        ),
        td_progression_positive=(
            "td_pointwise_binary",
            "sum"
        )
    )
    .reset_index()
)

print()
print("=" * 70)
print("Target distribution by split")
print("=" * 70)

display(target_summary)

# ------------------------------------------------------------
# 12. Save outputs
# ------------------------------------------------------------

split_manifest_path = (
    STAGE5_DIR
    / "gdp_final_split_manifest.csv"
)

summary_path = (
    STAGE5_DIR
    / "gdp_final_split_summary.csv"
)

gdp_split.to_csv(
    split_manifest_path,
    index=False
)

target_summary.to_csv(
    summary_path,
    index=False
)

print()
print("=" * 70)
print("Stage 5.4 completed.")
print("=" * 70)

print(
    f"Saved split manifest: {split_manifest_path}"
)

print(
    f"Saved split summary : {summary_path}"
)

## Final OCT preprocessing manifest

Generate the repaired manifest and train-only intensity statistics required by the frozen Stage 10 protocol.

In [ ]:
from pathlib import Path

STAGE6_DIR = PROJECT_DIR / "results" / "stage6"
STAGE6_DIR.mkdir(parents=True, exist_ok=True)
print(f"Stage 6 output directory: {STAGE6_DIR}")

In [ ]:
# ============================================================
# Stage 6.3 — Intensity Preprocessing and Train-Only Statistics
# ============================================================

import io
import json
import zipfile
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd


assert "PROJECT_DIR" in globals()
assert "STAGE6_DIR" in globals()
assert "manifest" in globals()
assert "read_npz" in globals()


# ============================================================
# Paths
# ============================================================

STAGE6_INTENSITY_DIR = (
    STAGE6_DIR
    / "intensity_preprocessing"
)

STAGE6_INTENSITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GDP_SPLIT_PATH = (
    PROJECT_DIR
    / "results"
    / "stage5"
    / "gdp_final_split_manifest.csv"
)

if not GDP_SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"GDP split manifest not found: {GDP_SPLIT_PATH}"
    )

gdp_split = pd.read_csv(
    GDP_SPLIT_PATH
)


# ============================================================
# Recover official Harvard-GF split from oct_member
# ============================================================

def infer_harvard_gf_split_from_member(member):
    if pd.isna(member):
        return None

    parts = [
        part.lower()
        for part in PurePosixPath(
            str(member).replace("\\", "/")
        ).parts
    ]

    if "training" in parts:
        return "train"

    if "validation" in parts:
        return "validation"

    if "test" in parts:
        return "test"

    return None


gf_rows = manifest[
    manifest["dataset"].astype(str).eq("harvard_gf")
].copy()

gf_rows["recovered_split"] = (
    gf_rows["oct_member"]
    .apply(
        infer_harvard_gf_split_from_member
    )
)

expected_gf_counts = {
    "train": 2100,
    "validation": 300,
    "test": 900,
}

observed_gf_counts = (
    gf_rows["recovered_split"]
    .value_counts()
    .to_dict()
)

print("=" * 90)
print("Stage 6.3 — Split Verification")
print("=" * 90)

print()
print("Harvard-GF recovered split:")
display(
    gf_rows["recovered_split"]
    .fillna("unresolved")
    .value_counts()
)

if gf_rows["recovered_split"].isna().any():
    raise RuntimeError(
        "Some Harvard-GF samples have unresolved split metadata."
    )

if observed_gf_counts != expected_gf_counts:
    raise RuntimeError(
        f"Harvard-GF split mismatch. "
        f"Observed={observed_gf_counts}, "
        f"Expected={expected_gf_counts}"
    )


# Update only Harvard-GF split metadata.
manifest.loc[
    gf_rows.index,
    "split"
] = gf_rows[
    "recovered_split"
].values


# ============================================================
# Verify final Harvard-GDP split
# ============================================================

if "final_split" not in gdp_split.columns:
    raise RuntimeError(
        "Stage 5.4 GDP split manifest does not contain "
        "'final_split'."
    )

expected_gdp_counts = {
    "train": 350,
    "validation": 75,
    "test": 75,
    "ssl_pool": 500,
}

observed_gdp_counts = (
    gdp_split["final_split"]
    .value_counts()
    .to_dict()
)

print()
print("Harvard-GDP final split:")
display(
    gdp_split["final_split"]
    .value_counts()
)

if observed_gdp_counts != expected_gdp_counts:
    raise RuntimeError(
        f"Harvard-GDP split mismatch. "
        f"Observed={observed_gdp_counts}, "
        f"Expected={expected_gdp_counts}"
    )


# ============================================================
# Build train-only pools
# ============================================================

gf_train = manifest[
    manifest["dataset"].astype(str).eq("harvard_gf")
    & manifest["split"].astype(str).eq("train")
].copy()

gdp_train_ids = set(
    gdp_split.loc[
        gdp_split["final_split"].eq("train"),
        "sample_id"
    ]
    .astype(str)
)

gdp_train = manifest[
    manifest["dataset"].astype(str).eq("harvard_gdp")
    & manifest["sample_id"].astype(str).isin(
        gdp_train_ids
    )
].copy()


print()
print("Training pools:")
print(
    f"Harvard-GF train:  {len(gf_train)}"
)
print(
    f"Harvard-GDP train: {len(gdp_train)}"
)


if len(gf_train) != 2100:
    raise RuntimeError(
        f"Expected 2100 Harvard-GF training samples, "
        f"found {len(gf_train)}."
    )

if len(gdp_train) != 350:
    raise RuntimeError(
        f"Expected 350 Harvard-GDP training samples, "
        f"found {len(gdp_train)}."
    )


# ============================================================
# Deterministic intensity conversion
# ============================================================

def to_float01(image):
    image = np.asarray(
        image,
        dtype=np.float32
    )

    image = image / 255.0

    return np.clip(
        image,
        0.0,
        1.0
    )


# ============================================================
# Streaming statistics
#
# Only training samples are used.
# ============================================================

def compute_intensity_statistics(
    rows,
    dataset_name,
    n_bscans_per_volume=5,
    n_pixels_per_bscan=4096,
    seed=42
):
    rng = np.random.default_rng(
        seed
    )

    total_count = 0
    total_sum = 0.0
    total_sq_sum = 0.0

    observed_min = np.inf
    observed_max = -np.inf

    for _, row in rows.iterrows():

        data = read_npz(
            row["oct_locator"]
        )

        if dataset_name == "harvard_gf":
            volume = np.asarray(
                data["oct_bscans"]
            )
        elif dataset_name == "harvard_gdp":
            volume = np.asarray(
                data["bscans"]
            )
        else:
            raise ValueError(
                f"Unsupported dataset: {dataset_name}"
            )

        # Stage 5.1 selected axis 2 as the B-scan axis.
        volume = np.moveaxis(
            volume,
            2,
            0
        )

        n_scans = volume.shape[0]

        scan_count = min(
            n_bscans_per_volume,
            n_scans
        )

        scan_indices = np.linspace(
            0,
            n_scans - 1,
            scan_count
        ).astype(int)

        for scan_index in scan_indices:

            scan = to_float01(
                volume[scan_index]
            ).reshape(-1)

            if len(scan) > n_pixels_per_bscan:

                pixel_indices = rng.choice(
                    len(scan),
                    size=n_pixels_per_bscan,
                    replace=False
                )

                values = scan[
                    pixel_indices
                ]

            else:
                values = scan

            total_count += len(values)

            total_sum += float(
                values.sum()
            )

            total_sq_sum += float(
                np.square(values).sum()
            )

            observed_min = min(
                observed_min,
                float(values.min())
            )

            observed_max = max(
                observed_max,
                float(values.max())
            )

    if total_count == 0:
        raise RuntimeError(
            f"No intensity samples were collected "
            f"for {dataset_name}."
        )

    mean = (
        total_sum
        / total_count
    )

    variance = (
        total_sq_sum
        / total_count
        - mean ** 2
    )

    variance = max(
        variance,
        0.0
    )

    std = float(
        np.sqrt(variance)
    )

    return {
        "dataset": dataset_name,
        "samples_used": len(rows),
        "bscans_per_volume": n_bscans_per_volume,
        "pixels_per_bscan": n_pixels_per_bscan,
        "count": total_count,
        "min": observed_min,
        "max": observed_max,
        "mean": mean,
        "std": std,
        "normalization": (
            "uint8_to_float32_then_divide_by_255"
        ),
        "statistics_scope": "training_only",
        "seed": seed,
    }


# ============================================================
# Compute training-only statistics
# ============================================================

gf_stats = compute_intensity_statistics(
    gf_train,
    "harvard_gf"
)

gdp_stats = compute_intensity_statistics(
    gdp_train,
    "harvard_gdp"
)

intensity_stats = pd.DataFrame(
    [
        gf_stats,
        gdp_stats
    ]
)


# ============================================================
# Intensity policy
# ============================================================

INTENSITY_POLICY = {
    "base_conversion": (
        "uint8 -> float32 -> divide by 255 -> [0,1]"
    ),
    "standardization": (
        "optional z-score using training-only mean/std"
    ),
    "harvard_gf": {
        "statistics_source": (
            "official training split only"
        ),
        "validation_used": False,
        "test_used": False,
    },
    "harvard_gdp": {
        "statistics_source": (
            "final supervised progression training split only"
        ),
        "ssl_pool_used_for_statistics": False,
        "validation_used": False,
        "test_used": False,
    },
    "external_datasets": {
        "statistics_source": None,
        "policy": (
            "Do not fit normalization statistics "
            "at this stage."
        ),
    },
}


# ============================================================
# Save repaired manifest
# ============================================================

REPAIRED_MANIFEST_PATH = (
    STAGE6_DIR
    / "manifest_with_repaired_splits.csv"
)

manifest.to_csv(
    REPAIRED_MANIFEST_PATH,
    index=False
)


# ============================================================
# Save intensity statistics
# ============================================================

STATS_PATH = (
    STAGE6_INTENSITY_DIR
    / "intensity_statistics.csv"
)

POLICY_PATH = (
    STAGE6_INTENSITY_DIR
    / "intensity_normalization_policy.json"
)

intensity_stats.to_csv(
    STATS_PATH,
    index=False
)

with open(
    POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        INTENSITY_POLICY,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# Display results
# ============================================================

print()
print("=" * 90)
print("Stage 6.3 — Intensity Statistics")
print("=" * 90)

display(
    intensity_stats[
        [
            "dataset",
            "samples_used",
            "bscans_per_volume",
            "pixels_per_bscan",
            "count",
            "min",
            "max",
            "mean",
            "std",
            "normalization",
            "statistics_scope",
        ]
    ]
)

print()
print("Saved files:")
print(
    f"Repaired manifest: {REPAIRED_MANIFEST_PATH}"
)
print(
    f"Intensity statistics: {STATS_PATH}"
)
print(
    f"Normalization policy: {POLICY_PATH}"
)

print()
print("=" * 90)
print("Stage 6.3 completed successfully.")
print("=" * 90)

In [ ]:
# The rest of Stage 6.3 — Finalize repaired split metadata

GDP_SPLIT_PATH = "/content/drive/MyDrive/glaucoma_oct_research/results/stage5/gdp_final_split_manifest.csv"

gdp_split = pd.read_csv(GDP_SPLIT_PATH)

split_map = (
    gdp_split
    .assign(sample_id=lambda x: x["sample_id"].astype(str))
    .set_index("sample_id")["final_split"]
)

mask = stage6_manifest["dataset"].eq("harvard_gdp")

stage6_manifest.loc[mask, "split"] = (
    stage6_manifest.loc[mask, "sample_id"]
    .astype(str)
    .map(split_map)
    .str.lower()
)

stage6_manifest.to_csv(MANIFEST_PATH, index=False)
print(stage6_manifest.loc[mask, "split"].value_counts(dropna=False))

## Protocol freeze and leakage audit

Create the final experiment manifest and the label-free SSL manifest. These files are the authoritative inputs for self-supervised pretraining and downstream evaluation.

In [ ]:
# ============================================================
# Stage 10 — Protocol Freeze, Leakage Audit, and SSL Manifest
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------

PROJECT_DIR = globals().get(
    "PROJECT_DIR",
    "/content/drive/MyDrive/glaucoma_oct_research"
)

STAGE10_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "stage10_protocol"
)

BASE_MANIFEST_PATH = os.path.join(
    PROJECT_DIR,
    "results",
    "stage6",
    "manifest_with_repaired_splits.csv"
)

GDP_PROGRESSION_SPLIT_PATH = os.path.join(
    PROJECT_DIR,
    "results",
    "stage5",
    "gdp_final_split_manifest.csv"
)

MAIN_MANIFEST_PATH = os.path.join(
    STAGE10_DIR,
    "main_experiment_manifest.csv"
)

SSL_MANIFEST_PATH = os.path.join(
    STAGE10_DIR,
    "ssl_training_manifest.csv"
)

GDP_DETECTION_SPLIT_PATH = os.path.join(
    STAGE10_DIR,
    "gdp_glaucoma_detection_split.csv"
)

AUDIT_PATH = os.path.join(
    STAGE10_DIR,
    "leakage_audit.csv"
)

SUMMARY_PATH = os.path.join(
    STAGE10_DIR,
    "main_experiment_manifest.json"
)

os.makedirs(STAGE10_DIR, exist_ok=True)

for path in [
    BASE_MANIFEST_PATH,
    GDP_PROGRESSION_SPLIT_PATH
]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Required source file was not found:\n{path}"
        )

PRIMARY_PROGRESSION_ENDPOINT = "progression_td_pointwise"

EXPECTED_GF_SPLITS = {
    "train": 2100,
    "validation": 300,
    "test": 900,
}

EXPECTED_GDP_PROGRESSION_SPLITS = {
    "train": 350,
    "validation": 75,
    "test": 75,
    "ssl_pool": 500,
}

EXPECTED_GDP_GLAUCOMA_SPLITS = {
    "train": 850,
    "validation": 75,
    "test": 75,
}

# ------------------------------------------------------------
# 2. Load authoritative source manifests
# ------------------------------------------------------------

manifest = pd.read_csv(BASE_MANIFEST_PATH)
gdp_split = pd.read_csv(GDP_PROGRESSION_SPLIT_PATH)

required_manifest_columns = [
    "dataset",
    "sample_id",
    "split",
    "glaucoma_label",
    PRIMARY_PROGRESSION_ENDPOINT,
    "oct_locator",
    "rnflt_locator",
]

missing_columns = [
    column
    for column in required_manifest_columns
    if column not in manifest.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in source manifest: {missing_columns}"
    )

if not {"sample_id", "final_split"}.issubset(
    gdp_split.columns
):
    raise ValueError(
        "GDP progression split file must contain "
        "'sample_id' and 'final_split'."
    )


def normalize_id(value):
    """Normalize IDs to avoid numeric string mismatches."""
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.endswith(".0") and value[:-2].isdigit():
        value = value[:-2]

    return value


manifest["dataset"] = (
    manifest["dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
)

manifest["sample_id"] = (
    manifest["sample_id"].map(normalize_id)
)

manifest["split"] = (
    manifest["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

gdp_split["sample_id"] = (
    gdp_split["sample_id"].map(normalize_id)
)

gdp_split["final_split"] = (
    gdp_split["final_split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if manifest["sample_id"].eq("").any():
    raise ValueError("Empty sample_id found in source manifest.")

if gdp_split["sample_id"].eq("").any():
    raise ValueError("Empty sample_id found in GDP split manifest.")

if manifest.duplicated(["dataset", "sample_id"]).any():
    raise ValueError(
        "Duplicate dataset/sample_id keys found in source manifest."
    )

if gdp_split["sample_id"].duplicated().any():
    raise ValueError(
        "Duplicate sample_id values found in GDP split manifest."
    )

# ------------------------------------------------------------
# 3. Separate Harvard-GF and Harvard-GDP
# ------------------------------------------------------------

gf = manifest.loc[
    manifest["dataset"].eq("harvard_gf")
].copy()

gdp = manifest.loc[
    manifest["dataset"].eq("harvard_gdp")
].copy()

if len(gf) != 3300:
    raise ValueError(
        f"Expected 3300 Harvard-GF samples; found {len(gf)}."
    )

if len(gdp) != 1000:
    raise ValueError(
        f"Expected 1000 Harvard-GDP samples; found {len(gdp)}."
    )

# ------------------------------------------------------------
# 4. Lock Harvard-GF splits
# ------------------------------------------------------------

gf["protocol_split"] = gf["split"]

gf_split_counts = {
    str(key): int(value)
    for key, value in gf["protocol_split"].value_counts().items()
}

if gf_split_counts != EXPECTED_GF_SPLITS:
    raise ValueError(
        "Harvard-GF split mismatch.\n"
        f"Observed: {gf_split_counts}\n"
        f"Expected: {EXPECTED_GF_SPLITS}"
    )

# ------------------------------------------------------------
# 5. Lock Harvard-GDP progression splits
# ------------------------------------------------------------

gdp = gdp.drop(
    columns=["final_split"],
    errors="ignore"
)

gdp = gdp.merge(
    gdp_split[["sample_id", "final_split"]],
    on="sample_id",
    how="left",
    validate="one_to_one"
)

if gdp["final_split"].isna().any():
    missing_ids = gdp.loc[
        gdp["final_split"].isna(),
        "sample_id"
    ].tolist()

    raise ValueError(
        "GDP samples are missing their final progression split: "
        f"{missing_ids[:10]}"
    )

gdp["protocol_split"] = gdp["final_split"]

gdp_split_counts = {
    str(key): int(value)
    for key, value in gdp["protocol_split"].value_counts().items()
}

if gdp_split_counts != EXPECTED_GDP_PROGRESSION_SPLITS:
    raise ValueError(
        "Harvard-GDP progression split mismatch.\n"
        f"Observed: {gdp_split_counts}\n"
        f"Expected: {EXPECTED_GDP_PROGRESSION_SPLITS}"
    )

# ------------------------------------------------------------
# 6. Define the separate glaucoma detection split
# ------------------------------------------------------------

gdp["glaucoma_split"] = pd.Series(
    index=gdp.index,
    dtype="object"
)

gdp.loc[
    gdp["protocol_split"].isin(["train", "ssl_pool"]),
    "glaucoma_split"
] = "train"

gdp.loc[
    gdp["protocol_split"].eq("validation"),
    "glaucoma_split"
] = "validation"

gdp.loc[
    gdp["protocol_split"].eq("test"),
    "glaucoma_split"
] = "test"

if gdp["glaucoma_split"].isna().any():
    raise ValueError(
        "Some GDP samples do not have a glaucoma split."
    )

gdp_glaucoma_counts = {
    str(key): int(value)
    for key, value in gdp["glaucoma_split"].value_counts().items()
}

if gdp_glaucoma_counts != EXPECTED_GDP_GLAUCOMA_SPLITS:
    raise ValueError(
        "GDP glaucoma split mismatch.\n"
        f"Observed: {gdp_glaucoma_counts}\n"
        f"Expected: {EXPECTED_GDP_GLAUCOMA_SPLITS}"
    )

# ------------------------------------------------------------
# 7. Build the unified experiment manifest
# ------------------------------------------------------------

gf["glaucoma_split"] = gf["protocol_split"]
gf["progression_split"] = "not_available"

gdp["progression_split"] = np.where(
    gdp["protocol_split"].isin(
        ["train", "validation", "test"]
    ),
    gdp["protocol_split"],
    "not_available"
)

manifest = pd.concat(
    [gf, gdp],
    ignore_index=True,
    sort=False
)

manifest["subject_key"] = (
    manifest["dataset"]
    + "::"
    + manifest["sample_id"]
)

if manifest["subject_key"].duplicated().any():
    raise ValueError(
        "Duplicate subject_key values found in unified manifest."
    )

# ------------------------------------------------------------
# 8. Validate and define the clinical task masks
# ------------------------------------------------------------

manifest["glaucoma_label"] = pd.to_numeric(
    manifest["glaucoma_label"],
    errors="coerce"
)

manifest[PRIMARY_PROGRESSION_ENDPOINT] = pd.to_numeric(
    manifest[PRIMARY_PROGRESSION_ENDPOINT],
    errors="coerce"
)

manifest["glaucoma_available"] = (
    manifest["glaucoma_label"].notna()
)

manifest["progression_available"] = (
    manifest[PRIMARY_PROGRESSION_ENDPOINT].notna()
)

glaucoma_values = set(
    manifest.loc[
        manifest["glaucoma_available"],
        "glaucoma_label"
    ].unique().tolist()
)

progression_values = set(
    manifest.loc[
        manifest["progression_available"],
        PRIMARY_PROGRESSION_ENDPOINT
    ].unique().tolist()
)

if not manifest["glaucoma_available"].all():
    raise ValueError(
        "Missing glaucoma labels found in the main datasets."
    )

if not glaucoma_values.issubset({0, 1}):
    raise ValueError(
        f"Glaucoma labels must be binary; found {glaucoma_values}."
    )

gdp_mask = manifest["dataset"].eq("harvard_gdp")

expected_progression_mask = (
    gdp_mask
    & manifest["protocol_split"].isin(
        ["train", "validation", "test"]
    )
)

if not manifest["progression_available"].equals(
    expected_progression_mask
):
    actual_ids = set(
        manifest.loc[
            manifest["progression_available"],
            "subject_key"
        ]
    )

    expected_ids = set(
        manifest.loc[
            expected_progression_mask,
            "subject_key"
        ]
    )

    raise ValueError(
        "Progression labels do not match the locked progression split. "
        f"Unexpected labeled samples: "
        f"{list(actual_ids - expected_ids)[:10]}; "
        f"missing labels: {list(expected_ids - actual_ids)[:10]}"
    )

if len(progression_values) == 0 or not progression_values.issubset({0, 1}):
    raise ValueError(
        "The primary progression endpoint must contain binary labels "
        f"0/1; found {progression_values}."
    )

if int(manifest["progression_available"].sum()) != 500:
    raise ValueError(
        "Expected progression labels for exactly 500 GDP samples."
    )

# ------------------------------------------------------------
# 9. Explicitly lock SSL eligibility
#
# SSL uses:
#   - Harvard-GF train: 2100 volumes
#   - Harvard-GDP ssl_pool: 500 volumes
#
# GDP progression-train/validation/test are excluded.
# ------------------------------------------------------------

gf_mask = manifest["dataset"].eq("harvard_gf")

manifest["ssl_eligible"] = (
    (
        gf_mask
        & manifest["protocol_split"].eq("train")
    )
    |
    (
        gdp_mask
        & manifest["protocol_split"].eq("ssl_pool")
    )
)

manifest["ssl_role"] = "not_used_for_ssl"

manifest.loc[
    gf_mask & manifest["protocol_split"].eq("train"),
    "ssl_role"
] = "gf_train"

manifest.loc[
    gdp_mask & manifest["protocol_split"].eq("ssl_pool"),
    "ssl_role"
] = "gdp_ssl_pool"

# ------------------------------------------------------------
# 10. Define modality availability and planned input modes
# ------------------------------------------------------------

for column in ["oct_locator", "rnflt_locator"]:
    manifest[column] = manifest[column].astype("string")

manifest["oct_available"] = (
    manifest["oct_locator"].notna()
    & manifest["oct_locator"].str.strip().ne("")
)

manifest["rnflt_available"] = (
    manifest["rnflt_locator"].notna()
    & manifest["rnflt_locator"].str.strip().ne("")
)

manifest["oct_rnflt_available"] = (
    manifest["oct_available"]
    & manifest["rnflt_available"]
)

manifest["allowed_input_modes"] = np.where(
    manifest["oct_rnflt_available"],
    "OCT|RNFLT|OCT+RNFLT",
    np.where(
        manifest["oct_available"],
        "OCT",
        np.where(
            manifest["rnflt_available"],
            "RNFLT",
            "NONE"
        )
    )
)

if not manifest["oct_available"].all():
    raise ValueError("At least one sample is missing its OCT locator.")

if not manifest["rnflt_available"].all():
    raise ValueError("At least one sample is missing its RNFLT locator.")

# These are policy fields, not model input columns.
manifest["progression_input_policy"] = (
    "OCT + device-derived RNFLT + OCT-derived anatomical representation"
)

manifest["excluded_progression_features"] = (
    "MD|VFI|TDS|MD_fast|MD_fast_no_p_cut|"
    "TD_pointwise_no_p_cut|age|gender|race|hispanic"
)

# ------------------------------------------------------------
# 11. Build an image-only SSL manifest
#
# This file intentionally contains NO clinical label columns.
# The SSL loader should read only this file.
# ------------------------------------------------------------

ssl_rows = manifest.loc[
    manifest["ssl_eligible"]
].copy()

ssl_manifest = ssl_rows[
    [
        "dataset",
        "sample_id",
        "subject_key",
        "oct_locator",
        "ssl_role",
    ]
].copy()

ssl_manifest["source_split"] = ssl_rows[
    "protocol_split"
].to_numpy()

ssl_manifest = ssl_manifest[
    [
        "dataset",
        "sample_id",
        "subject_key",
        "source_split",
        "ssl_role",
        "oct_locator",
    ]
]

# Verify exact SSL composition.
ssl_role_counts = {
    str(key): int(value)
    for key, value in ssl_manifest["ssl_role"].value_counts().items()
}

expected_ssl_roles = {
    "gf_train": 2100,
    "gdp_ssl_pool": 500,
}

if len(ssl_manifest) != 2600:
    raise ValueError(
        f"Expected 2600 SSL volumes; found {len(ssl_manifest)}."
    )

if ssl_role_counts != expected_ssl_roles:
    raise ValueError(
        "Incorrect SSL composition.\n"
        f"Observed: {ssl_role_counts}\n"
        f"Expected: {expected_ssl_roles}"
    )

if ssl_manifest["subject_key"].duplicated().any():
    raise ValueError("Duplicate sample found in SSL manifest.")

# This is the central progression leakage safeguard.
ssl_keys = set(ssl_manifest["subject_key"])

progression_labeled_keys = set(
    manifest.loc[
        manifest["progression_available"],
        "subject_key"
    ]
)

if not ssl_keys.isdisjoint(progression_labeled_keys):
    leaked_keys = ssl_keys & progression_labeled_keys

    raise ValueError(
        "Progression-labeled samples leaked into SSL: "
        f"{list(leaked_keys)[:10]}"
    )

if (
    ssl_manifest["dataset"].eq("harvard_gdp")
    & ssl_manifest["source_split"].ne("ssl_pool")
).any():
    raise ValueError(
        "Only GDP ssl_pool samples may enter SSL."
    )

if (
    ssl_manifest["dataset"].eq("harvard_gf")
    & ssl_manifest["source_split"].ne("train")
).any():
    raise ValueError(
        "Only GF train samples may enter SSL."
    )

# Prevent clinical targets being accidentally passed from the SSL manifest.
for forbidden_column in [
    "glaucoma_label",
    PRIMARY_PROGRESSION_ENDPOINT,
    "progression_available",
    "glaucoma_available",
]:
    if forbidden_column in ssl_manifest.columns:
        raise ValueError(
            f"Forbidden label column in SSL manifest: {forbidden_column}"
        )

# ------------------------------------------------------------
# 12. Create a dedicated GDP glaucoma split file
# ------------------------------------------------------------

gdp_detection_manifest = manifest.loc[
    gdp_mask,
    [
        "sample_id",
        "protocol_split",
        "glaucoma_split",
        "glaucoma_label",
        "oct_locator",
        "rnflt_locator",
    ]
].copy()

gdp_detection_manifest = gdp_detection_manifest.rename(
    columns={
        "protocol_split": "progression_source_split",
        "glaucoma_split": "final_glaucoma_split",
    }
)

# ------------------------------------------------------------
# 13. Leakage and protocol audit
# ------------------------------------------------------------

audit_rows = []


def add_check(name, passed, observed, expected):
    audit_rows.append({
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": str(observed),
        "expected": str(expected),
    })


def split_ids(frame, split_column, split_name):
    return set(
        frame.loc[
            frame[split_column].eq(split_name),
            "subject_key"
        ]
    )


gf_rows = manifest.loc[gf_mask].copy()
gdp_rows = manifest.loc[gdp_mask].copy()

# Split overlap checks for GF.
for split_a, split_b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = (
        split_ids(gf_rows, "protocol_split", split_a)
        &
        split_ids(gf_rows, "protocol_split", split_b)
    )

    add_check(
        f"GF {split_a}-{split_b} overlap",
        len(overlap) == 0,
        len(overlap),
        0,
    )

# Split overlap checks for GDP progression.
for split_a, split_b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
    ("ssl_pool", "train"),
    ("ssl_pool", "validation"),
    ("ssl_pool", "test"),
]:
    overlap = (
        split_ids(gdp_rows, "protocol_split", split_a)
        &
        split_ids(gdp_rows, "protocol_split", split_b)
    )

    add_check(
        f"GDP {split_a}-{split_b} overlap",
        len(overlap) == 0,
        len(overlap),
        0,
    )

# Check the exact SSL composition.
add_check(
    "Total unified samples",
    len(manifest) == 4300,
    len(manifest),
    4300,
)

add_check(
    "SSL sample count",
    len(ssl_manifest) == 2600,
    len(ssl_manifest),
    2600,
)

add_check(
    "SSL role counts",
    ssl_role_counts == expected_ssl_roles,
    ssl_role_counts,
    expected_ssl_roles,
)

add_check(
    "No progression-labeled samples in SSL",
    ssl_keys.isdisjoint(progression_labeled_keys),
    len(ssl_keys & progression_labeled_keys),
    0,
)

gdp_ssl_keys = set(
    ssl_manifest.loc[
        ssl_manifest["dataset"].eq("harvard_gdp"),
        "subject_key"
    ]
)

gdp_progression_train_keys = set(
    gdp_rows.loc[
        gdp_rows["protocol_split"].eq("train"),
        "subject_key"
    ]
)

add_check(
    "GDP progression-train excluded from SSL",
    gdp_ssl_keys.isdisjoint(gdp_progression_train_keys),
    len(gdp_ssl_keys & gdp_progression_train_keys),
    0,
)

# No validation/test images enter SSL.
validation_test_keys = set(
    manifest.loc[
        manifest["protocol_split"].isin(
            ["validation", "test"]
        ),
        "subject_key"
    ]
)

add_check(
    "No validation/test images in SSL",
    ssl_keys.isdisjoint(validation_test_keys),
    len(ssl_keys & validation_test_keys),
    0,
)

add_check(
    "SSL manifest excludes clinical labels",
    not any(
        column in ssl_manifest.columns
        for column in [
            "glaucoma_label",
            PRIMARY_PROGRESSION_ENDPOINT,
            "progression_available",
        ]
    ),
    "No clinical target columns",
    "Image metadata only",
)

add_check(
    "All samples have OCT and RNFLT locators",
    manifest["oct_available"].all()
    and manifest["rnflt_available"].all(),
    (
        f"OCT={int(manifest['oct_available'].sum())}, "
        f"RNFLT={int(manifest['rnflt_available'].sum())}"
    ),
    f"OCT={len(manifest)}, RNFLT={len(manifest)}",
)

add_check(
    "Progression label count",
    int(manifest["progression_available"].sum()) == 500,
    int(manifest["progression_available"].sum()),
    500,
)

# ------------------------------------------------------------
# 14. Save all Stage 10 outputs
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_rows)

manifest.to_csv(
    MAIN_MANIFEST_PATH,
    index=False,
)

ssl_manifest.to_csv(
    SSL_MANIFEST_PATH,
    index=False,
)

gdp_detection_manifest.to_csv(
    GDP_DETECTION_SPLIT_PATH,
    index=False,
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
)

protocol_summary = {
    "stage": "10",
    "protocol_status": (
        "PASS"
        if audit_df["status"].eq("PASS").all()
        else "FAIL"
    ),
    "seed": 42,
    "primary_progression_endpoint": PRIMARY_PROGRESSION_ENDPOINT,
    "harvard_gf_glaucoma": EXPECTED_GF_SPLITS,
    "harvard_gdp_progression": EXPECTED_GDP_PROGRESSION_SPLITS,
    "harvard_gdp_glaucoma": EXPECTED_GDP_GLAUCOMA_SPLITS,
    "ssl_policy": {
        "harvard_gf_train": 2100,
        "harvard_gdp_ssl_pool": 500,
        "total_ssl_volumes": 2600,
        "exclude_gdp_progression_train": True,
        "exclude_all_validation": True,
        "exclude_all_test": True,
        "ssl_manifest_contains_clinical_labels": False,
    },
    "progression_input_policy": (
        "OCT + device-derived RNFLT + OCT-derived anatomical representation"
    ),
    "excluded_progression_features": [
        "MD",
        "VFI",
        "TDS",
        "MD_fast",
        "MD_fast_no_p_cut",
        "TD_pointwise_no_p_cut",
        "age",
        "gender",
        "race",
        "hispanic",
    ],
    "input_modes": [
        "OCT",
        "RNFLT",
        "OCT+RNFLT",
    ],
    "output_files": {
        "main_manifest": MAIN_MANIFEST_PATH,
        "ssl_training_manifest": SSL_MANIFEST_PATH,
        "gdp_glaucoma_split": GDP_DETECTION_SPLIT_PATH,
        "leakage_audit": AUDIT_PATH,
    },
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(protocol_summary, file, indent=2)

# ------------------------------------------------------------
# 15. Final report
# ------------------------------------------------------------

print("=" * 88)
print("Stage 10 — Protocol Freeze, Leakage Audit, and SSL Manifest")
print("=" * 88)

print("\nHarvard-GF split counts:")
print(
    manifest.loc[gf_mask, "protocol_split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nHarvard-GDP progression split counts:")
print(
    manifest.loc[gdp_mask, "protocol_split"]
    .value_counts()
    .reindex(["train", "validation", "test", "ssl_pool"])
)

print("\nHarvard-GDP glaucoma split counts:")
print(
    manifest.loc[gdp_mask, "glaucoma_split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nSSL composition:")
print(
    ssl_manifest.groupby(
        ["dataset", "source_split", "ssl_role"]
    ).size()
)

print("\nProgression label distribution:")
print(
    manifest.loc[manifest["progression_available"]]
    .groupby("progression_split")[
        PRIMARY_PROGRESSION_ENDPOINT
    ]
    .value_counts()
    .unstack(fill_value=0)
)

print("\nLeakage audit:")
display(audit_df)

print("\nSaved files:")
print("Main manifest:", MAIN_MANIFEST_PATH)
print("SSL-only manifest:", SSL_MANIFEST_PATH)
print("GDP glaucoma split:", GDP_DETECTION_SPLIT_PATH)
print("Leakage audit:", AUDIT_PATH)
print("Protocol summary:", SUMMARY_PATH)

if audit_df["status"].ne("PASS").any():
    raise RuntimeError(
        "Stage 10 FAILED. Resolve the failed checks before proceeding."
    )

print("\nStage 10 status: PASS")
print(
    "\nIMPORTANT: Stage 11 must load SSL samples exclusively from:\n"
    f"{SSL_MANIFEST_PATH}"
)

In [ ]:
import os
import pandas as pd

stage10_dir = (
    "/content/drive/MyDrive/glaucoma_oct_research/"
    "results/stage10_protocol"
)

main_path = os.path.join(
    stage10_dir, "main_experiment_manifest.csv"
)

ssl_path = os.path.join(
    stage10_dir, "ssl_training_manifest.csv"
)

main = pd.read_csv(main_path)
ssl = pd.read_csv(ssl_path)

key = ["dataset", "sample_id"]

main["sample_id"] = main["sample_id"].astype(str).str.strip()
ssl["sample_id"] = ssl["sample_id"].astype(str).str.strip()

# Match the saved SSL records against the full manifest.
check = ssl.merge(
    main[
        key + [
            "protocol_split",
            "progression_available",
            "progression_td_pointwise"
        ]
    ],
    on=key,
    how="left",
    validate="one_to_one",
    indicator=True
)

assert check["_merge"].eq("both").all(), (
    "Some SSL samples do not match the main manifest."
)

gdp_ssl = check[
    check["dataset"].eq("harvard_gdp")
].copy()

gdp_progression_train = check[
    check["dataset"].eq("harvard_gdp")
    & check["protocol_split"].eq("train")
]

assert len(ssl) == 2600
assert len(gdp_ssl) == 500
assert gdp_ssl["protocol_split"].eq("ssl_pool").all()

assert not gdp_ssl["progression_available"].fillna(False).any()
assert gdp_ssl["progression_td_pointwise"].isna().all()

assert not (
    check["dataset"].eq("harvard_gdp")
    & check["protocol_split"].eq("train")
).any()

print("Total SSL samples:", len(ssl))
print("Harvard-GF train in SSL:", (ssl["dataset"] == "harvard_gf").sum())
print("Harvard-GDP ssl_pool in SSL:", len(gdp_ssl))
print("GDP progression-labeled samples in SSL:",
      int(gdp_ssl["progression_td_pointwise"].notna().sum()))
print("GDP progression-train samples in SSL:",
      int((check["dataset"].eq("harvard_gdp")
           & check["protocol_split"].eq("train")).sum()))
print("\nSaved-file verification: PASS")

## Native-axis SSL cache

Build and validate the fixed 12-B-scan cache used by DINO-style self-supervised pretraining.

In [ ]:
# ============================================================
# Stage 11.1 — Build and Validate Native-Axis SSL B-scan Caches
# ============================================================

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

assert "PROJECT_DIR" in globals(), "Run Stage 0 first."
assert "read_npz" in globals(), "Run the Stage 4 reader first."

SSL_MANIFEST_PATH = (
    PROJECT_DIR / "results" / "stage10_protocol" / "ssl_training_manifest.csv"
)
if not SSL_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"SSL manifest not found: {SSL_MANIFEST_PATH}")

ssl_manifest = pd.read_csv(SSL_MANIFEST_PATH)
required_columns = {"dataset", "sample_id", "oct_locator", "source_split", "ssl_role"}
missing_columns = required_columns - set(ssl_manifest.columns)
if missing_columns:
    raise ValueError(f"SSL manifest is missing columns: {sorted(missing_columns)}")

expected_roles = {"gf_train": 2100, "gdp_ssl_pool": 500}
actual_roles = {str(k): int(v) for k, v in ssl_manifest["ssl_role"].value_counts().items()}
if len(ssl_manifest) != 2600 or actual_roles != expected_roles:
    raise ValueError(f"Unexpected SSL composition: {actual_roles}")
if ssl_manifest["oct_locator"].isna().any():
    raise ValueError("Missing OCT locators in SSL manifest.")
if any(c in ssl_manifest.columns for c in ["glaucoma_label", "progression_td_pointwise"]):
    raise ValueError("Clinical label columns must not be present in the SSL manifest.")

eligible = (
    (ssl_manifest["dataset"].eq("harvard_gf") & ssl_manifest["source_split"].eq("train"))
    | (ssl_manifest["dataset"].eq("harvard_gdp") & ssl_manifest["source_split"].eq("ssl_pool"))
)
if not eligible.all():
    raise ValueError("An ineligible sample was found in the SSL manifest.")

signature_input = ssl_manifest[
    ["dataset", "sample_id", "oct_locator", "ssl_role"]
].to_csv(index=False)
manifest_signature = hashlib.sha256(signature_input.encode("utf-8")).hexdigest()

SEED = 42
BSCANS_PER_VOLUME = 12
DATASET_CONFIG = {
    "harvard_gf": {
        "volume_key": "oct_bscans",
        "expected_volume_shape": (200, 200, 200),
        "expected_count": 2100,
        "cache_shape": (2100, BSCANS_PER_VOLUME, 200, 200),
    },
    "harvard_gdp": {
        "volume_key": "bscans",
        "expected_volume_shape": (200, 200, 300),
        "expected_count": 500,
        "cache_shape": (500, BSCANS_PER_VOLUME, 200, 300),
    },
}

CACHE_DIR = Path("/content/stage11_ssl_cache_axis0")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CACHE_DIR / "cache_config.json"
CACHE_INDEX_PATH = CACHE_DIR / "cache_index.csv"
SLICE_MANIFEST_PATH = CACHE_DIR / "selected_bscan_indices.csv"

expected_config = {
    "manifest_signature": manifest_signature,
    "num_volumes": len(ssl_manifest),
    "bscans_per_volume": BSCANS_PER_VOLUME,
    "axis_policy": "native_axis0; no axis permutation, rotation, flip, resize, or padding",
    "axis_basis": "Stage 3 visual inspection; physical orientation metadata remains unverified",
    "dtype": "uint8",
    "seed": SEED,
    "datasets": {
        name: {
            "num_volumes": cfg["expected_count"],
            "native_volume_shape": list(cfg["expected_volume_shape"]),
            "cache_shape": list(cfg["cache_shape"]),
            "volume_key": cfg["volume_key"],
        }
        for name, cfg in DATASET_CONFIG.items()
    },
}

core_cache_files = [CONFIG_PATH]
provenance_files = [CACHE_INDEX_PATH, SLICE_MANIFEST_PATH]
for dataset_name in DATASET_CONFIG:
    core_cache_files.extend([
        CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy",
        CACHE_DIR / f"completed_samples_{dataset_name}.npy",
        CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy",
    ])
cache_files = core_cache_files + provenance_files

# An interrupted but structurally valid cache can resume from its completed-sample flags.
cache_valid = False
if all(path.is_file() for path in core_cache_files):
    try:
        with open(CONFIG_PATH, "r", encoding="utf-8") as file:
            old_config = json.load(file)
        cache_valid = old_config == expected_config
        if cache_valid:
            for dataset_name, cfg in DATASET_CONFIG.items():
                cache_path = CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy"
                done_path = CACHE_DIR / f"completed_samples_{dataset_name}.npy"
                indices_path = CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy"
                arr = np.load(cache_path, mmap_mode="r")
                completed = np.load(done_path)
                indices = np.load(indices_path)
                valid_completed_indices = (
                    indices[completed].size == 0
                    or np.all((indices[completed] >= 0) & (indices[completed] < 200))
                )
                cache_valid = (
                    cache_valid
                    and arr.shape == cfg["cache_shape"]
                    and arr.dtype == np.uint8
                    and completed.shape == (cfg["expected_count"],)
                    and indices.shape == (cfg["expected_count"], BSCANS_PER_VOLUME)
                    and valid_completed_indices
                )
                del arr, completed, indices
                if not cache_valid:
                    break
    except Exception:
        cache_valid = False

if not cache_valid:
    # Rebuild only the new axis-0 cache directory; the legacy axis-2 cache is not reused.
    for path in cache_files:
        if path.is_file():
            path.unlink()

    with open(CONFIG_PATH, "w", encoding="utf-8") as file:
        json.dump(expected_config, file, indent=2)

    for dataset_name, cfg in DATASET_CONFIG.items():
        subset = ssl_manifest.loc[ssl_manifest["dataset"].eq(dataset_name)].reset_index(drop=True)
        if len(subset) != cfg["expected_count"]:
            raise ValueError(
                f"{dataset_name}: expected {cfg['expected_count']} rows, found {len(subset)}"
            )

        cache_path = CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy"
        done_path = CACHE_DIR / f"completed_samples_{dataset_name}.npy"
        indices_path = CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy"

        cache = np.lib.format.open_memmap(
            cache_path, mode="w+", dtype=np.uint8, shape=cfg["cache_shape"]
        )
        completed = np.zeros(cfg["expected_count"], dtype=bool)
        selected_indices = np.full(
            (cfg["expected_count"], BSCANS_PER_VOLUME), -1, dtype=np.int32
        )
        cache.flush()
        np.save(done_path, completed)
        np.save(indices_path, selected_indices)
        del cache, completed, selected_indices

# Resume-safe cache fill, one dataset at a time; preserve every source pixel.
for dataset_name, cfg in DATASET_CONFIG.items():
    subset = ssl_manifest.loc[ssl_manifest["dataset"].eq(dataset_name)].reset_index(drop=True)
    cache_path = CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy"
    done_path = CACHE_DIR / f"completed_samples_{dataset_name}.npy"
    indices_path = CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy"

    cache = np.load(cache_path, mmap_mode="r+")
    completed = np.load(done_path)
    selected_indices = np.load(indices_path)

    for i in tqdm(range(len(subset)), desc=f"Caching {dataset_name}"):
        if completed[i]:
            continue

        row = subset.iloc[i]
        data = read_npz(row["oct_locator"])
        available_keys = list(data.keys())

        if cfg["volume_key"] not in data:
            raise KeyError(
                f"{cfg['volume_key']!r} missing for {dataset_name}, sample {row['sample_id']}; "
                f"available keys: {available_keys}"
            )

        volume = np.asarray(data[cfg["volume_key"]])
        del data

        if volume.shape != cfg["expected_volume_shape"]:
            raise ValueError(
                f"Unexpected native volume shape for {dataset_name}, sample {row['sample_id']}: "
                f"{volume.shape}; expected {cfg['expected_volume_shape']}"
            )
        if volume.dtype != np.uint8:
            raise TypeError(
                f"Expected uint8 OCT data for {dataset_name}; got {volume.dtype}"
            )

        # Stage 3 convention: a B-scan is volume[index, :, :].
        # Select scans across the native axis-0 sequence; do not rotate or resize.
        rng = np.random.default_rng(SEED + i + (0 if dataset_name == "harvard_gf" else 10_000))
        edges = np.linspace(0, volume.shape[0], BSCANS_PER_VOLUME + 1, dtype=int)
        indices = np.array([
            rng.integers(edges[j], edges[j + 1])
            for j in range(BSCANS_PER_VOLUME)
        ], dtype=np.int32)

        cache[i] = volume[indices]
        selected_indices[i] = indices
        completed[i] = True
        del volume

        if (i + 1) % 20 == 0 or i == len(subset) - 1:
            cache.flush()
            np.save(indices_path, selected_indices)
            np.save(done_path, completed)

    cache.flush()
    if not completed.all():
        raise RuntimeError(f"Incomplete cache for {dataset_name}: {completed.sum()}/{len(completed)}")
    if cache.shape != cfg["cache_shape"]:
        raise ValueError(f"Final cache shape mismatch for {dataset_name}: {cache.shape}")
    del cache, completed, selected_indices

# Save one index linking manifest rows to the two native-resolution cache files.
index_rows = []
slice_rows = []
for dataset_name, cfg in DATASET_CONFIG.items():
    subset = ssl_manifest.loc[ssl_manifest["dataset"].eq(dataset_name)].reset_index(drop=True)
    indices = np.load(CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy")
    cache_name = f"ssl_bscans_uint8_{dataset_name}.npy"
    for i, row in subset.iterrows():
        index_rows.append({
            "dataset": dataset_name,
            "sample_id": row["sample_id"],
            "source_split": row["source_split"],
            "ssl_role": row["ssl_role"],
            "cache_file": cache_name,
            "cache_row": int(i),
        })
        for slot, bscan_index in enumerate(indices[i]):
            slice_rows.append({
                "dataset": dataset_name,
                "sample_id": row["sample_id"],
                "ssl_role": row["ssl_role"],
                "cache_file": cache_name,
                "cache_row": int(i),
                "slice_slot": int(slot),
                "bscan_index_axis0": int(bscan_index),
            })

pd.DataFrame(index_rows).to_csv(CACHE_INDEX_PATH, index=False)
pd.DataFrame(slice_rows).to_csv(SLICE_MANIFEST_PATH, index=False)

# Final native-shape and representative-image validation.
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row_idx, dataset_name in enumerate(["harvard_gf", "harvard_gdp"]):
    cfg = DATASET_CONFIG[dataset_name]
    subset = ssl_manifest.loc[ssl_manifest["dataset"].eq(dataset_name)].reset_index(drop=True)
    cache = np.load(CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy", mmap_mode="r")
    indices = np.load(CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy")
    if cache.shape != cfg["cache_shape"] or cache.dtype != np.uint8:
        raise ValueError(f"Cache validation failed for {dataset_name}")
    if not np.isfinite(cache[0]).all():
        raise ValueError(f"Non-finite pixels found in {dataset_name} cache")
    for col_idx, slot in enumerate([0, BSCANS_PER_VOLUME // 2, BSCANS_PER_VOLUME - 1]):
        axes[row_idx, col_idx].imshow(cache[0, slot], cmap="gray", vmin=0, vmax=255)
        axes[row_idx, col_idx].set_title(
            f"{dataset_name} | axis-0 index {indices[0, slot]}\n{cache[0, slot].shape}"
        )
        axes[row_idx, col_idx].axis("off")
    del cache, indices

plt.tight_layout()
plt.show()
plt.close(fig)

print("=" * 78)
print("Stage 11.1 — Native-axis SSL cache validation")
print("SSL volumes:", len(ssl_manifest))
print("Harvard-GF train volumes:", DATASET_CONFIG["harvard_gf"]["expected_count"])
print("Harvard-GDP SSL-pool volumes:", DATASET_CONFIG["harvard_gdp"]["expected_count"])
print("B-scans cached:", len(ssl_manifest) * BSCANS_PER_VOLUME)
for dataset_name, cfg in DATASET_CONFIG.items():
    path = CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy"
    print(f"{dataset_name}: shape={cfg['cache_shape']}, size={path.stat().st_size / (1024**3):.2f} GiB")
print("Spatial transforms: none (source pixels and native axis order preserved)")
print("Legacy axis-2 cache: NOT USED")
print("Cache directory:", CACHE_DIR)
print("Sampling manifest:", SLICE_MANIFEST_PATH)
print("\nStage 11.1 status: PASS")

In [ ]:
# ============================================================
# Stage 11.1A — Persist Native-Axis SSL Caches on Google Drive
# ============================================================

import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

LOCAL_CACHE_DIR = Path("/content/stage11_ssl_cache_axis0")
DRIVE_CACHE_DIR = (
    Path("/content/drive/MyDrive/glaucoma_oct_research")
    / "data" / "processed" / "stage11_ssl_cache_axis0"
)

DATASET_SHAPES = {
    "harvard_gf": (2100, 12, 200, 200),
    "harvard_gdp": (500, 12, 200, 300),
}

cache_files = [
    "cache_config.json",
    "cache_index.csv",
    "selected_bscan_indices.csv",
]
for dataset_name in DATASET_SHAPES:
    cache_files.extend([
        f"ssl_bscans_uint8_{dataset_name}.npy",
        f"completed_samples_{dataset_name}.npy",
        f"selected_bscan_indices_{dataset_name}.npy",
    ])

for filename in cache_files:
    source = LOCAL_CACHE_DIR / filename
    if not source.is_file():
        raise FileNotFoundError(f"Required cache file is missing: {source}")

with open(LOCAL_CACHE_DIR / "cache_config.json", "r", encoding="utf-8") as file:
    local_config = json.load(file)

assert local_config["axis_policy"].startswith("native_axis0")
assert local_config["num_volumes"] == 2600
assert local_config["bscans_per_volume"] == 12

for dataset_name, expected_shape in DATASET_SHAPES.items():
    cache = np.load(
        LOCAL_CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy",
        mmap_mode="r",
    )
    completed = np.load(LOCAL_CACHE_DIR / f"completed_samples_{dataset_name}.npy")
    indices = np.load(LOCAL_CACHE_DIR / f"selected_bscan_indices_{dataset_name}.npy")
    assert cache.shape == expected_shape, (dataset_name, cache.shape)
    assert cache.dtype == np.uint8
    assert completed.shape == (expected_shape[0],) and completed.all()
    assert indices.shape == (expected_shape[0], expected_shape[1])
    assert np.all((indices >= 0) & (indices < 200))
    del cache, completed, indices

DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
for filename in cache_files:
    source = LOCAL_CACHE_DIR / filename
    destination = DRIVE_CACHE_DIR / filename
    temporary = DRIVE_CACHE_DIR / f"{filename}.partial"
    shutil.copy2(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        raise IOError(f"File size mismatch during copy: {filename}")
    os.replace(temporary, destination)

with open(DRIVE_CACHE_DIR / "cache_config.json", "r", encoding="utf-8") as file:
    drive_config = json.load(file)
assert drive_config == local_config

for dataset_name, expected_shape in DATASET_SHAPES.items():
    cache_path = DRIVE_CACHE_DIR / f"ssl_bscans_uint8_{dataset_name}.npy"
    cache = np.load(cache_path, mmap_mode="r")
    assert cache.shape == expected_shape, (dataset_name, cache.shape)
    assert cache.dtype == np.uint8
    print(f"{dataset_name}: {cache.shape}, {cache_path.stat().st_size / (1024**3):.2f} GiB")
    del cache

cache_index = pd.read_csv(DRIVE_CACHE_DIR / "cache_index.csv")
assert len(cache_index) == 2600
assert cache_index["dataset"].value_counts().to_dict() == {
    "harvard_gf": 2100,
    "harvard_gdp": 500,
}

print("Stage 11.1A — Native-axis Drive cache verification: PASS")
print("Total volumes: 2600")
print("Total sampled B-scans: 31200")
print("Drive cache:", DRIVE_CACHE_DIR)
print("The legacy stage11_ssl_cache/ cache is not used by this notebook.")


## DINO-style OCT self-supervised pretraining

Train the OCT encoder only on the label-free Stage 11 cache. The EMA teacher encoder is exported for downstream supervised training.

In [ ]:
# ============================================================
# Stage 11.2 — DINO-style self-supervised pretraining for OCT
# Trains only on the label-free Stage 11.1 axis-0 B-scan caches.
# Saves resumable checkpoints and a transferable teacher encoder to Drive.
# ============================================================

import hashlib, json, math, os, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm


# 1. Configuration
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU in Colab before running Stage 11.2.")

PROJECT_DIR = Path("/content/drive/MyDrive/glaucoma_oct_research")
if not PROJECT_DIR.exists():
    raise FileNotFoundError("Mount Google Drive and confirm PROJECT_DIR first.")

LOCAL_CACHE = Path("/content/stage11_ssl_cache_axis0")
DRIVE_CACHE = PROJECT_DIR / "data/processed/stage11_ssl_cache_axis0"
SSL_MANIFEST = PROJECT_DIR / "results/stage10_protocol/ssl_training_manifest.csv"
CKPT_DIR = PROJECT_DIR / "checkpoints/stage11_ssl_dino_v2_1"
RESULTS_DIR = PROJECT_DIR / "results/stage11_ssl_dino_v2_1"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED, SCANS_PER_VOLUME = 42, 12
IMAGE_H, IMAGE_W = 224, 320
BATCH_SIZE, EPOCHS, NUM_WORKERS = 32, 50, 4
BASE_LR, MIN_LR, WEIGHT_DECAY = 1.25e-4, 1e-5, 0.04
WARMUP_EPOCHS = 5
STUDENT_TEMP = 0.10
TEACHER_TEMP_START, TEACHER_TEMP_END, TEMP_WARMUP_EPOCHS = 0.04, 0.07, 5
TEACHER_MOMENTUM_START, CENTER_MOMENTUM = 0.996, 0.9
OUTPUT_DIM, BOTTLENECK_DIM = 4096, 256
TRAINING_VERSION = "dino_v2_1_parametrized_weight_norm_separate_teacher_fp32_loss"

DATASETS = {
    "harvard_gf": {"count": 2100, "shape": (2100, 12, 200, 200), "key": "oct_bscans"},
    "harvard_gdp": {"count": 500, "shape": (500, 12, 200, 300), "key": "bscans"},
}
CACHE_FILES = ["cache_config.json", "cache_index.csv", "selected_bscan_indices.csv"]
for name in DATASETS:
    CACHE_FILES += [
        f"ssl_bscans_uint8_{name}.npy",
        f"completed_samples_{name}.npy",
        f"selected_bscan_indices_{name}.npy",
    ]


# 2. Validate the label-free SSL manifest
if not SSL_MANIFEST.is_file():
    raise FileNotFoundError(f"SSL manifest not found: {SSL_MANIFEST}")
ssl_manifest = pd.read_csv(SSL_MANIFEST)
required = {"dataset", "sample_id", "oct_locator", "source_split", "ssl_role"}
if required - set(ssl_manifest.columns):
    raise ValueError(f"SSL manifest missing columns: {sorted(required - set(ssl_manifest.columns))}")
for col in ("glaucoma_label", "progression_td_pointwise"):
    if col in ssl_manifest.columns:
        raise ValueError(f"Clinical label column {col!r} must not be part of SSL inputs.")
roles = {str(k): int(v) for k, v in ssl_manifest["ssl_role"].value_counts().items()}
if len(ssl_manifest) != 2600 or roles != {"gf_train": 2100, "gdp_ssl_pool": 500}:
    raise ValueError(f"Unexpected SSL composition: rows={len(ssl_manifest)}, roles={roles}")
eligible = (
    (ssl_manifest.dataset.eq("harvard_gf") & ssl_manifest.source_split.eq("train"))
    | (ssl_manifest.dataset.eq("harvard_gdp") & ssl_manifest.source_split.eq("ssl_pool"))
)
if not eligible.all() or ssl_manifest.oct_locator.isna().any():
    raise ValueError("SSL manifest contains an ineligible sample or missing OCT locator.")
signature_text = ssl_manifest[["dataset", "sample_id", "oct_locator", "ssl_role"]].to_csv(index=False)
manifest_signature = hashlib.sha256(signature_text.encode("utf-8")).hexdigest()


def validate_cache(folder):
    folder = Path(folder)
    if not all((folder / f).is_file() for f in CACHE_FILES):
        return False, f"Required cache files missing in {folder}"
    try:
        cfg = json.loads((folder / "cache_config.json").read_text(encoding="utf-8"))
        if cfg.get("manifest_signature") != manifest_signature:
            return False, f"Manifest signature mismatch in {folder}"
        if cfg.get("num_volumes") != 2600 or cfg.get("bscans_per_volume") != SCANS_PER_VOLUME:
            return False, f"Unexpected cache counts in {folder}"
        if not str(cfg.get("axis_policy", "")).startswith("native_axis0") or cfg.get("dtype") != "uint8":
            return False, f"Cache axis/dtype policy mismatch in {folder}"
        for name, info in DATASETS.items():
            arr = np.load(folder / f"ssl_bscans_uint8_{name}.npy", mmap_mode="r")
            done = np.load(folder / f"completed_samples_{name}.npy")
            ix = np.load(folder / f"selected_bscan_indices_{name}.npy")
            ok = (arr.shape == info["shape"] and arr.dtype == np.uint8
                  and done.shape == (info["count"],) and done.all()
                  and ix.shape == (info["count"], SCANS_PER_VOLUME)
                  and np.all((ix >= 0) & (ix < 200)))
            del arr, done, ix
            if not ok:
                return False, f"Invalid shape/dtype/completion flags/indices for {name} in {folder}"
        ci = pd.read_csv(folder / "cache_index.csv")
        sm = pd.read_csv(folder / "selected_bscan_indices.csv")
        if len(ci) != 2600 or ci.dataset.value_counts().to_dict() != {"harvard_gf": 2100, "harvard_gdp": 500}:
            return False, f"Invalid cache_index.csv in {folder}"
        if len(sm) != 2600 * SCANS_PER_VOLUME:
            return False, f"Invalid selected-slice manifest in {folder}"
        return True, "OK"
    except Exception as exc:
        return False, f"Cache validation error in {folder}: {exc}"


def copy_cache(source, destination):
    source, destination = Path(source), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for filename in CACHE_FILES:
        src, dst = source / filename, destination / filename
        tmp = destination / (filename + ".partial")
        shutil.copy2(src, tmp)
        if tmp.stat().st_size != src.stat().st_size:
            raise IOError(f"Copy size mismatch for {filename}")
        os.replace(tmp, dst)


local_ok, local_msg = validate_cache(LOCAL_CACHE)
drive_ok, drive_msg = validate_cache(DRIVE_CACHE)
if local_ok:
    CACHE_DIR = LOCAL_CACHE
    if not drive_ok:
        print("Saving verified local SSL cache to Drive for reset-safe training...")
        copy_cache(LOCAL_CACHE, DRIVE_CACHE)
        drive_ok, drive_msg = validate_cache(DRIVE_CACHE)
        if not drive_ok:
            raise RuntimeError(f"Drive cache verification failed: {drive_msg}")
elif drive_ok:
    print("Restoring verified SSL cache from Drive to local runtime storage...")
    copy_cache(DRIVE_CACHE, LOCAL_CACHE)
    local_ok, local_msg = validate_cache(LOCAL_CACHE)
    if not local_ok:
        raise RuntimeError(f"Local cache verification failed: {local_msg}")
    CACHE_DIR = LOCAL_CACHE
else:
    raise RuntimeError(
        "No valid Stage 11.1 axis-0 cache found.\n"
        f"Local: {local_msg}\nDrive: {drive_msg}\n"
        "Run Stage 11.1 in this runtime, or confirm its cache was saved to Drive."
    )
print(f"Verified cache: {CACHE_DIR} | 2,600 volumes | 31,200 B-scans")


# 3. Reproducibility and DINO image views
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")


def letterbox(raw):
    """Fit a source B-scan to a common canvas while preserving aspect ratio."""
    x = torch.from_numpy(np.array(raw, dtype=np.uint8, copy=True)).float()[None] / 255.0
    _, h, w = x.shape
    scale = min(IMAGE_H / h, IMAGE_W / w)
    nh, nw = max(1, round(h * scale)), max(1, round(w * scale))
    x = F.interpolate(x[None], size=(nh, nw), mode="bilinear", align_corners=False)[0]
    canvas = torch.zeros((1, IMAGE_H, IMAGE_W), dtype=torch.float32)
    top, left = (IMAGE_H - nh) // 2, (IMAGE_W - nw) // 2
    canvas[:, top:top + nh, left:left + nw] = x
    return canvas


def global_view(canvas):
    """Mild DINO crop/intensity/noise/blur augmentation; no rotations or flips."""
    _, h, w = canvas.shape
    target_ratio = IMAGE_W / IMAGE_H
    crop = None
    for _ in range(10):
        area = random.uniform(0.70, 1.0) * h * w
        ratio = target_ratio * math.exp(random.uniform(-0.12, 0.12))
        ch, cw = int(round(math.sqrt(area / ratio))), int(round(math.sqrt(area * ratio)))
        if 1 <= ch <= h and 1 <= cw <= w:
            top, left = random.randint(0, h - ch), random.randint(0, w - cw)
            crop = canvas[:, top:top + ch, left:left + cw]
            break
    if crop is None:
        crop = canvas
    x = F.interpolate(crop[None], size=(IMAGE_H, IMAGE_W), mode="bilinear", align_corners=False)[0]
    brightness, contrast = random.uniform(0.80, 1.20), random.uniform(0.75, 1.25)
    mean = x.mean(dim=(-2, -1), keepdim=True)
    x = (((x - mean) * contrast + mean) * brightness).clamp(0, 1)
    x = x.pow(random.uniform(0.80, 1.25))
    if random.random() < 0.20:
        x = F.avg_pool2d(x[None], 3, stride=1, padding=1)[0]
    if random.random() < 0.50:
        x = (x + torch.randn_like(x) * 0.012).clamp(0, 1)
    return x


class BScanViews(Dataset):
    def __init__(self, folder):
        self.gf = np.load(Path(folder) / "ssl_bscans_uint8_harvard_gf.npy", mmap_mode="r")
        self.gdp = np.load(Path(folder) / "ssl_bscans_uint8_harvard_gdp.npy", mmap_mode="r")
        self.gf_n = self.gf.shape[0] * self.gf.shape[1]

    def __len__(self):
        return self.gf.shape[0] * self.gf.shape[1] + self.gdp.shape[0] * self.gdp.shape[1]

    def __getitem__(self, index):
        if index < self.gf_n:
            v, s = divmod(index, SCANS_PER_VOLUME)
            raw = self.gf[v, s]
        else:
            v, s = divmod(index - self.gf_n, SCANS_PER_VOLUME)
            raw = self.gdp[v, s]
        canvas = letterbox(raw)
        return global_view(canvas), global_view(canvas)


# 4. Grayscale ResNet-18 DINO student/teacher
class DINOHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(512, 2048), nn.GELU(),
            nn.Linear(2048, 2048), nn.GELU(),
            nn.Linear(2048, BOTTLENECK_DIM),
        )
        # Weight normalization keeps each prototype direction at unit norm.
        # A frozen scale of 1 follows the stable DINO last-layer design.
        self.last = nn.utils.parametrizations.weight_norm(
            nn.Linear(BOTTLENECK_DIM, OUTPUT_DIM, bias=False)
        )
        # The first original is the per-prototype magnitude (g).
        with torch.no_grad():
            self.last.parametrizations.weight.original0.fill_(1.0)
        self.last.parametrizations.weight.original0.requires_grad_(False)

    def forward(self, x):
        return self.last(F.normalize(self.mlp(x), dim=-1))


class DINOResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Identity()
        self.head = DINOHead()

    def forward(self, x):
        return self.head(self.backbone(x))


student = DINOResNet18().to(DEVICE)
# Build Teacher separately; deepcopy can fail for computed weight-normalized tensors.
teacher = DINOResNet18().to(DEVICE)
teacher.load_state_dict(student.state_dict())
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()
optimizer = torch.optim.AdamW(student.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=True)
center = torch.zeros(1, OUTPUT_DIM, device=DEVICE)


def dino_loss(s_logits, t_logits, teacher_temp):
    # Keep softmax/log-softmax and cross-entropy in FP32 under mixed precision.
    s_logits = s_logits.float()
    t_logits = t_logits.float()
    t_prob = F.softmax((t_logits - center.float()) / teacher_temp, dim=-1).detach()
    t1, t2 = t_prob.chunk(2)
    s1, s2 = s_logits.chunk(2)
    loss_v1 = -(t1 * F.log_softmax(s2 / STUDENT_TEMP, dim=-1)).sum(-1).mean()
    loss_v2 = -(t2 * F.log_softmax(s1 / STUDENT_TEMP, dim=-1)).sum(-1).mean()
    loss = 0.5 * (loss_v1 + loss_v2)
    entropy = -(t_prob * torch.log(t_prob.clamp_min(1e-12))).sum(-1).mean()
    probability_std = t_prob.std(dim=-1, unbiased=False).mean()
    max_probability = t_prob.max(dim=-1).values.mean()
    return loss, entropy, probability_std, max_probability


def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".partial")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def lr_at(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return 1e-5 + (BASE_LR - 1e-5) * step / max(1, warmup_steps)
    p = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (1 + math.cos(math.pi * p))


def momentum_at(step, total_steps):
    p = min(1.0, step / max(1, total_steps))
    return 1 - (1 - TEACHER_MOMENTUM_START) * (math.cos(math.pi * p) + 1) / 2


def temp_at(epoch, batch_idx, steps_per_epoch):
    p = min(1.0, (epoch + batch_idx / max(1, steps_per_epoch)) / TEMP_WARMUP_EPOCHS)
    return TEACHER_TEMP_START + p * (TEACHER_TEMP_END - TEACHER_TEMP_START)


# 5. DataLoader and resume state
loader_gen = torch.Generator().manual_seed(SEED)
dataset = BScanViews(CACHE_DIR)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=False, generator=loader_gen,
)
steps_per_epoch = len(loader)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
LATEST = CKPT_DIR / "dino_ssl_latest.pt"
HISTORY_CSV = RESULTS_DIR / "dino_ssl_training_history.csv"
ENCODER_EXPORT = CKPT_DIR / "dino_teacher_resnet18_encoder_axis0_ssl_v2.pt"

train_config = {
    "training_version": TRAINING_VERSION,
    "method": "DINO-style two-view teacher-student self-distillation",
    "backbone": "torchvision ResNet-18, one-channel input, randomly initialized",
    "output_dim": OUTPUT_DIM, "bottleneck_dim": BOTTLENECK_DIM,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "input_canvas_hw": [IMAGE_H, IMAGE_W], "scans_per_volume": SCANS_PER_VOLUME,
    "axis_policy": "native axis 0; no axis permutation/rotation/flip in stored cache",
    "augmentations": "aspect-preserving letterbox; mild random crop, intensity, noise and blur",
    "optimizer": "AdamW",
    "base_lr": BASE_LR, "min_lr": MIN_LR, "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS, "student_temperature": STUDENT_TEMP,
    "teacher_temperature": [TEACHER_TEMP_START, TEACHER_TEMP_END, TEMP_WARMUP_EPOCHS],
    "teacher_momentum_start": TEACHER_MOMENTUM_START,
    "center_momentum": CENTER_MOMENTUM,
    "manifest_signature": manifest_signature,
}

start_epoch, global_step, history = 0, 0, []
if LATEST.is_file():
    try:
        state = torch.load(LATEST, map_location="cpu", weights_only=False)
    except TypeError:
        state = torch.load(LATEST, map_location="cpu")
    if state.get("train_config") != train_config:
        raise ValueError("Existing SSL checkpoint uses a different configuration; refusing unsafe resume.")
    student.load_state_dict(state["student"]); teacher.load_state_dict(state["teacher"])
    optimizer.load_state_dict(state["optimizer"]); scaler.load_state_dict(state["scaler"])
    center.copy_(state["center"].to(DEVICE))
    start_epoch, global_step = int(state["completed_epoch"]) + 1, int(state["global_step"])
    history = state.get("history", [])
    loader_gen.set_state(state["loader_generator_state"])
    random.setstate(state["python_rng_state"]); np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"]); torch.cuda.set_rng_state_all(state["cuda_rng_states"])
    print(f"Resuming after epoch {start_epoch}; global step={global_step}.")
else:
    print("Starting corrected DINO SSL v2 pretraining from scratch.")


# 6. Train; checkpoint after every completed epoch
for epoch in range(start_epoch, EPOCHS):
    student.train(); teacher.eval()
    loss_sum = entropy_sum = prob_std_sum = max_prob_sum = teacher_logit_std_sum = 0.0
    bar = tqdm(loader, desc=f"DINO SSL {epoch + 1}/{EPOCHS}")

    for batch_idx, (view1, view2) in enumerate(bar):
        view1 = view1.to(DEVICE, non_blocking=True)
        view2 = view2.to(DEVICE, non_blocking=True)
        lr = lr_at(global_step, total_steps, warmup_steps)
        for group in optimizer.param_groups:
            group["lr"] = lr

        views = torch.cat([view1, view2], dim=0)
        with torch.cuda.amp.autocast(enabled=True):
            s_logits = student(views)
            with torch.no_grad():
                t_logits = teacher(views)
            loss, entropy, prob_std, max_prob = dino_loss(
                s_logits, t_logits, temp_at(epoch, batch_idx, steps_per_epoch)
            )

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student.parameters(), 3.0)
        scaler.step(optimizer); scaler.update()

        mom = momentum_at(global_step, total_steps)
        with torch.no_grad():
            for ps, pt in zip(student.parameters(), teacher.parameters()):
                pt.data.mul_(mom).add_(ps.data, alpha=1 - mom)
            # Keep teacher BatchNorm running statistics aligned with the student.
            student_buffers = dict(student.named_buffers())
            for name, teacher_buffer in teacher.named_buffers():
                teacher_buffer.copy_(student_buffers[name])
            center.mul_(CENTER_MOMENTUM).add_(t_logits.detach().mean(0, keepdim=True), alpha=1 - CENTER_MOMENTUM)

        loss_sum += float(loss.item())
        entropy_sum += float(entropy.item())
        prob_std_sum += float(prob_std.item())
        max_prob_sum += float(max_prob.item())
        teacher_logit_std_sum += float(t_logits.float().std(unbiased=False).item())
        global_step += 1
        bar.set_postfix(loss=f"{loss_sum / (batch_idx + 1):.4f}", lr=f"{lr:.2e}")

    record = {
        "epoch": epoch + 1, "global_step": global_step,
        "dino_loss": loss_sum / steps_per_epoch,
        "teacher_entropy": entropy_sum / steps_per_epoch,
        "teacher_probability_std": prob_std_sum / steps_per_epoch,
        "teacher_max_probability": max_prob_sum / steps_per_epoch,
        "teacher_logit_std": teacher_logit_std_sum / steps_per_epoch,
        "last_learning_rate": lr_at(max(0, global_step - 1), total_steps, warmup_steps),
        "last_teacher_momentum": momentum_at(max(0, global_step - 1), total_steps),
    }
    history.append(record)
    state = {
        "completed_epoch": epoch, "global_step": global_step, "train_config": train_config,
        "student": student.state_dict(), "teacher": teacher.state_dict(),
        "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
        "center": center.detach().cpu(), "history": history,
        "loader_generator_state": loader_gen.get_state(),
        "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(), "cuda_rng_states": torch.cuda.get_rng_state_all(),
    }
    atomic_save(state, LATEST)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)
    print(
        f"Epoch {epoch + 1}/{EPOCHS}: "
        f"loss={record['dino_loss']:.4f}; "
        f"teacher_entropy={record['teacher_entropy']:.4f}/{math.log(OUTPUT_DIM):.4f}; "
        f"teacher_prob_std={record['teacher_probability_std']:.3e}; "
        f"teacher_logit_std={record['teacher_logit_std']:.3e}; checkpoint saved."
    )

    # Stop early if the teacher has become numerically uniform for two consecutive epochs.
    collapsed_now = (
        epoch >= 1
        and record["teacher_entropy"] >= math.log(OUTPUT_DIM) - 1e-5
        and record["teacher_probability_std"] < 1e-7
    )
    prior_collapsed = False
    if len(history) >= 2:
        prev = history[-2]
        prior_collapsed = (
            prev["teacher_entropy"] >= math.log(OUTPUT_DIM) - 1e-5
            and prev.get("teacher_probability_std", float("inf")) < 1e-7
        )
    if collapsed_now and prior_collapsed:
        raise RuntimeError(
            "DINO collapse guard triggered: teacher probabilities are effectively uniform "
            "for two consecutive epochs. Check the latest checkpoint and diagnostics."
        )


# 7. Export the EMA teacher backbone for initialization of the main OCT encoder
export = {
    "training_version": TRAINING_VERSION,
    "architecture": "torchvision_resnet18_grayscale",
    "state_dict": teacher.backbone.state_dict(),
    "source": "Stage 11.2 DINO teacher backbone; self-supervised OCT pretraining",
    "input_channels": 1, "input_canvas_hw": [IMAGE_H, IMAGE_W],
    "axis_policy": "axis-0 B-scans; cache stores native pixels without axis changes",
    "train_config": train_config, "completed_epochs": EPOCHS,
    "manifest_signature": manifest_signature,
}
atomic_save(export, ENCODER_EXPORT)

print("=" * 78)
print("Stage 11.2 completed")
print("Training history:", HISTORY_CSV)
print("Resume checkpoint:", LATEST)
print("Encoder to initialize the main OCT branch:", ENCODER_EXPORT)
print("Clinical labels used in SSL loss: NO")
print("Segmentation U-Net weights: unchanged and separate")

In [ ]:
# ============================================================
# Stage 11.3 — Inspect learned representations from DINO OCT SSL
# Loads the EMA Teacher encoder exported by Stage 11.2 and visualizes
# intermediate features, local/global feature similarity, and Grad-CAM
# for the embedding norm. No clinical labels are used.
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import models

# 1. Paths and settings
PROJECT_DIR = Path("/content/drive/MyDrive/glaucoma_oct_research")
CKPT_DIR = PROJECT_DIR / "checkpoints/stage11_ssl_dino_v2_1"
ENCODER_PATH = CKPT_DIR / "dino_teacher_resnet18_encoder_axis0_ssl_v2.pt"
LATEST_PATH = CKPT_DIR / "dino_ssl_latest.pt"
LOCAL_CACHE = Path("/content/stage11_ssl_cache_axis0")
DRIVE_CACHE = PROJECT_DIR / "data/processed/stage11_ssl_cache_axis0"
OUTPUT_DIR = PROJECT_DIR / "results/stage11_ssl_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
SAMPLES_PER_DATASET = 4
INPUT_H, INPUT_W = 224, 320
BATCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not PROJECT_DIR.exists():
    raise FileNotFoundError("Google Drive is not mounted or PROJECT_DIR is missing.")

# 2. Load the final EMA Teacher encoder (the SSL script does not select a best epoch).
def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

backbone = models.resnet18(weights=None)
backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
backbone.fc = nn.Identity()

if ENCODER_PATH.is_file():
    export = safe_torch_load(ENCODER_PATH)
    state_dict = export["state_dict"]
    checkpoint_source = str(ENCODER_PATH)
    print("Loaded exported EMA Teacher encoder.")
    print("Completed epochs:", export.get("completed_epochs", "not recorded"))
    print("Training version:", export.get("training_version", "not recorded"))
elif LATEST_PATH.is_file():
    state = safe_torch_load(LATEST_PATH)
    teacher_state = state["teacher"]
    state_dict = {
        key.removeprefix("backbone."): value
        for key, value in teacher_state.items()
        if key.startswith("backbone.")
    }
    checkpoint_source = str(LATEST_PATH) + " (Teacher backbone extracted from resumable checkpoint)"
    print("Encoder export not found; loaded EMA Teacher from latest checkpoint.")
    print("Completed epoch:", int(state.get("completed_epoch", -1)) + 1)
else:
    raise FileNotFoundError(
        f"Neither exported encoder nor latest checkpoint was found. Checked:\n{ENCODER_PATH}\n{LATEST_PATH}"
    )

missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
if missing or unexpected:
    raise RuntimeError(f"Encoder state mismatch. Missing={missing}; unexpected={unexpected}")
backbone = backbone.to(BATCH_DEVICE).eval()
for parameter in backbone.parameters():
    parameter.requires_grad_(False)
print("Checkpoint source:", checkpoint_source)
print("Device:", BATCH_DEVICE)

# 3. Find the validated Stage 11.1 cache; cache pixels are kept in native axis-0 order.
required_cache_files = [
    "ssl_bscans_uint8_harvard_gf.npy",
    "ssl_bscans_uint8_harvard_gdp.npy",
    "selected_bscan_indices_harvard_gf.npy",
    "selected_bscan_indices_harvard_gdp.npy",
]

def has_cache(folder):
    return all((folder / name).is_file() for name in required_cache_files)

if has_cache(LOCAL_CACHE):
    CACHE_DIR = LOCAL_CACHE
elif has_cache(DRIVE_CACHE):
    CACHE_DIR = DRIVE_CACHE
else:
    raise FileNotFoundError(
        "Stage 11.1 cache files were not found in local or Drive cache directories. "
        "Run Stage 11.1 or check its saved cache path."
    )

print("Using cache:", CACHE_DIR)

# 4. Deterministic letterbox matching the DINO training input canvas.
def make_canvas(raw):
    image = torch.from_numpy(np.array(raw, dtype=np.uint8, copy=True)).float()[None, None] / 255.0
    _, _, height, width = image.shape
    scale = min(INPUT_H / height, INPUT_W / width)
    new_h = max(1, round(height * scale))
    new_w = max(1, round(width * scale))
    image = F.interpolate(image, size=(new_h, new_w), mode="bilinear", align_corners=False)
    canvas = torch.zeros((1, 1, INPUT_H, INPUT_W), dtype=torch.float32)
    top, left = (INPUT_H - new_h) // 2, (INPUT_W - new_w) // 2
    canvas[:, :, top:top + new_h, left:left + new_w] = image
    return canvas

# 5. Hooks capture activations for the same input, without training or augmentation.
def capture_activations(x):
    activations = {}
    modules = {
        "Stem": backbone.relu,
        "Layer 1": backbone.layer1,
        "Layer 2": backbone.layer2,
        "Layer 3": backbone.layer3,
        "Layer 4": backbone.layer4,
    }
    handles = []
    for name, module in modules.items():
        handles.append(module.register_forward_hook(
            lambda _module, _inputs, output, key=name: activations.__setitem__(key, output)
        ))
    with torch.no_grad():
        embedding = backbone(x.to(BATCH_DEVICE))
    for handle in handles:
        handle.remove()
    return activations, embedding


def scale_map(values):
    values = np.asarray(values, dtype=np.float32)
    low, high = np.percentile(values, [1, 99])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return np.zeros_like(values, dtype=np.float32)
    return np.clip((values - low) / (high - low), 0, 1)


def upsample_map(values, mode="bilinear"):
    tensor = torch.as_tensor(values, dtype=torch.float32)[None, None]
    return F.interpolate(tensor, size=(INPUT_H, INPUT_W), mode=mode,
                         align_corners=False if mode == "bilinear" else None)[0, 0].numpy()


def representation_gradcam(x):
    """Grad-CAM for the L2 norm of the 512-D embedding, not a class prediction."""
    holder = {}
    def hook(_module, _inputs, output):
        holder["features"] = output
        output.retain_grad()
    handle = backbone.layer4.register_forward_hook(hook)
    backbone.zero_grad(set_to_none=True)
    x = x.to(BATCH_DEVICE).requires_grad_(True)
    embedding = backbone(x)
    target = embedding.norm(p=2, dim=1).sum()
    target.backward()
    features = holder["features"].detach()
    gradients = holder["features"].grad.detach()
    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * features).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(INPUT_H, INPUT_W), mode="bilinear", align_corners=False)[0, 0]
    handle.remove()
    backbone.zero_grad(set_to_none=True)
    x.grad = None
    return scale_map(cam.detach().cpu().numpy())

# 6. Choose a small, reproducible set of examples from both data sources.
rng = np.random.default_rng(SEED)
selected = []
cache_arrays = {}
scan_indices = {}
for dataset in ("harvard_gf", "harvard_gdp"):
    cache_arrays[dataset] = np.load(CACHE_DIR / f"ssl_bscans_uint8_{dataset}.npy", mmap_mode="r")
    scan_indices[dataset] = np.load(CACHE_DIR / f"selected_bscan_indices_{dataset}.npy")
    n_volumes, n_scans = cache_arrays[dataset].shape[:2]
    chosen_volumes = rng.choice(n_volumes, size=min(SAMPLES_PER_DATASET, n_volumes), replace=False)
    for volume_idx in chosen_volumes:
        scan_slot = int(rng.integers(0, n_scans))
        selected.append({
            "dataset": dataset,
            "volume_index": int(volume_idx),
            "cache_scan_slot": scan_slot,
            "native_axis0_index": int(scan_indices[dataset][volume_idx, scan_slot]),
            "raw": np.array(cache_arrays[dataset][volume_idx, scan_slot], copy=True),
        })

# 7. Create overview figure, feature vectors, and representative channel maps.
columns = ["Input", "Stem energy", "Layer 1 energy", "Layer 2 energy", "Layer 3 energy",
           "Layer 4 energy", "Local/global similarity", "Embedding Grad-CAM"]
fig, axes = plt.subplots(len(selected), len(columns), figsize=(22, 2.8 * len(selected)), squeeze=False)
metadata_rows, vectors = [], []
representative_channel_data = {}

for row, sample in enumerate(selected):
    x = make_canvas(sample["raw"])
    image = x[0, 0].numpy()
    activations, embedding = capture_activations(x)
    vectors.append(embedding[0].detach().cpu().numpy())

    maps = {}
    for name in ("Stem", "Layer 1", "Layer 2", "Layer 3", "Layer 4"):
        feature = activations[name]
        energy = torch.sqrt(torch.mean(feature.float() ** 2, dim=1))[0]
        maps[name] = scale_map(upsample_map(energy.detach().cpu().numpy()))

    layer4 = activations["Layer 4"].detach().float()
    local = F.normalize(layer4, dim=1)
    global_vector = F.normalize(layer4.mean(dim=(2, 3), keepdim=True), dim=1)
    similarity = (local * global_vector).sum(dim=1)[0].cpu().numpy()
    similarity = upsample_map(similarity)
    gradcam = representation_gradcam(x)

    axes[row, 0].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_ylabel(
        f"{sample['dataset']}\nvol={sample['volume_index']}\naxis-0={sample['native_axis0_index']}",
        fontsize=7, rotation=0, labelpad=48, va="center"
    )
    axes[row, 1].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].imshow(maps["Stem"], cmap="magma", alpha=0.48, vmin=0, vmax=1)
    for col, layer_name in enumerate(("Layer 1", "Layer 2", "Layer 3", "Layer 4"), start=2):
        axes[row, col].imshow(image, cmap="gray", vmin=0, vmax=1)
        axes[row, col].imshow(maps[layer_name], cmap="magma", alpha=0.48, vmin=0, vmax=1)
    axes[row, 6].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[row, 6].imshow(similarity, cmap="coolwarm", alpha=0.55, vmin=-1, vmax=1)
    axes[row, 7].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[row, 7].imshow(gradcam, cmap="magma", alpha=0.55, vmin=0, vmax=1)

    for ax in axes[row]:
        ax.set_xticks([]); ax.set_yticks([])
    metadata_rows.append({
        "dataset": sample["dataset"],
        "volume_index_in_cache": sample["volume_index"],
        "cache_scan_slot": sample["cache_scan_slot"],
        "native_axis0_index": sample["native_axis0_index"],
        "embedding_l2_norm": float(np.linalg.norm(vectors[-1])),
    })
    if sample["dataset"] not in representative_channel_data:
        # Save six layer-4 channels with the largest mean activation for one sample per dataset.
        channel_means = layer4[0].mean(dim=(1, 2))
        top_channels = torch.topk(channel_means, k=6).indices.tolist()
        representative_channel_data[sample["dataset"]] = (image.copy(), layer4[0].cpu().numpy(), top_channels)

for col, title in enumerate(columns):
    axes[0, col].set_title(title, fontsize=10)
fig.suptitle(
    "DINO EMA Teacher — OCT representation inspection\n"
    "Activation-energy overlays; similarity compares local Layer-4 features with the global feature vector",
    fontsize=13, y=1.005
)
fig.tight_layout()
overview_path = OUTPUT_DIR / "ssl_representation_overview.png"
fig.savefig(overview_path, dpi=180, bbox_inches="tight")
plt.show()

# 8. Plot individual Layer-4 channels for one GF and one GDP example.
for dataset, (image, feature_map, channel_ids) in representative_channel_data.items():
    fig_ch, ax_ch = plt.subplots(2, 3, figsize=(12, 5))
    for ax, channel_id in zip(ax_ch.flat, channel_ids):
        channel_map = upsample_map(feature_map[channel_id])
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.imshow(scale_map(channel_map), cmap="viridis", alpha=0.55, vmin=0, vmax=1)
        ax.set_title(f"Layer 4 channel {channel_id}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig_ch.suptitle(f"Most-active Layer-4 channels — {dataset}", fontsize=12)
    fig_ch.tight_layout()
    fig_ch.savefig(OUTPUT_DIR / f"layer4_channels_{dataset}.png", dpi=180, bbox_inches="tight")
    plt.show()

# 9. Save selected 512-D representations and their sample provenance.
np.save(OUTPUT_DIR / "selected_bscan_embeddings_512d.npy", np.asarray(vectors, dtype=np.float32))
pd.DataFrame(metadata_rows).to_csv(OUTPUT_DIR / "selected_bscan_embedding_metadata.csv", index=False)

summary = {
    "checkpoint_source": checkpoint_source,
    "device": str(BATCH_DEVICE),
    "cache_dir": str(CACHE_DIR),
    "samples_visualized": len(selected),
    "embedding_dimensions": int(np.asarray(vectors).shape[1]),
    "input_canvas_hw": [INPUT_H, INPUT_W],
    "axis_policy": "native axis 0; the cached B-scan slice is not rotated or flipped",
    "interpretation_note": (
        "ResNet-18 has no native attention map. Grad-CAM here targets embedding L2 norm, not a clinical class. "
        "Maps are diagnostic views of feature activations/sensitivity, not proof of semantic attention."
    ),
}
(OUTPUT_DIR / "inspection_config.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\nStage 11.3 inspection completed.")
print("Overview figure:", overview_path)
print("Outputs folder:", OUTPUT_DIR)
print("Embedding array:", OUTPUT_DIR / "selected_bscan_embeddings_512d.npy")
print("Metadata:", OUTPUT_DIR / "selected_bscan_embedding_metadata.csv")
print("\nInterpretation: these maps reveal intermediate activations and representation sensitivity; they are not clinical explanations or proof of what the network 'understands'.")

## Reproducibility note

For exact published downstream results, keep the frozen manifests and exported checkpoints unchanged. Training is deterministic under the original seed/configuration, but exact floating-point trajectories can still depend on the CUDA/PyTorch environment. The downstream notebooks therefore verify checkpoint provenance by hash.